# Ledger format x grouping guidance: eight-generation ablation
Create a new private notebook. Enable internet, GPU T4 x2 and HF_TOKEN in Kaggle Secrets. Only GPU 0 is visible. Attach a private Dataset containing ONLY ledger_ablation_pack.json. Keep all inputs and outputs private.
Two already-observed development failures, four conditions each: sentence-only evidence without/with the original grouping plan; cue-rich evidence without/with that same plan. Removing protected_evidence fields is the only ledger change. Rich controls exactly repeat the prior prompts. Sentence count stays free.
Pinned base Gemma 3 4B, NF4/FP32, greedy decoding, 768 new tokens. No adapters, training or repairs. This is a post-hoc mechanism diagnostic, not fresh validation or a learned taxonomy mapper. No reference text is given to Gemma.
The passed v3 verifier calibration is repeated before Gemma Secret access. NLI is unloaded before generation, then reloaded after Gemma unload. Evaluation reports cue artifacts, sentence inflation, old-control output hashes, semantics, copying, style and cost separately. All are imperfect diagnostic proxies.
Outputs: /kaggle/working/ledger_grouping_ablation_v1/evaluation.json, generations.private.jsonl, metrics.private.jsonl, token_audit.json, run_status.json and manifest.json. Nothing was submitted by this builder.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate==1.14.0" "bitsandbytes==0.50.2" "huggingface-hub==1.33.0" "sentencepiece>=0.2"


In [ ]:
import json, sys
from pathlib import Path
runtime = Path("/kaggle/working/_ledger_ablation_runtime")
files = json.loads('{"src/fama_style/__init__.py": "\\"\\"\\"Fama-style project modules; imports do not load model weights.\\"\\"\\"\\n", "src/fama_style/prompting/__init__.py": "\\"\\"\\"Fama-style prompting modules; imports do not load model weights.\\"\\"\\"\\n", "src/fama_style/prompting/corpus.py": "\\"\\"\\"Frozen multi-paper v1 instruction, identical to the curated data export.\\"\\"\\"\\n\\nFORMAT = \\"fama_multi_paper_v1\\"\\nINSTRUCTION = (\\n    \\"Rewrite the passage below in the formal academic economics style of Eugene Fama \\"\\n    \\"and his coauthors. Preserve every substantive claim, actor, attribution, \\"\\n    \\"condition, qualification, uncertainty, negation, quantity, comparison, and \\"\\n    \\"causal relationship. Do not add or omit claims or strengthen tentative claims. \\"\\n    \\"Return only one coherent paragraph, without headings, Markdown, explanations, \\"\\n    \\"or commentary.\\"\\n)\\n\\n\\ndef make_prompt(text):\\n    return f\\"{INSTRUCTION}\\\\n\\\\nPassage:\\\\n{text}\\"\\n", "src/fama_style/research/__init__.py": "\\"\\"\\"CPU-first claim-preserving rewriting research; no training launchers.\\"\\"\\"\\n", "src/fama_style/research/checker_benchmark.py": "\\"\\"\\"Frozen CPU benchmark for claim-risk detection, false alarms and abstention.\\"\\"\\"\\nimport argparse\\nfrom collections import Counter\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nfrom .claim_audit import claim_checks\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'claim_checker_benchmark_v1\'\\n\\n\\ndef digest(path):return hashlib.sha256(path.read_bytes()).hexdigest()\\ndef rate(a,b):return a/b if b else None\\n\\n\\ndef classify(audit):\\n    if audit[\'events\']:return \'risk_signal\'\\n    return \'surface_accept\' if audit[\'status\']==\'surface_replay_only\' else \'abstain\'\\n\\n\\ndef summarize(records):\\n    faithful=[r for r in records if r[\'expected\']==\'faithful\'];changed=[r for r in records if r[\'expected\']==\'changed\']\\n    hard=[r for r in faithful if r[\'variant\']==\'paraphrase\']\\n    return {\'rows\':len(records),\'faithful_rows\':len(faithful),\'changed_rows\':len(changed),\\n        \'prediction_counts\':dict(Counter(r[\'prediction\'] for r in records)),\\n        \'error_detection_rate\':rate(sum(r[\'prediction\']==\'risk_signal\' for r in changed),len(changed)),\\n        \'faithful_false_alarm_rate\':rate(sum(r[\'prediction\']==\'risk_signal\' for r in faithful),len(faithful)),\\n        \'faithful_surface_accept_rate\':rate(sum(r[\'prediction\']==\'surface_accept\' for r in faithful),len(faithful)),\\n        \'nontrivial_faithful_surface_accept_rate\':rate(sum(r[\'prediction\']==\'surface_accept\' for r in hard),len(hard)),\\n        \'changed_surface_accept_rate\':rate(sum(r[\'prediction\']==\'surface_accept\' for r in changed),len(changed)),\\n        \'abstention_rate\':rate(sum(r[\'prediction\']==\'abstain\' for r in records),len(records)),\\n        \'changed_abstention_rate\':rate(sum(r[\'prediction\']==\'abstain\' for r in changed),len(changed)),\\n        \'faithful_abstention_rate\':rate(sum(r[\'prediction\']==\'abstain\' for r in faithful),len(faithful)),\\n        \'nontrivial_faithful_false_alarm_rate\':rate(sum(r[\'prediction\']==\'risk_signal\' for r in hard),len(hard))}\\n\\n\\ndef passes_gate(metrics,gate):\\n    checks={\\n        \'error_detection\':metrics[\'error_detection_rate\'] is not None and metrics[\'error_detection_rate\']>=gate[\'minimum_error_detection_rate\'],\\n        \'faithful_false_alarms\':metrics[\'faithful_false_alarm_rate\'] is not None and metrics[\'faithful_false_alarm_rate\']<=gate[\'maximum_faithful_false_alarm_rate\'],\\n        \'faithful_acceptance\':metrics[\'faithful_surface_accept_rate\'] is not None and metrics[\'faithful_surface_accept_rate\']>=gate[\'minimum_faithful_surface_accept_rate\'],\\n        \'nontrivial_acceptance\':metrics[\'nontrivial_faithful_surface_accept_rate\'] is not None and metrics[\'nontrivial_faithful_surface_accept_rate\']>=gate[\'minimum_nontrivial_faithful_surface_accept_rate\']}\\n    return {\'passed\':all(checks.values()),\'checks\':checks}\\n\\n\\ndef run(protocol_dir,output):\\n    if output.exists():raise ValueError(\'Use a fresh output directory\')\\n    protocol_path=protocol_dir/\'frozen_protocol.json\';p=json.loads(protocol_path.read_text())\\n    if p[\'protocol\']!=PROTOCOL:raise ValueError(\'Wrong benchmark protocol\')\\n    paths={ROOT/name:h for name,h in p[\'checker_files\'].items()}\\n    paths.update({protocol_dir/name:h for name,h in p[\'fixture_files\'].items()})\\n    if not all(digest(path)==h for path,h in paths.items()):raise ValueError(\'Frozen checker or fixtures changed; define a new benchmark version\')\\n    groups={};cases=[]\\n    for split in (\'development\',\'held_out\'):\\n        rows=json.loads((protocol_dir/(split+\'.json\')).read_text())\\n        if len(rows)!=36 or any(r[\'split\']!=split for r in rows):raise ValueError(\'Wrong benchmark split\')\\n        groups[split]={r[\'scenario_group\'] for r in rows};cases+=rows\\n    if groups[\'development\']&groups[\'held_out\'] or len({r[\'id\'] for r in cases})!=72:raise ValueError(\'Overlapping scenario groups or IDs\')\\n    if {r[\'source\'] for r in cases if r[\'split\']==\'development\'}&{r[\'source\'] for r in cases if r[\'split\']==\'held_out\'}:raise ValueError(\'Leaked source scenario\')\\n    records=[]\\n    # Development evaluated first; no learned thresholds or rule mutations occur.\\n    for c in cases:\\n        audit=claim_checks(c[\'source\'],c[\'output\'])\\n        records.append({**c,\'prediction\':classify(audit),\'audit\':audit})\\n    summary={\'protocol\':PROTOCOL,\'execution_kind\':\'synthetic_checker_evaluation\',\'model_calls\':0,\'gpu_jobs_submitted\':0,\\n        \'reserved_paper_test_rows_used\':0,\'rules_tuned_on_benchmark\':False,\'checker_files\':p[\'checker_files\'],\\n        \'splits\':{split:summarize([r for r in records if r[\'split\']==split]) for split in groups},\\n        \'held_out_by_category\':{cat:summarize([r for r in records if r[\'split\']==\'held_out\' and r[\'category\']==cat]) for cat in sorted({r[\'category\'] for r in records})},\\n        \'provenance\':p[\'provenance\'],\'limitations\':\'Small authored synthetic controls; not blinded, independent annotator labels or economics-domain accuracy. Surface acceptance is not semantic proof. Held-out scenarios have different wording but shared categories.\'}\\n    summary[\'readiness_gate\']=passes_gate(summary[\'splits\'][\'held_out\'],p[\'readiness_gate\'])\\n    for name,choice in ((\'reject_everything\',\'risk_signal\'),(\'abstain_everything\',\'abstain\')):\\n        m=summarize([{**r,\'prediction\':choice} for r in records if r[\'split\']==\'held_out\'])\\n        summary.setdefault(\'degenerate_controls\',{})[name]={\'metrics\':m,\'gate\':passes_gate(m,p[\'readiness_gate\'])}\\n    if not all(digest(path)==h for path,h in paths.items()):raise ValueError(\'Inputs changed during benchmark\')\\n    output.mkdir(parents=True)\\n    for name,data in ((\'results.json\',records),(\'summary.json\',summary)):(output/name).write_text(json.dumps(data,indent=2)+\'\\\\n\')\\n    (output/\'manifest.json\').write_text(json.dumps({\'protocol\':PROTOCOL,\'input_hashes\':{str(path):h for path,h in paths.items()},\\n        \'protocol_sha256\':digest(protocol_path),\'runner_sha256\':digest(Path(__file__)),\\n        \'files\':{name:digest(output/name) for name in (\'results.json\',\'summary.json\')}},indent=2)+\'\\\\n\')\\n    print(json.dumps({\'splits\':summary[\'splits\'],\'readiness_gate\':summary[\'readiness_gate\']},indent=2));return summary\\n\\n\\ndef main(argv=None):\\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--protocol-dir\',type=Path,default=ROOT/\'docs/claim_checker_benchmark_v1\')\\n    p.add_argument(\'--output-dir\',type=Path,required=True);a=p.parse_args(argv);run(a.protocol_dir,a.output_dir)\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/research/claim_audit.py": "\\"\\"\\"CPU-only, evidence-linked claim-risk checks; no general semantic certification.\\n\\nFinite detectors expose scoped counterexamples and abstain on paraphrases. They\\nsupplement saved NLI rather than promote a high score to a semantic guarantee.\\n\\"\\"\\"\\nimport argparse\\nfrom collections import Counter\\nfrom decimal import Decimal\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport re\\n\\nfrom .claims import sentences,sha,verify\\n\\nPROTOCOL=\'saved_claim_audit_v1\'\\nBOUNDS=re.compile(r\'\\\\b(?P<upper>does not exceed|do not exceed|never exceeds?|never more than|no more than|at most|does not surpass|do not surpass)\\\\b|\\\\b(?P<lower>does not fall below|do not fall below|no less than|at least|not less than)\\\\b\',re.I)\\nSTOP=set(\'a an the actual specified predefined pre defined minimum maximum acceptable among successive of in on for to is are was were be by it its this that these those and or but not does do when if under upon condition provided remains remain\'.split())\\nFREQUENCY=re.compile(r\'\\\\b(?:each year|every year|per year|annually|annual)\\\\b\',re.I)\\nTENTATIVE=re.compile(r\'\\\\b(?:may|might|could|appears?|seems?|unlikely|possible|possibly|potentially)\\\\b\',re.I)\\nCERTAIN=re.compile(r\'\\\\b(?:invariably|always|necessarily|must|certainly|definitely)\\\\b\',re.I)\\nCOPULA=re.compile(r\'\\\\b(?P<quantifier>most|many|some|all|every|no)\\\\s+(?P<subject>[^,.;:!?]{1,70}?)\\\\s+(?:are|is|were|was)\\\\s+(?P<property>[^,.;:!?]{1,60})\',re.I)\\nCONCEPTS={\\n    \'predictive_validity\':r\'\\\\b(?:predictive power|predictive validity|predictive accuracy)\\\\b\',\\n    \'systematic_bias\':r\'\\\\b(?:systematic biases?|systematic errors?)\\\\b\',\\n    \'unobserved_mechanism\':r\'\\\\b(?:unobserved (?:dependencies|effects|mechanisms)|causal mechanisms?)\\\\b\',\\n    \'policy_implication\':r\'\\\\b(?:policy implications?|policy recommendations?)\\\\b\',\\n}\\nNUMBER_WORDS=dict(zip(\'one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty\'.split(),range(1,21)))\\n\\n\\ndef content(text):\\n    return {w[:-1] if len(w)>4 and w.endswith(\'s\') and not w.endswith(\'ss\') else w\\n            for w in re.findall(r\'[a-z]+\',text.lower()) if w not in STOP}\\n\\n\\ndef related(left,right):\\n    a,b=content(left),content(right)\\n    return bool(a and a&b) and len(a&b)/len(a)>=.5\\n\\n\\ndef bound_evidence(text):\\n    records=[]\\n    for index,unit in enumerate(sentences(text),1):\\n        for m in BOUNDS.finditer(unit):\\n            prefix=re.split(r\'[;:!?]|\\\\b(?:when|if|provided that)\\\\b\',unit[:m.start()],flags=re.I)[-1]\\n            left=\' \'.join(prefix.split()[-12:]);right=unit[m.end():m.end()+100]\\n            records.append({\'unit_id\':index,\'direction\':\'upper\' if m.group(\'upper\') else \'lower\',\\n                \'cue\':m.group(),\'left_context\':left,\'right_context\':right,\'unit_text\':unit})\\n    return records\\n\\n\\ndef numeric_atoms(text):\\n    # Expand abbreviated year endpoints only for an explicit four-digit year.\\n    def years(m):\\n        start,end=m.groups();century=int(start)//100*100;finish=century+int(end)\\n        if finish<int(start):finish+=100\\n        return start+\' to \'+str(finish)\\n    normalized=re.sub(r\'\\\\b((?:18|19|20)\\\\d{2})\\\\s*[-–—]\\\\s*(\\\\d{2})\\\\b\',years,text)\\n    normalized=re.sub(r\'\\\\b(\'+ \'|\'.join(NUMBER_WORDS)+r\')\\\\b\',lambda m:str(NUMBER_WORDS[m.group().lower()]),normalized,flags=re.I)\\n    return {format(Decimal(m.group()).normalize(),\'f\') for m in re.finditer(r\'\\\\b\\\\d+(?:\\\\.\\\\d+)?\\\\b\',normalized)}\\n\\n\\ndef claim_checks(source,output,nli=None):\\n    source_units=sentences(source);target_units=sentences(output);events=[];obligations=[]\\n    a,b=bound_evidence(source),bound_evidence(output)\\n    for bound in a:\\n        candidates=[x for x in b if related(bound[\'left_context\'],x[\'left_context\'])]\\n        reversed_=[x for x in candidates if x[\'direction\']!=bound[\'direction\']]\\n        obligations.append({\'kind\':\'bound_direction\',\'source\':bound,\\n            \'status\':\'opposite_direction_candidate\' if reversed_ else \'same_direction_surface\' if candidates else \'unresolved\',\\n            \'output_evidence\':candidates})\\n        for x in reversed_:\\n            events.append({\'kind\':\'scoped_bound_reversal_candidate\',\'source_unit_id\':bound[\'unit_id\'],\\n                \'source_evidence\':bound,\'output_evidence\':x,\\n                \'certainty\':\'finite_cue_and_context_match_not_verified_relation_alignment\'})\\n    output_copulas=[(i,m) for i,s in enumerate(target_units,1) for m in COPULA.finditer(s)]\\n    for index,unit in enumerate(source_units,1):\\n        for m in COPULA.finditer(unit):\\n            # Compare the quantified subject AND its property; an occurrence of\\n            # \\"small\\" in a market-value share cannot discharge a firm-size claim.\\n            candidates=[(i,x) for i,x in output_copulas if related(m[\'subject\'],x[\'subject\']) and related(m[\'property\'],x[\'property\'])]\\n            exact=[(i,x) for i,x in candidates if x[\'quantifier\'].lower()==m[\'quantifier\'].lower()]\\n            status=\'same_scoped_surface\' if exact else \'different_quantifier_candidate\' if candidates else \'unresolved_paraphrase_or_omission\'\\n            obligation={\'kind\':\'quantified_property\',\'source_unit_id\':index,\'source_evidence\':m.group(),\\n                \'subject\':m[\'subject\'],\'property\':m[\'property\'],\'quantifier\':m[\'quantifier\'],\'status\':status,\\n                \'output_evidence\':[{\'unit_id\':i,\'text\':x.group()} for i,x in candidates]}\\n            obligations.append(obligation)\\n            if not exact:events.append({\'kind\':\'quantified_property_unresolved\',\'source_unit_id\':index,\'evidence\':obligation,\\n                \'certainty\':\'could_be_paraphrase_not_confirmed_omission\'})\\n        if FREQUENCY.search(unit):\\n            supported=[{\'unit_id\':i,\'text\':s} for i,s in enumerate(target_units,1) if FREQUENCY.search(s) and related(unit,s)]\\n            obligations.append({\'kind\':\'annual_scope\',\'source_unit_id\':index,\'status\':\'same_scoped_surface\' if supported else \'unresolved_paraphrase_or_omission\',\'output_evidence\':supported})\\n            if not supported:events.append({\'kind\':\'annual_scope_unresolved\',\'source_unit_id\':index,\'source_evidence\':unit,\\n                \'certainty\':\'could_be_paraphrase_not_confirmed_omission\'})\\n        if TENTATIVE.search(unit):\\n            stronger=[{\'unit_id\':i,\'text\':s} for i,s in enumerate(target_units,1)\\n                if CERTAIN.search(s) and not TENTATIVE.search(s) and related(unit,s)]\\n            if stronger:events.append({\'kind\':\'modality_strengthening_candidate\',\'source_unit_id\':index,\\n                \'source_evidence\':unit,\'output_evidence\':stronger,\\n                \'certainty\':\'surface_modal_and_context_match_negation_and_modal_scope_not_verified\'})\\n    source_nums,output_nums=numeric_atoms(source),numeric_atoms(output)\\n    for kind,values in ((\'source_quantity_missing_candidate\',source_nums-output_nums),(\'new_quantity_candidate\',output_nums-source_nums)):\\n        for value in sorted(values):events.append({\'kind\':kind,\'canonical_value\':value,\'certainty\':\'numeric_atom_only_roles_units_and_ranges_not_verified\'})\\n    for concept,pattern in CONCEPTS.items():\\n        if re.search(pattern,source,re.I):continue\\n        evidence=[{\'unit_id\':i,\'text\':s} for i,s in enumerate(target_units,1) if re.search(pattern,s,re.I)]\\n        if evidence:events.append({\'kind\':\'new_concept_candidate\',\'concept\':concept,\'output_evidence\':evidence,\\n            \'certainty\':\'lexical_novelty_may_be_valid_paraphrase_not_confirmed_addition\'})\\n    checks=(nli or {}).get(\'checks\',[])\\n    unit_checks={int(r[\'direction\'].split(\'_\')[-1]):r[\'probabilities\'] for r in checks if r[\'direction\'].startswith(\'source_unit_\')}\\n    units=[]\\n    for i,text in enumerate(source_units,1):\\n        p=unit_checks.get(i)\\n        units.append({\'source_unit_id\':i,\'source_evidence\':text,\'saved_nli_probabilities\':p,\\n            \'saved_nli_status\':\'unavailable\' if p is None else \'proxy_consistent\' if p[\'entailment\']>=.8 and p[\'contradiction\']<=.1 else \'proxy_flagged\',\\n            \'event_indices\':[j for j,event in enumerate(events) if event.get(\'source_unit_id\')==i]})\\n    replay=verify(source,output);same=replay[\'status\']==\'surface_preserved\'\\n    return {\'protocol\':PROTOCOL,\'source_sha256\':sha(source),\'output_sha256\':sha(output),\\n        \'status\':\'surface_replay_only\' if same and not events else \'risk_flagged\' if events else \'unresolved\',\\n        \'surface_replay\':replay,\'events\':events,\'obligations\':obligations,\'source_units\':units,\\n        \'saved_nli_status\':nli.get(\'status\') if nli else \'not_available\',\'semantic_proof\':False,\\n        \'limitations\':\'Finite cue detectors and sentence-level cached NLI; no atomic claim, actor/coreference, quantity-role, modality or causal completeness proof. No-events is not a pass. Copular and concept rules abstain on unsupported paraphrases.\'}\\n\\n\\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\\ndef load_lines(path):return [json.loads(x) for x in path.read_text().splitlines() if x.strip()]\\n\\n\\ndef audit_saved(pack_path,result_dir,verification_path,output):\\n    \\"\\"\\"Require existing audited bytes. Never access a model, network or test split.\\"\\"\\"\\n    if output.exists():raise ValueError(\'Use a fresh audit directory; existing audits are immutable\')\\n    verification=json.loads(verification_path.read_text())\\n    if not verification.get(\'checks\') or not all(verification[\'checks\'].values()):raise ValueError(\'Prior verification did not pass\')\\n    for name,digest in verification[\'artifact_hashes\'].items():\\n        if file_sha(result_dir/name)!=digest:raise ValueError(\'Previously audited result changed\')\\n    manifest=json.loads((result_dir/\'manifest.json\').read_text())\\n    status=json.loads((result_dir/\'run_status.json\').read_text())\\n    if status.get(\'complete\') is not True or status.get(\'training_called\') is not False:raise ValueError(\'Expected completed inference run\')\\n    if file_sha(pack_path)!=manifest[\'input_pack_sha256\']:raise ValueError(\'Input pack changed\')\\n    pack=json.loads(pack_path.read_text());rows={r[\'id\']:r for r in pack[\'rows\']}\\n    if len(rows)!=4 or any(r[\'split\']!=\'train\' for r in rows.values()):raise ValueError(\'Expected four frozen training examples\')\\n    metrics=load_lines(result_dir/\'metrics.private.jsonl\');gens=load_lines(result_dir/\'generations.private.jsonl\')\\n    if len(metrics)!=16 or len({(r[\'id\'],r[\'condition\']) for r in metrics})!=16:raise ValueError(\'Incomplete saved metrics\')\\n    # Pure normalization/evaluation of already-generated text, no runtime imports.\\n    from fama_style.rhetoric.claim_constrained import jobs_for,realize\\n    jobs=jobs_for(pack);expected={(j[\'id\'],j[\'condition\']):j for j in jobs}\\n    generated={(r[\'id\'],r[\'condition\']):r for r in gens}\\n    if len(gens)!=16 or set(generated)!=set(expected):raise ValueError(\'Incomplete generations\')\\n    results=[]\\n    for r in metrics:\\n        key=(r[\'id\'],r[\'condition\']);g=generated[key];row=rows[r[\'id\']]\\n        if g[\'prompt_sha256\']!=expected[key][\'prompt_sha256\'] or g[\'hit_output_limit\'] or g[\'synthetic\']:raise ValueError(\'Changed or incomplete generation\')\\n        parsed=realize(g[\'output\'],g[\'condition\'],row[\'plan\'][\'sentence_count\'])\\n        if parsed[\'text\']!=r[\'text\'] or parsed[\'format_valid\']!=r[\'format_valid\']:raise ValueError(\'Saved normalization changed\')\\n        audit=claim_checks(row[\'source\'],r[\'text\'],r[\'nli\'])\\n        results.append({\'id\':r[\'id\'],\'condition\':r[\'condition\'],\'format_valid\':r[\'format_valid\'],**audit})\\n    conditions={}\\n    for condition in sorted({r[\'condition\'] for r in results}):\\n        group=[r for r in results if r[\'condition\']==condition];types=Counter(event[\'kind\'] for r in group for event in r[\'events\'])\\n        conditions[condition]={\'rows\':len(group),\'status_counts\':dict(Counter(r[\'status\'] for r in group)),\\n            \'outputs_with_event\':{kind:sum(any(x[\'kind\']==kind for x in r[\'events\']) for r in group) for kind in sorted(types)},\\n            \'saved_nli_pass_with_events\':sum(r[\'saved_nli_status\']==\'proxy_consistent\' and bool(r[\'events\']) for r in group),\\n            \'invalid_format_rows_included\':sum(not r[\'format_valid\'] for r in group)}\\n    summary={\'protocol\':PROTOCOL,\'rows\':16,\'conditions\':conditions,\'model_calls\':0,\'gpu_jobs_submitted\':0,\\n        \'held_out_rows_used\':0,\'all_claims_verified\':False,\'semantic_accuracy\':None,\\n        \'limitations\':\'Post-hoc diagnostic on four inspected training pairs, not an independent validation set. Risk counts are neither confirmed error counts nor precision/recall. A conservative replay is not general semantic certification.\'}\\n    output.mkdir(parents=True)\\n    (output/\'claims.private.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\\\n\' for r in results))\\n    (output/\'summary.json\').write_text(json.dumps(summary,indent=2)+\'\\\\n\')\\n    inputs={str(p):file_sha(p) for p in (pack_path,verification_path,*[result_dir/n for n in verification[\'artifact_hashes\']])}\\n    for name,digest in verification[\'artifact_hashes\'].items():\\n        if file_sha(result_dir/name)!=digest:raise ValueError(\'Result changed during audit\')\\n    if file_sha(pack_path)!=manifest[\'input_pack_sha256\']:raise ValueError(\'Input pack changed during audit\')\\n    (output/\'manifest.json\').write_text(json.dumps({\'protocol\':PROTOCOL,\'input_hashes\':inputs,\\n        \'code_sha256\':file_sha(Path(__file__)),\'files\':{n:file_sha(output/n) for n in (\'claims.private.jsonl\',\'summary.json\')},\\n        \'model_calls\':0,\'training_called\':False},indent=2)+\'\\\\n\')\\n    print(json.dumps(summary,indent=2));return summary\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--pack\',type=Path,required=True);parser.add_argument(\'--results\',type=Path,required=True)\\n    parser.add_argument(\'--verification\',type=Path,required=True);parser.add_argument(\'--output-dir\',type=Path,required=True)\\n    a=parser.parse_args(argv);audit_saved(a.pack,a.results,a.verification,a.output_dir)\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/research/claim_verifier.py": "\\"\\"\\"Claim-unit NLI and bounded role guards. Calibration only; no generation/training.\\"\\"\\"\\nimport argparse\\nfrom collections import Counter\\nfrom decimal import Decimal\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport time\\n\\nfrom .claim_audit import claim_checks,NUMBER_WORDS\\nfrom .claims import sentences,sha\\nfrom fama_style.rhetoric.semantics import assess,checked_prediction,NLI_MODEL\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'claim_verifier_calibration_v2\'\\nNLI_REVISION=\'6f5cf0a2b59cabb106aca4c287eed12e357e90eb\'\\nMAX_UNITS=12\\nMAX_PAIRS=256\\nNUMBER=r\'(?:\\\\d+(?:\\\\.\\\\d+)?|\'+ \'|\'.join(NUMBER_WORDS)+\')\'\\nACTOR=r\'(?:the\\\\s+)?[a-z]+(?:\\\\s+[a-z]+){0,3}?\'\\nQ_ACTIVE=re.compile(r\'(?P<actor>\'+ACTOR+r\')\\\\s+(?:owns?|holds?)\\\\s+(?P<number>\'+NUMBER+r\')\\\\s+(?P<unit>shares?|bonds?|notes?|units?)\',re.I)\\nQ_PASSIVE=re.compile(r\'(?P<number>\'+NUMBER+r\')\\\\s+(?P<unit>shares?|bonds?|notes?|units?)\\\\s+(?:are|is)\\\\s+(?:owned|held)\\\\s+by\\\\s+(?P<actor>\'+ACTOR+r\')\',re.I)\\nPAY_ACTIVE=re.compile(r\'(?P<actor>\'+ACTOR+r\')\\\\s+(?P<verb>pays?|reimburses?)\\\\s+(?P<recipient>\'+ACTOR+r\')\',re.I)\\nPAY_PASSIVE=re.compile(r\'(?P<recipient>\'+ACTOR+r\')\\\\s+(?:is|are)\\\\s+(?P<verb>paid|reimbursed)\\\\s+by\\\\s+(?P<actor>\'+ACTOR+r\')\',re.I)\\n\\n\\ndef clean_actor(value):return re.sub(r\'^the\\\\s+\',\'\',value.lower().strip())\\ndef atom(value):return format(Decimal(str(NUMBER_WORDS.get(value.lower(),value))).normalize(),\'f\')\\n\\n\\ndef explicit_actor(actor):\\n    blocked={\'it\',\'they\',\'he\',\'she\',\'we\',\'may\',\'might\',\'could\',\'must\',\'should\',\'will\',\'would\',\'not\',\'no\',\'all\',\'every\',\'some\',\'many\',\'most\',\'if\',\'unless\',\'when\',\'because\',\'that\',\'which\'}\\n    return bool(actor) and not (set(actor.split())&blocked)\\n\\n\\ndef role_fact(unit):\\n    body=unit.strip().rstrip(\'.!?\').strip()\\n    for pattern in (Q_ACTIVE,Q_PASSIVE):\\n        m=pattern.fullmatch(body)\\n        if m:\\n            actor=clean_actor(m[\'actor\'])\\n            if not explicit_actor(actor):return None\\n            return {\'relation\':\'holding\',\'actor\':actor,\'unit\':m[\'unit\'].lower().rstrip(\'s\'),\'value\':atom(m[\'number\'])}\\n    for pattern in (PAY_ACTIVE,PAY_PASSIVE):\\n        m=pattern.fullmatch(body)\\n        if m:\\n            actor,recipient=clean_actor(m[\'actor\']),clean_actor(m[\'recipient\'])\\n            if not explicit_actor(actor) or not explicit_actor(recipient):return None\\n            return {\'relation\':\'reimburse\' if m[\'verb\'].lower().startswith(\'reimburs\') else \'pay\',\'actor\':actor,\'recipient\':recipient}\\n    return None\\n\\n\\ndef units(text):\\n    \\"\\"\\"Sentence spans; split coordination only if both sides fully parse as facts.\\n\\n    No claim of general atomic extraction. Conditions and causal links stay intact.\\n    \\"\\"\\"\\n    result=[]\\n    for si,sentence in enumerate(sentences(text),1):\\n        pieces=re.split(r\';\\\\s*|\\\\s+(?:and|while)\\\\s+\',sentence)\\n        parsed=[role_fact(p) for p in pieces]\\n        if len(pieces)<2 or not all(parsed):pieces=[sentence]\\n        for part in pieces:result.append({\'id\':\'u\'+str(len(result)+1),\'sentence_id\':si,\'text\':part.strip()})\\n    return result\\n\\n\\ndef role_guards(source_units,output_units):\\n    a=[{\'unit_id\':u[\'id\'],\'fact\':role_fact(u[\'text\'])} for u in source_units];b=[{\'unit_id\':u[\'id\'],\'fact\':role_fact(u[\'text\'])} for u in output_units]\\n    a=[r for r in a if r[\'fact\']];b=[r for r in b if r[\'fact\']];conflicts=[]\\n    # Only compare fully parsed fact groups with the same relation and entity set.\\n    for relation in (\'holding\',\'pay\',\'reimburse\'):\\n        left=[r for r in a if r[\'fact\'][\'relation\']==relation];right=[r for r in b if r[\'fact\'][\'relation\']==relation]\\n        if not left or not right:continue\\n        def entities(group):\\n            return {v for r in group for k,v in r[\'fact\'].items() if k in (\'actor\',\'recipient\')}\\n        if len(left)!=len(right) or entities(left)!=entities(right):continue\\n        canon=lambda group:Counter(json.dumps(r[\'fact\'],sort_keys=True) for r in group)\\n        if canon(left)!=canon(right):\\n            conflicts.append({\'kind\':\'quantity_role_conflict\' if relation==\'holding\' else \'actor_role_conflict\',\'source_facts\':left,\'output_facts\':right,\\n                \'scope\':\'Fully parsed finite grammar; not general entity/coreference resolution\'})\\n    return conflicts\\n\\n\\ndef bundles(parts):\\n    result=[{\'unit_ids\':[p[\'id\']],\'text\':p[\'text\']} for p in parts]\\n    result += [{\'unit_ids\':[a[\'id\'],b[\'id\']],\'text\':a[\'text\'].rstrip(\'.!?\')+\'. \'+b[\'text\']} for a,b in zip(parts,parts[1:])]\\n    return result\\n\\n\\ndef verify(source,output,predict):\\n    a,b=units(source),units(output)\\n    conflicts=role_guards(a,b)\\n    surface=claim_checks(source,output)\\n    conflicts += [e for e in surface[\'events\'] if e[\'kind\']==\'scoped_bound_reversal_candidate\']\\n    if not source.strip() or not output.strip() or max(len(a),len(b))>MAX_UNITS:\\n        return {\'status\':\'abstain\',\'reason\':\'empty_or_unit_budget\',\'semantic_proof\':False,\'role_conflicts\':conflicts}\\n    cache={}\\n    def cached(premise,hypothesis):\\n        key=(sha(premise),sha(hypothesis))\\n        if key not in cache:\\n            if len(cache)>=MAX_PAIRS:return None\\n            cache[key]=checked_prediction(predict,premise,hypothesis)\\n        return cache[key]\\n    global_check=assess(source,output,cached);alignments=[]\\n    # A best matching local unit or adjacent bundle supports each candidate claim.\\n    # Both directions matter: source coverage and output support are separate.\\n    for direction,claims,evidence in ((\'source_coverage\',a,b),(\'output_support\',b,a)):\\n        candidates=bundles(evidence)\\n        for claim in claims:\\n            pairs=[{\'evidence_unit_ids\':c[\'unit_ids\'],\'probabilities\':cached(c[\'text\'],claim[\'text\'])} for c in candidates]\\n            known=[p for p in pairs if p[\'probabilities\'] is not None]\\n            best=max(known,key=lambda p:p[\'probabilities\'][\'entailment\']) if known else None\\n            alignments.append({\'direction\':direction,\'claim_unit_id\':claim[\'id\'],\'claim_text\':claim[\'text\'],\\n                \'candidates\':pairs,\'best_evidence_unit_ids\':best[\'evidence_unit_ids\'] if best else None,\\n                \'status\':\'abstain\' if len(known)!=len(pairs) else \'supported_proxy\' if best[\'probabilities\'][\'entailment\']>=.8 and best[\'probabilities\'][\'contradiction\']<=.1 else \'flagged_proxy\'})\\n    unknown=global_check[\'status\']==\'unknown_over_context\' or any(r[\'status\']==\'abstain\' for r in alignments)\\n    flagged=global_check[\'status\']==\'proxy_flagged\' or any(r[\'status\']==\'flagged_proxy\' for r in alignments)\\n    status=\'risk_signal\' if conflicts else \'abstain\' if unknown else \'risk_signal\' if flagged else \'model_consistent\'\\n    return {\'status\':status,\'semantic_proof\':False,\'global_nli\':global_check,\'alignments\':alignments,\\n        \'role_conflicts\':conflicts,\'other_surface_warnings\':surface[\'events\'],\'source_units\':a,\'output_units\':b,\\n        \'prediction_pairs\':len(cache),\'limitations\':\'Candidate sentence/clause units, finite role grammar and fallible NLI. Model-consistent is not verified truth; no general claim extraction, coreference or scope guarantee.\'}\\n\\n\\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef validate_protocol(folder):\\n    p=json.loads((folder/\'protocol.json\').read_text())\\n    if p[\'protocol\']!=PROTOCOL:raise ValueError(\'Wrong calibration protocol\')\\n    for name,h in p[\'code_files\'].items():\\n        if file_sha(ROOT/name)!=h:raise ValueError(\'Frozen verifier source changed; create a new calibration version\')\\n    for name,h in p[\'fixture_files\'].items():\\n        if file_sha(folder/name)!=h:raise ValueError(\'Frozen synthetic cases changed\')\\n    cases=[]\\n    for phase in (\'development\',\'held_out\'):\\n        group=json.loads((folder/(phase+\'.json\')).read_text())\\n        if len(group)!=p[\'expected_counts\'][phase] or any(c[\'split\']!=phase for c in group):raise ValueError(\'Wrong split size or label\')\\n        if any(c[\'expected\'] not in (\'faithful\',\'changed\') or c[\'variant\'] not in (\'identity\',\'paraphrase\',\'mutation\') for c in group):raise ValueError(\'Invalid case labels\')\\n        cases+=group\\n    if len({c[\'id\'] for c in cases})!=len(cases):raise ValueError(\'Duplicate case IDs\')\\n    for key in (\'source\',\'scenario_group\'):\\n        if {c[key] for c in cases if c[\'split\']==\'development\'}&{c[key] for c in cases if c[\'split\']==\'held_out\'}:raise ValueError(\'Calibration split leakage\')\\n    return p,cases\\n\\n\\ndef score(records):\\n    good=[r for r in records if r[\'expected\']==\'faithful\'];bad=[r for r in records if r[\'expected\']==\'changed\'];hard=[r for r in good if r[\'variant\']==\'paraphrase\']\\n    rate=lambda rows,status:sum(r[\'verdict\'][\'status\']==status for r in rows)/len(rows) if rows else None\\n    return {\'rows\':len(records),\'faithful_rows\':len(good),\'changed_rows\':len(bad),\'error_detection_rate\':rate(bad,\'risk_signal\'),\\n        \'faithful_false_alarm_rate\':rate(good,\'risk_signal\'),\'nontrivial_faithful_acceptance_rate\':rate(hard,\'model_consistent\'),\\n        \'changed_false_acceptance_rate\':rate(bad,\'model_consistent\'),\'abstention_rate\':rate(records,\'abstain\'),\\n        \'status_counts\':dict(Counter(r[\'verdict\'][\'status\'] for r in records))}\\n\\n\\ndef gate(metrics,criteria):\\n    checks={\\n        \'error_detection\':metrics[\'error_detection_rate\'] is not None and metrics[\'error_detection_rate\']>=criteria[\'minimum_error_detection_rate\'],\\n        \'faithful_false_alarms\':metrics[\'faithful_false_alarm_rate\'] is not None and metrics[\'faithful_false_alarm_rate\']<=criteria[\'maximum_faithful_false_alarm_rate\'],\\n        \'paraphrase_acceptance\':metrics[\'nontrivial_faithful_acceptance_rate\'] is not None and metrics[\'nontrivial_faithful_acceptance_rate\']>=criteria[\'minimum_nontrivial_faithful_acceptance_rate\'],\\n        \'changed_false_acceptance\':metrics[\'changed_false_acceptance_rate\'] is not None and metrics[\'changed_false_acceptance_rate\']<=criteria[\'maximum_changed_false_acceptance_rate\']}\\n    return {\'passed\':all(checks.values()),\'checks\':checks}\\n\\n\\ndef evaluate(folder,output,predict,execution_kind):\\n    if output.exists():raise ValueError(\'Use a fresh output directory\')\\n    p,cases=validate_protocol(folder);output.mkdir(parents=True);records=[];started=time.monotonic()\\n    (output/\'run_status.json\').write_text(json.dumps({\'complete\':False,\'completed_cases\':0,\'total_cases\':len(cases),\'training_called\':False})+\'\\\\n\')\\n    for i,c in enumerate(cases,1):\\n        records.append({**c,\'verdict\':verify(c[\'source\'],c[\'output\'],predict)})\\n        (output/\'partial_results.json\').write_text(json.dumps(records,indent=2)+\'\\\\n\')\\n        (output/\'run_status.json\').write_text(json.dumps({\'complete\':False,\'completed_cases\':i,\'total_cases\':len(cases),\'training_called\':False})+\'\\\\n\')\\n        print(\'Assessed synthetic case \'+str(i)+\'/\'+str(len(cases)),flush=True)\\n    summary={\'protocol\':PROTOCOL,\'execution_kind\':execution_kind,\'training_called\':False,\'gemma_generation_called\':False,\\n        \'private_dataset_rows_used\':0,\'splits\':{s:score([r for r in records if r[\'split\']==s]) for s in (\'development\',\'held_out\')},\\n        \'held_out_by_category\':{k:score([r for r in records if r[\'split\']==\'held_out\' and r[\'category\']==k]) for k in sorted({r[\'category\'] for r in records if r[\'split\']==\'held_out\'})},\\n        \'elapsed_seconds\':time.monotonic()-started,\'semantic_proof\':False,\\n        \'limitations\':\'Authored synthetic controls, not blinded or independently annotated domain accuracy. V1 examples are development-only; only fresh cases are held out.\'}\\n    summary[\'readiness_gate\']=gate(summary[\'splits\'][\'held_out\'],p[\'readiness_gate\']) if execution_kind==\'pinned_nli_inference\' else {\'passed\':False,\'reason\':\'Synthetic stub execution cannot validate a verifier\'}\\n    baseline=[{**r,\'verdict\':{\'status\':{\'proxy_consistent\':\'model_consistent\',\'proxy_flagged\':\'risk_signal\'}.get(r[\'verdict\'].get(\'global_nli\',{}).get(\'status\'),\'abstain\')}} for r in records]\\n    summary[\'baseline_global_sentence_nli\']={s:score([r for r in baseline if r[\'split\']==s]) for s in (\'development\',\'held_out\')}\\n    summary[\'held_out_baseline_passes_overridden\']=sum(r[\'split\']==\'held_out\' and r[\'verdict\'].get(\'global_nli\',{}).get(\'status\')==\'proxy_consistent\' and r[\'verdict\'][\'status\']==\'risk_signal\' for r in records)\\n    validate_protocol(folder)\\n    (output/\'results.json\').write_text(json.dumps(records,indent=2)+\'\\\\n\');(output/\'evaluation.json\').write_text(json.dumps(summary,indent=2)+\'\\\\n\')\\n    (output/\'run_status.json\').write_text(json.dumps({\'complete\':True,\'completed_cases\':len(cases),\'total_cases\':len(cases),\'training_called\':False})+\'\\\\n\')\\n    (output/\'manifest.json\').write_text(json.dumps({\'protocol\':PROTOCOL,\'protocol_sha256\':file_sha(folder/\'protocol.json\'),\\n        \'code_files\':p[\'code_files\'],\'fixture_files\':p[\'fixture_files\'],\\n        \'files\':{n:file_sha(output/n) for n in (\'results.json\',\'evaluation.json\',\'run_status.json\')}},indent=2)+\'\\\\n\')\\n    return summary\\n\\n\\ndef run_training(folder,output):\\n    \\"\\"\\"Pretrained NLI calibration only; lazy ML imports, no optimizer or trainer.\\"\\"\\"\\n    validate_protocol(folder)\\n    import os\\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\';os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\'\\n    import torch\\n    from transformers import AutoTokenizer,AutoModelForSequenceClassification\\n    from transformers.utils import logging\\n    from fama_style.rhetoric.experiment import with_heartbeat\\n    logging.set_verbosity_error()\\n    if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Select T4 x2 and expose GPU 0 before importing torch\')\\n    torch.cuda.set_device(0);torch.manual_seed(42);torch.cuda.reset_peak_memory_stats(0)\\n    print(\'Loading pinned public NLI model; no Secret required.\',flush=True)\\n    tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n    model=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False,dtype=torch.float32).to(\'cuda:0\'),\'NLI model load\')\\n    model.eval();model.requires_grad_(False);labels={int(k):v.lower() for k,v in model.config.id2label.items()}\\n    if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n    limit=min(512,int(tokenizer.model_max_length),int(getattr(model.config,\'max_position_embeddings\',512)));cache={}\\n    def predict(premise,hypothesis):\\n        key=(sha(premise),sha(hypothesis))\\n        if key in cache:return cache[key]\\n        batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n        if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n        with torch.inference_mode():\\n            values=torch.softmax(model(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n        cache[key]={labels[i]:float(v) for i,v in enumerate(values)};return cache[key]\\n    summary=evaluate(folder,output,predict,\'pinned_nli_inference\')\\n    runtime={\'model_id\':NLI_MODEL,\'model_revision\':NLI_REVISION,\'dtype\':\'float32\',\'device\':torch.cuda.get_device_name(0),\'context_limit\':limit,\\n        \'cache_entries\':len(cache),\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\\n        \'versions\':{k:importlib.metadata.version(k) for k in (\'torch\',\'transformers\',\'huggingface-hub\')},\'training_called\':False}\\n    (output/\'runtime.json\').write_text(json.dumps(runtime,indent=2)+\'\\\\n\')\\n    manifest=json.loads((output/\'manifest.json\').read_text());manifest[\'files\'][\'runtime.json\']=file_sha(output/\'runtime.json\')\\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\\\n\')\\n    print(json.dumps({\'readiness_gate\':summary[\'readiness_gate\'],\'splits\':summary[\'splits\']},indent=2))\\n\\n\\ndef main(argv=None):\\n    p=argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\'--protocol-dir\',type=Path,default=ROOT/\'docs/claim_verifier_calibration_v2\')\\n    p.add_argument(\'--output-dir\',type=Path,required=True);p.add_argument(\'--execute-calibration\',action=\'store_true\')\\n    a=p.parse_args(argv)\\n    if a.execute_calibration:run_training(a.protocol_dir,a.output_dir)\\n    else:\\n        protocol,cases=validate_protocol(a.protocol_dir)\\n        if a.output_dir.exists():raise ValueError(\'Use a fresh output directory\')\\n        a.output_dir.mkdir(parents=True)\\n        (a.output_dir/\'preflight.json\').write_text(json.dumps({\'protocol\':PROTOCOL,\'cases\':len(cases),\'model_calls\':0,\'training_called\':False,\'execution_kind\':\'preflight_only\'},indent=2)+\'\\\\n\')\\n        print(\'Preflight passed. No model download, Secret access, generation or training.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/research/claim_verifier_v3.py": "\\"\\"\\"Frozen v3: scoped endpoint/causality guards over the unchanged v2 verifier.\\"\\"\\"\\nimport argparse\\nfrom collections import Counter\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport time\\n\\nfrom . import claim_verifier as base\\nfrom .relation_guards import check\\nfrom .claims import sha,sentences\\nfrom fama_style.rhetoric.semantics import checked_prediction,NLI_MODEL\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'claim_verifier_calibration_v3\'\\nNLI_REVISION=base.NLI_REVISION\\n\\n\\ndef verify(source,output,predict):\\n    previous=base.verify(source,output,predict)\\n    evidence=check(source,output);status=previous[\'status\']\\n    if evidence[\'conflicts\']:status=\'risk_signal\'\\n    elif evidence[\'unresolved\'] and status==\'model_consistent\':status=\'abstain\'\\n    return {**previous,\'v2_status\':previous[\'status\'],\'status\':status,\'scoped_relation_guards\':evidence}\\n\\n\\ndef replay_v2(records):\\n    \\"\\"\\"Reuse saved probabilities as development regressions, never fresh evidence.\\"\\"\\"\\n    results=[]\\n    for row in records:\\n        old=row[\'verdict\'];source,output=row[\'source\'],row[\'output\'];cache={}\\n        def add(a,b,p):\\n            key=(sha(a),sha(b))\\n            if key in cache and cache[key]!=p:raise ValueError(\'Inconsistent cached probabilities\')\\n            cache[key]=p\\n        pairs=[(source,output),(output,source)]+[(output,s) for s in sentences(source)]+[(source,s) for s in sentences(output)]\\n        if len(pairs)!=len(old[\'global_nli\'][\'checks\']):raise ValueError(\'Incomplete global probabilities\')\\n        for (a,b),r in zip(pairs,old[\'global_nli\'][\'checks\']):add(a,b,r[\'probabilities\'])\\n        for alignment in old[\'alignments\']:\\n            evidence=old[\'output_units\'] if alignment[\'direction\']==\'source_coverage\' else old[\'source_units\']\\n            candidates={tuple(b[\'unit_ids\']):b[\'text\'] for b in base.bundles(evidence)}\\n            for c in alignment[\'candidates\']:add(candidates[tuple(c[\'evidence_unit_ids\'])],alignment[\'claim_text\'],c[\'probabilities\'])\\n        result=verify(source,output,lambda a,b:cache[(sha(a),sha(b))])\\n        if result[\'v2_status\']!=old[\'status\']:raise ValueError(\'V2 baseline changed\')\\n        results.append({**row,\'original_v2_split\':row[\'split\'],\'split\':\'development\',\'verdict\':result})\\n    return results\\n\\n\\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef validate_protocol(folder):\\n    p=json.loads((folder/\'protocol.json\').read_text())\\n    if p[\'protocol\']!=PROTOCOL:raise ValueError(\'Wrong calibration protocol\')\\n    for name,h in p[\'code_files\'].items():\\n        if file_sha(ROOT/name)!=h:raise ValueError(\'Frozen verifier source changed; create a new calibration version\')\\n    for name,h in p[\'fixture_files\'].items():\\n        if file_sha(folder/name)!=h:raise ValueError(\'Frozen synthetic cases changed\')\\n    cases=[]\\n    for phase in (\'development\',\'held_out\'):\\n        group=json.loads((folder/(phase+\'.json\')).read_text())\\n        if len(group)!=p[\'expected_counts\'][phase] or any(c[\'split\']!=phase for c in group):raise ValueError(\'Wrong split size or label\')\\n        if any(c[\'expected\'] not in (\'faithful\',\'changed\') or c[\'variant\'] not in (\'identity\',\'paraphrase\',\'mutation\') for c in group):raise ValueError(\'Invalid case labels\')\\n        cases+=group\\n    if len({c[\'id\'] for c in cases})!=len(cases):raise ValueError(\'Duplicate case IDs\')\\n    for key in (\'source\',\'scenario_group\'):\\n        if {c[key] for c in cases if c[\'split\']==\'development\'}&{c[key] for c in cases if c[\'split\']==\'held_out\'}:raise ValueError(\'Calibration split leakage\')\\n    return p,cases\\n\\n\\ndef score(records):\\n    good=[r for r in records if r[\'expected\']==\'faithful\'];bad=[r for r in records if r[\'expected\']==\'changed\'];hard=[r for r in good if r[\'variant\']==\'paraphrase\']\\n    rate=lambda rows,status:sum(r[\'verdict\'][\'status\']==status for r in rows)/len(rows) if rows else None\\n    return {\'rows\':len(records),\'faithful_rows\':len(good),\'changed_rows\':len(bad),\'error_detection_rate\':rate(bad,\'risk_signal\'),\\n        \'faithful_false_alarm_rate\':rate(good,\'risk_signal\'),\'nontrivial_faithful_acceptance_rate\':rate(hard,\'model_consistent\'),\\n        \'changed_false_acceptance_rate\':rate(bad,\'model_consistent\'),\'abstention_rate\':rate(records,\'abstain\'),\\n        \'status_counts\':dict(Counter(r[\'verdict\'][\'status\'] for r in records))}\\n\\n\\ndef gate(metrics,criteria):\\n    checks={\\n        \'error_detection\':metrics[\'error_detection_rate\'] is not None and metrics[\'error_detection_rate\']>=criteria[\'minimum_error_detection_rate\'],\\n        \'faithful_false_alarms\':metrics[\'faithful_false_alarm_rate\'] is not None and metrics[\'faithful_false_alarm_rate\']<=criteria[\'maximum_faithful_false_alarm_rate\'],\\n        \'paraphrase_acceptance\':metrics[\'nontrivial_faithful_acceptance_rate\'] is not None and metrics[\'nontrivial_faithful_acceptance_rate\']>=criteria[\'minimum_nontrivial_faithful_acceptance_rate\'],\\n        \'changed_false_acceptance\':metrics[\'changed_false_acceptance_rate\'] is not None and metrics[\'changed_false_acceptance_rate\']<=criteria[\'maximum_changed_false_acceptance_rate\']}\\n    return {\'passed\':all(checks.values()),\'checks\':checks}\\n\\n\\ndef evaluate(folder,output,predict,execution_kind):\\n    if output.exists():raise ValueError(\'Use a fresh output directory\')\\n    p,cases=validate_protocol(folder);output.mkdir(parents=True);records=[];started=time.monotonic()\\n    (output/\'run_status.json\').write_text(json.dumps({\'complete\':False,\'completed_cases\':0,\'total_cases\':len(cases),\'training_called\':False})+\'\\\\n\')\\n    for i,c in enumerate(cases,1):\\n        records.append({**c,\'verdict\':verify(c[\'source\'],c[\'output\'],predict)})\\n        (output/\'partial_results.json\').write_text(json.dumps(records,indent=2)+\'\\\\n\')\\n        (output/\'run_status.json\').write_text(json.dumps({\'complete\':False,\'completed_cases\':i,\'total_cases\':len(cases),\'training_called\':False})+\'\\\\n\')\\n        print(\'Assessed synthetic case \'+str(i)+\'/\'+str(len(cases)),flush=True)\\n    summary={\'protocol\':PROTOCOL,\'execution_kind\':execution_kind,\'training_called\':False,\'gemma_generation_called\':False,\\n        \'private_dataset_rows_used\':0,\'splits\':{s:score([r for r in records if r[\'split\']==s]) for s in (\'development\',\'held_out\')},\\n        \'held_out_by_category\':{k:score([r for r in records if r[\'split\']==\'held_out\' and r[\'category\']==k]) for k in sorted({r[\'category\'] for r in records if r[\'split\']==\'held_out\'})},\\n        \'elapsed_seconds\':time.monotonic()-started,\'semantic_proof\':False,\\n        \'limitations\':\'Authored synthetic controls, not blinded or independently annotated domain accuracy. All previously exposed v1/v2 examples are development-only; only fresh v3 cases are held out.\'}\\n    summary[\'readiness_gate\']=gate(summary[\'splits\'][\'held_out\'],p[\'readiness_gate\']) if execution_kind==\'pinned_nli_inference\' else {\'passed\':False,\'reason\':\'Synthetic stub execution cannot validate a verifier\'}\\n    baseline=[{**r,\'verdict\':{\'status\':{\'proxy_consistent\':\'model_consistent\',\'proxy_flagged\':\'risk_signal\'}.get(r[\'verdict\'].get(\'global_nli\',{}).get(\'status\'),\'abstain\')}} for r in records]\\n    summary[\'baseline_global_sentence_nli\']={s:score([r for r in baseline if r[\'split\']==s]) for s in (\'development\',\'held_out\')}\\n    summary[\'held_out_baseline_passes_overridden\']=sum(r[\'split\']==\'held_out\' and r[\'verdict\'].get(\'global_nli\',{}).get(\'status\')==\'proxy_consistent\' and r[\'verdict\'][\'status\']==\'risk_signal\' for r in records)\\n    v2_baseline=[{**r,\'verdict\':{\'status\':r[\'verdict\'][\'v2_status\']}} for r in records]\\n    summary[\'baseline_v2_claim_verifier\']={s:score([r for r in v2_baseline if r[\'split\']==s]) for s in (\'development\',\'held_out\')}\\n    summary[\'held_out_v2_passes_overridden\']=sum(r[\'split\']==\'held_out\' and r[\'verdict\'][\'v2_status\']==\'model_consistent\' and r[\'verdict\'][\'status\']==\'risk_signal\' for r in records)\\n    summary[\'held_out_v2_passes_abstained\']=sum(r[\'split\']==\'held_out\' and r[\'verdict\'][\'v2_status\']==\'model_consistent\' and r[\'verdict\'][\'status\']==\'abstain\' for r in records)\\n    validate_protocol(folder)\\n    (output/\'results.json\').write_text(json.dumps(records,indent=2)+\'\\\\n\');(output/\'evaluation.json\').write_text(json.dumps(summary,indent=2)+\'\\\\n\')\\n    (output/\'run_status.json\').write_text(json.dumps({\'complete\':True,\'completed_cases\':len(cases),\'total_cases\':len(cases),\'training_called\':False})+\'\\\\n\')\\n    (output/\'manifest.json\').write_text(json.dumps({\'protocol\':PROTOCOL,\'protocol_sha256\':file_sha(folder/\'protocol.json\'),\\n        \'code_files\':p[\'code_files\'],\'fixture_files\':p[\'fixture_files\'],\\n        \'files\':{n:file_sha(output/n) for n in (\'results.json\',\'evaluation.json\',\'run_status.json\')}},indent=2)+\'\\\\n\')\\n    return summary\\n\\n\\ndef run_training(folder,output):\\n    \\"\\"\\"Pretrained NLI calibration only; lazy ML imports, no optimizer or trainer.\\"\\"\\"\\n    validate_protocol(folder)\\n    import os\\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\';os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\'\\n    import torch\\n    from transformers import AutoTokenizer,AutoModelForSequenceClassification\\n    from transformers.utils import logging\\n    from fama_style.rhetoric.experiment import with_heartbeat\\n    logging.set_verbosity_error()\\n    if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Select T4 x2 and expose GPU 0 before importing torch\')\\n    torch.cuda.set_device(0);torch.manual_seed(42);torch.cuda.reset_peak_memory_stats(0)\\n    print(\'Loading pinned public NLI model; no Secret required.\',flush=True)\\n    tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n    model=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False,dtype=torch.float32).to(\'cuda:0\'),\'NLI model load\')\\n    model.eval();model.requires_grad_(False);labels={int(k):v.lower() for k,v in model.config.id2label.items()}\\n    if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n    limit=min(512,int(tokenizer.model_max_length),int(getattr(model.config,\'max_position_embeddings\',512)));cache={}\\n    def predict(premise,hypothesis):\\n        key=(sha(premise),sha(hypothesis))\\n        if key in cache:return cache[key]\\n        batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n        if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n        with torch.inference_mode():\\n            values=torch.softmax(model(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n        cache[key]={labels[i]:float(v) for i,v in enumerate(values)};return cache[key]\\n    summary=evaluate(folder,output,predict,\'pinned_nli_inference\')\\n    runtime={\'model_id\':NLI_MODEL,\'model_revision\':NLI_REVISION,\'dtype\':\'float32\',\'device\':torch.cuda.get_device_name(0),\'context_limit\':limit,\\n        \'cache_entries\':len(cache),\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\\n        \'versions\':{k:importlib.metadata.version(k) for k in (\'torch\',\'transformers\',\'huggingface-hub\')},\'training_called\':False}\\n    (output/\'runtime.json\').write_text(json.dumps(runtime,indent=2)+\'\\\\n\')\\n    manifest=json.loads((output/\'manifest.json\').read_text());manifest[\'files\'][\'runtime.json\']=file_sha(output/\'runtime.json\')\\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\\\n\')\\n    print(json.dumps({\'readiness_gate\':summary[\'readiness_gate\'],\'splits\':summary[\'splits\']},indent=2))\\n\\n\\ndef main(argv=None):\\n    p=argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\'--protocol-dir\',type=Path,default=ROOT/\'docs/claim_verifier_calibration_v3\')\\n    p.add_argument(\'--output-dir\',type=Path,required=True);p.add_argument(\'--execute-calibration\',action=\'store_true\')\\n    a=p.parse_args(argv)\\n    if a.execute_calibration:run_training(a.protocol_dir,a.output_dir)\\n    else:\\n        protocol,cases=validate_protocol(a.protocol_dir)\\n        if a.output_dir.exists():raise ValueError(\'Use a fresh output directory\')\\n        a.output_dir.mkdir(parents=True)\\n        (a.output_dir/\'preflight.json\').write_text(json.dumps({\'protocol\':PROTOCOL,\'cases\':len(cases),\'model_calls\':0,\'training_called\':False,\'execution_kind\':\'preflight_only\'},indent=2)+\'\\\\n\')\\n        print(\'Preflight passed. No model download, Secret access, generation or training.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/research/claims.py": "\\"\\"\\"Evidence-backed surface ledger and conservative rewrite verification.\\n\\nThis is a bounded English-rule prototype, not a semantic parser or NLI model.\\nUnrecognized paraphrases abstain; matching protected words alone never passes.\\n\\"\\"\\"\\nimport collections\\nimport hashlib\\nimport re\\n\\nTOKEN = re.compile(r\\"\\\\d+(?:\\\\.\\\\d+)?%?|[^\\\\W_]+(?:[\'’][^\\\\W_]+)?|[<>=+−-]\\", re.UNICODE)\\nGUARD = re.compile(r\\"\\\\b(if|unless|when|provided|assuming|because|therefore|thus|hence|not|no|without|may|might|could|would|should)\\\\b\\", re.I)\\nPATTERNS = {\\n    \\"quantities\\": r\\"\\\\b\\\\d+(?:[.,]\\\\d+)*(?:\\\\s*%)?\\",\\n    \\"negation\\": r\\"\\\\b(?:not|no|never|neither|nor|without)\\\\b\\",\\n    \\"modality\\": r\\"\\\\b(?:may|might|can|could|would|should|must|will|possible|possibly|expected|expect|expects)\\\\b\\",\\n    \\"scope\\": r\\"\\\\b(?:all|every|each|most|many|some|only|always|usually|generally|often|sometimes|exactly|at least|at most)\\\\b\\",\\n    \\"condition_cues\\": r\\"\\\\b(?:if|unless|when|provided that|assuming|as long as|conditional on)\\\\b\\",\\n    \\"causality\\": r\\"\\\\b(?:cause|causes|caused|lead to|leads to|because|therefore|thus|hence|implies|imply|increases|decreases|reduces|raises)\\\\b\\",\\n}\\n\\n\\ndef sha(text):\\n    return hashlib.sha256(text.encode()).hexdigest()\\n\\n\\ndef tokens(text):\\n    return TOKEN.findall(text.casefold())\\n\\n\\ndef sentences(text):\\n    return [part.strip() for part in re.split(r\\"(?<=[.!?])\\\\s+(?=[A-Z])\\", text.strip()) if part.strip()]\\n\\n\\ndef conditional(text):\\n    \\"\\"\\"Only recognize a single explicit if-condition with no nested guard.\\"\\"\\"\\n    body = text.rstrip(\\". \\")\\n    front = re.fullmatch(r\\"If ([^,;]+), ([^;]+)\\", body, re.I)\\n    back = re.fullmatch(r\\"([^,;]+) if ([^,;]+)\\", body, re.I)\\n    if front:\\n        condition, proposition = front.groups()\\n    elif back:\\n        proposition, condition = back.groups()\\n    else:\\n        return None\\n    if re.search(r\\"\\\\b(if|unless|only|because|then|otherwise)\\\\b\\", condition+\\" \\"+proposition, re.I):\\n        return None\\n    return {\\"condition\\": condition.strip(), \\"proposition\\": proposition.strip()}\\n\\n\\ndef ledger(text):\\n    claims = []\\n    for index, sentence in enumerate(sentences(text)):\\n        start = text.find(sentence, claims[-1][\\"evidence\\"][\\"end\\"] if claims else 0)\\n        protected = {key: [{\\"text\\": match.group(), \\"start\\": start+match.start(), \\"end\\": start+match.end()}\\n                           for match in re.finditer(pattern, sentence, re.I)] for key, pattern in PATTERNS.items()}\\n        condition = conditional(sentence)\\n        # Actor/predicate hypotheses retain exact evidence; unresolved syntax is explicit.\\n        actor = re.match(r\\"(.{1,100}?)\\\\s+(?:is|are|was|were|has|have|may|might|can|could|would|should|must|will|find|finds|use|uses|argue|argues|predict|predicts|reduce|reduces|increase|increases|cause|causes)\\\\b\\", sentence, re.I)\\n        actors = [actor.group(1)] if actor else []\\n        relations = []\\n        for cue in protected[\\"causality\\"]:\\n            offset = cue[\\"start\\"]-start\\n            left, right = sentence[:offset].strip(), sentence[offset+len(cue[\\"text\\"]):].strip()\\n            direction = \\"right_explains_left\\" if cue[\\"text\\"].lower() == \\"because\\" else \\"left_to_right_hypothesis\\"\\n            relations.append({\\"cue\\": cue[\\"text\\"], \\"left\\": left, \\"right\\": right,\\n                              \\"direction\\": direction, \\"verified_parse\\": False})\\n        claims.append({\\"claim_id\\": f\\"c{index+1}\\", \\"actors\\": actors,\\n                       \\"proposition\\": condition[\\"proposition\\"] if condition else sentence,\\n                       \\"conditions\\": ([{\\"text\\":condition[\\"condition\\"],\\"parse_status\\":\\"single_if_rule\\"}] if condition else\\n                                      [{\\"text\\":sentence,\\"cue\\":cue[\'text\'],\\"parse_status\\":\\"unresolved_scope\\"}\\n                                       for cue in protected[\'condition_cues\']]),\\n                       **protected, \\"causal_relations\\": relations,\\n                       \\"evidence\\": {\\"text\\": sentence, \\"start\\": start, \\"end\\": start+len(sentence)},\\n                       \\"actor_parse_status\\": \\"heuristic\\" if actor else \\"unresolved\\"})\\n    return {\\"source_sha256\\": sha(text), \\"claims\\": claims,\\n            \\"coverage\\": \\"All sentence spans retained; atomic claims, coreference and implicit relations are not fully parsed\\",\\n            \\"independently_semantically_verified\\": False}\\n\\n\\ndef invariants(text):\\n    return {key: collections.Counter(match.group().lower().replace(\\" \\", \\"\\")\\n                                     for match in re.finditer(pattern, text, re.I))\\n            for key, pattern in PATTERNS.items()}\\n\\n\\ndef risk_flags(source, target):\\n    a, b = invariants(source), invariants(target)\\n    return [key+\\"_surface_mismatch\\" for key in a if a[key] != b[key]]\\n\\n\\ndef variants(text):\\n    \\"\\"\\"Reproducible candidate grammar; no target/reference text is used.\\"\\"\\"\\n    result = [{\\"operation\\": \\"identity\\", \\"text\\": text}]\\n    parts = sentences(text)\\n    # Do not fuse conditional/negated/modal/causal clauses or quoted material.\\n    for i in range(len(parts)-1):\\n        left, right = parts[i:i+2]\\n        if GUARD.search(left+\\" \\"+right) or any(c in left+right for c in (\'\\"\', \'“\', \'”\')):\\n            continue\\n        if not left.endswith(\'.\') or re.search(r\\"(?:\\\\b[A-Z]|e\\\\.g|i\\\\.e|et al)\\\\.$\\", left):\\n            continue\\n        right=re.sub(r\\"^(The|This|These|Those|It|We|A|An)\\\\b\\",lambda match:match.group().lower(),right)\\n        joined = left[:-1]+\\"; \\"+right\\n        result.append({\\"operation\\": \\"join_independent_sentences\\", \\"text\\": \\" \\".join(parts[:i]+[joined]+parts[i+2:])})\\n        if len(result) >= 6:\\n            break\\n    for i, part in enumerate(parts):\\n        condition = conditional(part)\\n        if condition:\\n            c, p = condition[\\"condition\\"], condition[\\"proposition\\"]\\n            rewritten = p+\\" if \\"+c+\\".\\" if part.lower().startswith(\\"if \\") else \\"If \\"+c+\\", \\"+p+\\".\\"\\n            operation = \\"condition_to_end\\" if part.lower().startswith(\\"if \\") else \\"condition_to_front\\"\\n            result.append({\\"operation\\": operation, \\"text\\": \\" \\".join(parts[:i]+[rewritten]+parts[i+1:])})\\n        if \\";\\" in part and not GUARD.search(part) and not any(c in part for c in (\'\\"\',\'“\',\'”\')):\\n            pieces = part.split(\\";\\")\\n            if len(pieces) == 2 and all(piece.strip() for piece in pieces):\\n                result.append({\\"operation\\": \\"split_independent_clauses\\",\\n                               \\"text\\": \\" \\".join(parts[:i]+[pieces[0].strip()+\\". \\"+pieces[1].strip()[0].upper()+pieces[1].strip()[1:]]+parts[i+1:])})\\n    # Explicit author voice transformation is one-directional: do not guess that\\n    # an arbitrary \'the authors\' phrase refers to this paper rather than a citation.\\n    if re.search(r\\"\\\\bWe\\\\b\\", text) and not any(c in text for c in (\'\\"\',\'“\',\'”\')):\\n        result.append({\\"operation\\": \\"neutralize_author_voice\\", \\"text\\": re.sub(r\\"\\\\bWe\\\\b\\", \\"The authors\\", text)})\\n    unique = {}\\n    for candidate in result:\\n        unique.setdefault(candidate[\\"text\\"], candidate)\\n    return list(unique.values())\\n\\n\\ndef verify(source, candidate):\\n    \\"\\"\\"Pass only exact identity or a replayable conservative surface operation.\\"\\"\\"\\n    flags = risk_flags(source, candidate)\\n    if flags:\\n        return {\\"status\\": \\"rejected\\", \\"reasons\\": flags, \\"semantic_proof\\": False}\\n    replay = next((v for v in variants(source) if v[\\"text\\"] == candidate), None)\\n    if replay is not None:\\n        return {\\"status\\": \\"surface_preserved\\", \\"operation\\": replay[\\"operation\\"],\\n                \\"reasons\\": [], \\"semantic_proof\\": False,\\n                \\"scope\\": \\"Replayable bounded surface grammar; not full natural-language entailment\\"}\\n    return {\\"status\\": \\"unknown\\", \\"reasons\\": [\\"outside_verified_surface_grammar\\"], \\"semantic_proof\\": False}\\n", "src/fama_style/research/experiments.py": "\\"\\"\\"Prepare separate weakly supervised SFT/DPO research data; never trains.\\"\\"\\"\\nimport collections\\nimport re\\n\\nfrom .claims import sha, variants, verify, risk_flags\\nfrom .signal import overlap, features, distance\\n\\n\\ndef paraphrases(source, limit=4):\\n    \\"\\"\\"Two-hop bounded rewrites with replayable step-by-step provenance.\\"\\"\\"\\n    queue=[(source,[])];seen={source};outputs=[]\\n    for _ in range(2):\\n        next_queue=[]\\n        for text,path in queue:\\n            for candidate in variants(text):\\n                rewritten=candidate[\'text\']\\n                if rewritten in seen or verify(text,rewritten)[\'status\']!=\'surface_preserved\':\\n                    continue\\n                seen.add(rewritten)\\n                step={\\"operation\\":candidate[\'operation\'],\\"input_sha256\\":sha(text),\\"output_sha256\\":sha(rewritten),\\n                      \\"text\\":rewritten}\\n                chain=path+[step]\\n                outputs.append({\\"text\\":rewritten,\\"operations\\":chain,\\n                                \\"lexical_change\\":1-overlap(source,rewritten),\\"features\\":features(rewritten)})\\n                next_queue.append((rewritten,chain))\\n        queue=next_queue\\n    # Prefer a range of structural feature vectors; lexical-only micro-edits are reported.\\n    selected=[];feature_sets=set()\\n    for item in sorted(outputs,key=lambda item:-item[\'lexical_change\']):\\n        signature=tuple(round(v,3) for v in item[\'features\'].values())\\n        if signature not in feature_sets:\\n            selected.append(item);feature_sets.add(signature)\\n        if len(selected)>=limit:break\\n    return selected\\n\\n\\ndef corruption(source):\\n    replacements={\\"may\\":\\"will\\",\\"might\\":\\"will\\",\\"some\\":\\"all\\",\\"many\\":\\"all\\",\\"not\\":\\"\\"}\\n    for match in re.finditer(r\\"\\\\b(may|might|some|many|not)\\\\b\\",source,re.I):\\n        changed=source[:match.start()]+replacements[match.group().lower()]+source[match.end():]\\n        if risk_flags(source,changed):\\n            return {\\"text\\":changed,\\"operation\\":\\"deliberate_\\"+match.group().lower()+\\"_mutation\\",\\n                    \\"changed_span\\":[match.start(),match.end()]}\\n    return None\\n\\n\\ndef prepare(rows, profile):\\n    if any(row[\'split\']!=\'train\' for row in rows):raise ValueError(\'Training preparation requires training rows only\')\\n    sft,preferences,quarantine,coverage=[],[],[],[]\\n    for row in rows:\\n        source,target=row[\'plain_english\'],row[\'completion\']\\n        inherited=risk_flags(source,target)\\n        inputs=paraphrases(source)\\n        for variant in inputs:\\n            sft.append({\\"parent_id\\":row[\'id\'],\\"parent_paper\\":row[\'paper_key\'],\\"split\\":\\"train\\",\\n                        \\"input\\":variant[\'text\'],\\"target\\":target,\\"target_sha256\\":sha(target),\\n                        \\"original_input_sha256\\":sha(source),\\"provenance\\":variant,\\n                        \\"reference_alignment_flags\\":inherited,\\"semantic_status\\":\\"surface_equivalent_to_curated_input_only\\",\\n                        \\"training_ready\\":False})\\n        coverage.append({\\"id\\":row[\'id\'],\\"variants\\":len(inputs),\\n                         \\"lexically_diverse_variants\\":sum(item[\'lexical_change\']>=.1 for item in inputs),\\n                         \\"reference_alignment_flags\\":inherited})\\n        negatives=[]\\n        # Avoid automatically treating already-clear or low-contrast input as bad.\\n        contrast=distance(source,profile)-distance(target,profile)\\n        if overlap(source,target)<.8 and contrast>.1:\\n            negatives.append({\\"category\\":\\"copying\\",\\"text\\":source,\\"contrast_proxy\\":contrast})\\n        for candidate in variants(target):\\n            if candidate[\'operation\'] in (\'neutralize_author_voice\',\'split_independent_clauses\'):\\n                if verify(target,candidate[\'text\'])[\'status\']==\'surface_preserved\' and distance(candidate[\'text\'],profile)-distance(target,profile)>.05:\\n                    negatives.append({\\"category\\":\\"generic_surface_proxy\\",\\"text\\":candidate[\'text\'],\\n                                      \\"operation\\":candidate[\'operation\']})\\n        bad=corruption(source)\\n        if bad:negatives.append({\\"category\\":\\"semantic_error\\",\\"text\\":bad[\'text\'],\\"mutation\\":bad})\\n        for negative in negatives:\\n            item={\\"parent_id\\":row[\'id\'],\\"split\\":\\"train\\",\\"prompt\\":row.get(\'prompt\',source),\\n                  \\"chosen\\":target,\\"rejected\\":negative[\'text\'],\\"negative_category\\":negative[\'category\'],\\n                  \\"chosen_sha256\\":sha(target),\\"original_input_sha256\\":sha(source),\\"provenance\\":negative,\\n                  \\"reference_alignment_flags\\":inherited,\\"training_ready\\":False,\\n                  \\"label_status\\":\\"weak_preference_not_independent_semantic_verification\\"}\\n            if inherited:quarantine.append(item)\\n            else:preferences.append(item)\\n    summary={\\"training_parent_rows\\":len(rows),\\"sft_variants_staged\\":len(sft),\\n             \\"parents_with_two_or_more_variants\\":sum(item[\'variants\']>=2 for item in coverage),\\n             \\"parents_with_two_lexically_diverse_variants\\":sum(item[\'lexically_diverse_variants\']>=2 for item in coverage),\\n             \\"weak_preferences\\":len(preferences),\\"quarantined_preferences\\":len(quarantine),\\n             \\"weak_preference_categories\\":dict(collections.Counter(row[\'negative_category\'] for row in preferences)),\\n             \\"all_preference_categories\\":dict(collections.Counter(row[\'negative_category\'] for row in preferences+quarantine)),\\n             \\"training_ready\\":False,\\n             \\"reason\\":\\"Bounded surface paraphrases are not a rich semantic-invariance generator; reference alignment and style proxies need automated model-based calibration before training.\\",\\n             \\"generic_negative_caveat\\":\\"Third-person/clause-splitting controls proxy generic surface style, not the entire space of fluent generic prose.\\"}\\n    return {\\"sft\\":sft,\\"preferences\\":preferences,\\"quarantined_preferences\\":quarantine,\\"coverage\\":coverage,\\"summary\\":summary}\\n\\n\\ndef plans():\\n    common={\\"status\\":\\"prepared_not_executed\\",\\"gpu_submission_allowed\\":False,\\"model_id\\":\\"google/gemma-3-4b-it\\",\\n            \\"quantization\\":\\"NF4 FP32\\",\\"microbatch_size\\":1,\\"gradient_accumulation_steps\\":2,\\n            \\"sequence_limit\\":1024,\\"truncate\\":False,\\"lora_rank\\":8,\\"language_qv_only\\":True,\\n            \\"reserved_test_use\\":False,\\"human_review_required\\":False}\\n    return {\\"A_semantic_invariant_sft\\":{**common,\\"hypothesis\\":\\"Diverse content-equivalent inputs reduce the shortcut of copying one neutral phrasing.\\",\\n                \\"target\\":\\"unchanged original training reference\\",\\"maximum_paraphrases_per_parent\\":4,\\n                \\"sample_weight\\":\\"equal parent weight, not equal weight per generated row\\",\\n                \\"next_gate\\":\\"automatically verified diversity/coverage and no target leakage; three-example five-update memory probe only after later authorization\\"},\\n            \\"B_contrastive_dpo\\":{**common,\\"hypothesis\\":\\"Explicit faithful-style versus copy/generic/error contrasts change the generation preference, beyond target likelihood.\\",\\n                \\"reference_policy\\":\\"frozen pilot/SFT snapshot; never silently adapter-disabled raw base when SFT reference is intended\\",\\n                \\"inspected_trl_commit\\":\\"88365e546b5be84821f1a8bd9f167ef3230950e2\\",\\n                \\"peft_reference_plan\\":\\"Use the inspected implementation\'s frozen adapter named ref, or precomputed reference scores fingerprinted to the exact SFT snapshot; do not pass obsolete ref_adapter_name arguments\\",\\n                \\"unvalidated_compatibility_gates\\":[\\"Gemma text path accepts fused_lm_head=True and returns log_probs\\", \\"ref adapter frozen while policy adapter updates\\", \\"cached reference scores equal uncached scores\\", \\"chosen and rejected jointly fit one T4 without truncation\\"],\\n                \\"trl_proposed_options\\":{\\"loss_type\\":[\\"sigmoid\\"],\\"beta\\":.1,\\"learning_rate\\":5e-6,\\n                    \\"per_device_train_batch_size\\":1,\\"precompute_ref_log_probs\\":True,\\"precompute_ref_batch_size\\":1,\\n                    \\"max_length\\":None,\\"padding_free\\":False,\\"sync_ref_model\\":False},\\n                \\"compatibility_status\\":\\"current upstream API inspected; not installed or GPU validated\\",\\n                \\"next_gate\\":\\"cache reference log-probs against the exact frozen SFT adapter, explicit no-truncation audit; then separately authorized five-pair memory probe\\"}}\\n", "src/fama_style/research/prototype.py": "\\"\\"\\"Claim ledger → empirical plan → candidates → conservative automated checks.\\"\\"\\"\\nfrom .claims import ledger, variants, verify, sha\\nfrom .signal import distance, overlap\\n\\n\\ndef rewrite(text, profile):\\n    claim_ledger=ledger(text)\\n    plan={\\"fit_split\\":\\"train\\",\\"operations\\":profile[\'eligible_operations\'],\\n          \\"source_distance\\":distance(text,profile),\\"reference_features\\":profile[\'features\']}\\n    candidates=[]\\n    for candidate in variants(text):\\n        verification=verify(text,candidate[\'text\'])\\n        gain=plan[\'source_distance\']-distance(candidate[\'text\'],profile)\\n        similarity=overlap(text,candidate[\'text\'])\\n        admissible=(verification[\'status\']==\'surface_preserved\' and candidate[\'operation\'] in plan[\'operations\']\\n                    and gain>.05 and similarity<.9)\\n        candidates.append({**candidate,\\"verification\\":verification,\\"style_gain_proxy\\":gain,\\n                           \\"copy_similarity\\":similarity,\\"eligible_for_selection\\":admissible})\\n    good=[candidate for candidate in candidates if candidate[\'eligible_for_selection\']]\\n    selected=max(good,key=lambda item:item[\'style_gain_proxy\']) if good else candidates[0]\\n    return {\\"source_sha256\\":sha(text),\\"claim_ledger\\":claim_ledger,\\"style_plan\\":plan,\\"candidates\\":candidates,\\n            \\"selected_text\\":selected[\'text\'],\\"status\\":\\"selected_bounded_rewrite\\" if good else \\"abstained_unchanged\\",\\n            \\"semantic_guarantee\\":False,\\"scope\\":\\"CPU rule prototype; no LLM, no trained model, no independent entailment validation\\"}\\n", "src/fama_style/research/relation_guards.py": "\\"\\"\\"Finite numeric-endpoint and association/causality guards with scoped abstention.\\n\\nThese guards supplement a fallible verifier; they never certify equivalence.\\nOnly complete supported clauses and literal variable/entity matches are compared.\\n\\"\\"\\"\\nfrom decimal import Decimal\\nimport re\\n\\nfrom .claims import sentences\\nfrom .claim_audit import NUMBER_WORDS\\n\\nNUMBER=r\'-?(?:\\\\d+(?:\\\\.\\\\d+)?|\'+ \'|\'.join(NUMBER_WORDS)+\')\'\\nENTITY=r\'[A-Za-z][A-Za-z0-9 -]{0,100}\'\\nUNIT=r\'(?:percent|%|units?|points?|degrees?)\'\\nOPS={\\n    \'greater than or equal to\':\'>=\',\'higher than or equal to\':\'>=\',\'at least\':\'>=\',\'no less than\':\'>=\',\'not less than\':\'>=\',\\n    \'less than or equal to\':\'<=\',\'lower than or equal to\':\'<=\',\'at most\':\'<=\',\'no more than\':\'<=\',\'not more than\':\'<=\',\'no greater than\':\'<=\',\\n    \'greater than\':\'>\',\'higher than\':\'>\',\'more than\':\'>\',\'above\':\'>\',\\n    \'less than\':\'<\',\'lower than\':\'<\',\'below\':\'<\',\'>=\':\'>=\',\'<=\':\'<=\',\'≥\':\'>=\',\'≤\':\'<=\',\'>\':\'>\',\'<\':\'<\'}\\nOP_PATTERN=\'|\'.join(re.escape(x) for x in sorted(OPS,key=len,reverse=True))\\nBOUND=re.compile(r\'(?P<subject>\'+ENTITY+r\')\\\\s+(?:is|are|must be)\\\\s+(?P<operator>\'+OP_PATTERN+r\')\\\\s*(?P<value>\'+NUMBER+r\')\\\\s*(?P<unit>\'+UNIT+r\')?\',re.I)\\nBOUND_POST=re.compile(r\'(?P<subject>\'+ENTITY+r\')\\\\s+(?:is|are|must be)\\\\s+(?P<value>\'+NUMBER+r\')\\\\s*(?P<unit>\'+UNIT+r\')?\\\\s+or\\\\s+(?P<direction>higher|lower|more|less)\',re.I)\\nBOUND_NEG=re.compile(r\'(?P<subject>\'+ENTITY+r\')\\\\s+(?P<operator>does not exceed|do not exceed|must not exceed|does not fall below|do not fall below)\\\\s+(?P<value>\'+NUMBER+r\')\\\\s*(?P<unit>\'+UNIT+r\')?\',re.I)\\nBOUND_RELEVANT=re.compile(r\'(?:(?:\'+OP_PATTERN+r\'|does not exceed|do not exceed|must not exceed|does not fall below|do not fall below)\\\\s*\'+NUMBER+r\'\\\\b|\\\\b\'+NUMBER+r\'\\\\s*(?:\'+UNIT+r\')?\\\\s+or\\\\s+(?:higher|lower|more|less)\\\\b)\',re.I)\\nBLOCKED=set(\'not no never may might could can should would will all every some most many if unless when because that which only during after before under in according report reports reported claim claims hypothesize hypothesizes establishes establish\'.split())\\nASSOC=re.compile(r\'(?P<a>\'+ENTITY+r\')\\\\s+(?:(?:is|are)\\\\s+(?:associated|correlated)\\\\s+with|correlates?\\\\s+with)\\\\s+(?P<b>\'+ENTITY+r\')\',re.I)\\nASSOC_SYM=re.compile(r\'(?P<a>\'+ENTITY+r\')\\\\s+and\\\\s+(?P<b>\'+ENTITY+r\')\\\\s+(?:are|remain)\\\\s+(?:associated|correlated)\',re.I)\\nCAUSE=re.compile(r\'(?P<a>\'+ENTITY+r\')\\\\s+(?:causes?|leads? to|results? in)\\\\s+(?P<b>\'+ENTITY+r\')\',re.I)\\nCAUSE_PASSIVE=re.compile(r\'(?P<b>\'+ENTITY+r\')\\\\s+(?:(?:is|are)\\\\s+caused by|results? from)\\\\s+(?P<a>\'+ENTITY+r\')\',re.I)\\nREL_RELEVANT=re.compile(r\'\\\\b(?:associated|correlated|correlates?|causes?|caused|leads? to|results? (?:in|from))\\\\b\',re.I)\\n\\n\\ndef entity(text):\\n    text=re.sub(r\'^the\\\\s+\',\'\',text.strip().lower());words=text.split()\\n    if not words or set(words)&BLOCKED or words.count(\'and\'):return None\\n    return \' \'.join(words)\\n\\n\\ndef value(text):\\n    text=text.lower();sign=-1 if text.startswith(\'-\') else 1;body=text.lstrip(\'-\')\\n    return format((Decimal(str(NUMBER_WORDS.get(body,body)))*sign).normalize(),\'f\')\\n\\n\\ndef clauses(text):\\n    result=[]\\n    for i,sentence in enumerate(sentences(text),1):\\n        # Preserve sentence evidence even when a supported numeric condition is\\n        # isolated. Commas/qualifiers not recognized by the grammar cause abstention.\\n        for part in re.split(r\';|\\\\b(?:when|if|provided that)\\\\b\',sentence,flags=re.I):\\n            body=part.strip().rstrip(\'.!?\').strip()\\n            if body:result.append({\'sentence_id\':i,\'text\':body,\'sentence_evidence\':sentence})\\n    return result\\n\\n\\ndef observations(text):\\n    bounds=[];relations=[];unresolved=[]\\n    for c in clauses(text):\\n        body=c[\'text\'];parsed_bound=False;parsed_relation=False\\n        for pattern in (BOUND,BOUND_POST,BOUND_NEG):\\n            m=pattern.fullmatch(body)\\n            if not m:continue\\n            subject=entity(m[\'subject\'])\\n            if subject is None:continue\\n            if pattern is BOUND_POST:operator=\'>=\' if m[\'direction\'].lower() in (\'higher\',\'more\') else \'<=\'\\n            elif pattern is BOUND_NEG:operator=\'>=\' if \'fall below\' in m[\'operator\'].lower() else \'<=\'\\n            else:operator=OPS[m[\'operator\'].lower()]\\n            unit=(m[\'unit\'] or \'\').lower();unit=\'percent\' if unit==\'%\' else unit.rstrip(\'s\')\\n            bounds.append({**c,\'subject\':subject,\'operator\':operator,\'value\':value(m[\'value\']),\'unit\':unit});parsed_bound=True;break\\n        conditional_relation=bool(re.search(r\'\\\\b(?:if|when|provided that|unless)\\\\b\',c[\'sentence_evidence\'],re.I))\\n        for kind,pattern in (() if conditional_relation else ((\'association\',ASSOC),(\'association\',ASSOC_SYM),(\'cause\',CAUSE_PASSIVE),(\'cause\',CAUSE))):\\n            m=pattern.fullmatch(body)\\n            if not m:continue\\n            a,b=entity(m[\'a\']),entity(m[\'b\'])\\n            if a is None or b is None:continue\\n            relations.append({**c,\'kind\':kind,\'a\':a,\'b\':b});parsed_relation=True;break\\n        if BOUND_RELEVANT.search(body) and not parsed_bound:unresolved.append({**c,\'kind\':\'unsupported_numeric_bound\'})\\n        if REL_RELEVANT.search(body) and not parsed_relation:unresolved.append({**c,\'kind\':\'unsupported_relation_scope\'})\\n    return {\'bounds\':bounds,\'relations\':relations,\'unresolved\':unresolved}\\n\\n\\ndef check(source,output):\\n    a,b=observations(source),observations(output);conflicts=[]\\n    if \' \'.join(source.split())==\' \'.join(output.split()):\\n        return {\'conflicts\':[],\'unresolved\':[],\'source_observations\':a,\'output_observations\':b,\\n            \'identity_bypass\':True,\'semantic_proof\':False,\'limitations\':\'Identical text has no transformation; this is not general semantic verification.\'}\\n    unresolved=[{\'side\':side,**u} for side,obs in ((\'source\',a),(\'output\',b)) for u in obs[\'unresolved\']]\\n    key=lambda x:(x[\'subject\'],x[\'value\'],x[\'unit\'])\\n    for left in a[\'bounds\']:\\n        right=[r for r in b[\'bounds\'] if key(r)==key(left)]\\n        siblings=[r for r in a[\'bounds\'] if key(r)==key(left)]\\n        if len(right)!=1 or len(siblings)!=1:\\n            unresolved.append({\'kind\':\'ambiguous_or_unmatched_bound\',\'source_evidence\':left});continue\\n        if left[\'operator\']!=right[0][\'operator\']:\\n            conflicts.append({\'kind\':\'numeric_bound_relation_changed\',\'source_evidence\':left,\'output_evidence\':right[0],\\n                \'scope\':\'Literal variable, value and unit match; operator changed. No integer-domain or cross-condition inference.\'})\\n    for right in b[\'bounds\']:\\n        if not any(key(left)==key(right) for left in a[\'bounds\']):unresolved.append({\'kind\':\'unmatched_output_bound\',\'output_evidence\':right})\\n    pair=lambda r:frozenset((r[\'a\'],r[\'b\']))\\n    source_causes={(r[\'a\'],r[\'b\']) for r in a[\'relations\'] if r[\'kind\']==\'cause\'}\\n    for left in a[\'relations\']:\\n        aligned=[r for r in b[\'relations\'] if pair(r)==pair(left)]\\n        if not aligned:unresolved.append({\'kind\':\'unmatched_source_relation\',\'source_evidence\':left});continue\\n        if left[\'kind\']==\'association\':\\n            for right in aligned:\\n                if right[\'kind\']==\'cause\' and (right[\'a\'],right[\'b\']) not in source_causes:\\n                    conflicts.append({\'kind\':\'association_strengthened_to_causation\',\'source_evidence\':left,\'output_evidence\':right,\\n                        \'scope\':\'Literal entity pair matches. A causal claim elsewhere counts only for this directed pair.\'})\\n    for right in b[\'relations\']:\\n        if not any(pair(left)==pair(right) for left in a[\'relations\']):unresolved.append({\'kind\':\'unmatched_output_relation\',\'output_evidence\':right})\\n    return {\'conflicts\':conflicts,\'unresolved\':unresolved,\'source_observations\':a,\'output_observations\':b,\\n        \'semantic_proof\':False,\'limitations\':\'Finite full-clause grammar and literal entity/unit/value alignment. Unsupported wording, negation, attribution, modalities and ambiguous scopes abstain; no general arithmetic or causal proof.\'}\\n", "src/fama_style/research/run_extension.py": "\\"\\"\\"CPU-only signal diagnosis, constrained inference and training-data preparation.\\"\\"\\"\\nimport argparse\\nimport collections\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nfrom .claims import sha, risk_flags\\nfrom .signal import fit_profile, diagnose, overlap, distance\\nfrom .prototype import rewrite\\nfrom .experiments import prepare, plans\\n\\nROOT=Path(__file__).resolve().parents[3]\\nEXPECTED={\\"train\\":\\"a108416c1935615a7e01bb4d3fa86e0818b961bfb16f15d3883e6905654ee248\\",\\n          \\"validation\\":\\"f1c75c29fa93f8458e7a3b2f8996bbd13ff023f92e4a74f1d086d22af5e546b8\\"}\\n\\n\\ndef read_split(path, split):\\n    if split not in EXPECTED or Path(path).name==\'test.jsonl\':\\n        raise ValueError(\'Reserved test input is forbidden\')\\n    raw=Path(path).read_bytes()\\n    if hashlib.sha256(raw).hexdigest()!=EXPECTED[split]:raise ValueError(\'Frozen \'+split+\' hash mismatch\')\\n    rows=[json.loads(line) for line in raw.decode().splitlines() if line.strip()]\\n    if any(row[\'split\']!=split for row in rows):raise ValueError(\'Mixed split input\')\\n    return rows\\n\\n\\ndef write_json(path,value):\\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\\\n\',encoding=\'utf-8\')\\n\\n\\ndef write_lines(path,rows):\\n    path.write_text(\'\'.join(json.dumps(row,ensure_ascii=False)+\'\\\\n\' for row in rows),encoding=\'utf-8\')\\n\\n\\ndef synthetic():\\n    # Original synthetic examples; no paper text or model outputs.\\n    inputs=[(\'We use data to measure output. Firms report revenue.\',\'We use data to measure output; Firms report revenue.\'),\\n            (\'If demand falls, firms reduce output.\',\'Firms reduce output if demand falls.\'),\\n            (\'Some firms may not increase investment. We find no effect.\',\'Some firms may not increase investment. We find no effect.\')]\\n    train=[{\'id\':f\'synthetic-train-{i}\',\'paper_key\':\'SYN\',\'split\':\'train\',\'plain_english\':a,\'completion\':b,\'prompt\':a}\\n           for i,(a,b) in enumerate(inputs)]\\n    validation=[{\'id\':\'synthetic-validation\',\'paper_key\':\'SYN-HELD\',\'split\':\'validation\',\\n                 \'plain_english\':\'Firms report sales. Investors use the reports.\',\\n                 \'completion\':\'Firms report sales; Investors use the reports.\'}]\\n    return train,validation\\n\\n\\ndef execute(train, validation, directory, synthetic_run=False):\\n    if any(row[\'split\']!=\'train\' for row in train) or any(row[\'split\']!=\'validation\' for row in validation):\\n        raise ValueError(\'Split leakage detected\')\\n    if set(row[\'id\'] for row in train)&set(row[\'id\'] for row in validation):raise ValueError(\'Overlapping IDs\')\\n    if directory.exists() and any(directory.iterdir()):raise ValueError(\'Refusing to overwrite research outputs\')\\n    directory.mkdir(parents=True,exist_ok=True)\\n    profile=fit_profile(train)\\n    pair_diagnostics,phase_a=diagnose(train,profile)\\n    write_json(directory/\'A_train_signal_summary.json\',phase_a)\\n    write_json(directory/\'A_style_profile.json\',profile)\\n    write_lines(directory/\'A_pair_diagnostics.private.jsonl\',pair_diagnostics)\\n    outputs=[]\\n    for row in validation:\\n        result=rewrite(row[\'plain_english\'],profile)\\n        outputs.append({\'id\':row[\'id\'],**result,\\n                        \'reference_risk_screen\':risk_flags(row[\'plain_english\'],row[\'completion\'])})\\n    write_lines(directory/\'B_validation_prototype.private.jsonl\',outputs)\\n    phase_b={\'validation_rows\':len(validation),\'generated_candidates\':sum(len(r[\'candidates\']) for r in outputs),\\n             \'surface_accepted_candidates\':sum(c[\'verification\'][\'status\']==\'surface_preserved\' for r in outputs for c in r[\'candidates\']),\\n             \'selected_rewrites\':sum(r[\'status\']==\'selected_bounded_rewrite\' for r in outputs),\\n             \'abstained_unchanged\':sum(r[\'status\']==\'abstained_unchanged\' for r in outputs),\\n             \'mean_selected_source_overlap\':sum(overlap(row[\'plain_english\'],r[\'selected_text\']) for row,r in zip(validation,outputs))/len(outputs),\\n             \'mean_selected_style_gain_proxy\':sum(distance(row[\'plain_english\'],profile)-distance(r[\'selected_text\'],profile)\\n                                                   for row,r in zip(validation,outputs))/len(outputs),\\n             \'model_generation_executed\':False,\'human_judgment_used\':False,\\n             \'semantic_success_rate\':None,\'semantic_success_reason\':\'No calibrated semantic parser/NLI; bounded surface checks do not establish general entailment\',\\n             \'test_rows_used\':0}\\n    write_json(directory/\'B_prototype_summary.json\',phase_b)\\n    training=prepare(train,profile)\\n    for name,key in ((\'C_A_sft_candidates.private.jsonl\',\'sft\'),(\'C_B_weak_preferences.private.jsonl\',\'preferences\'),\\n                     (\'C_B_quarantined_preferences.private.jsonl\',\'quarantined_preferences\'),(\'C_parent_coverage.private.jsonl\',\'coverage\')):\\n        write_lines(directory/name,training[key])\\n    write_json(directory/\'C_preparation_summary.json\',training[\'summary\'])\\n    write_json(directory/\'C_experiment_plans.json\',plans())\\n    summary={\'run_kind\':\'synthetic_dry_run\' if synthetic_run else \'executed_cpu_research\',\\n             \'phase_a\':phase_a,\'phase_b\':phase_b,\'phase_c\':training[\'summary\'],\\n             \'gpu_jobs_submitted\':0,\'gpu_training_executed\':False,\'test_rows_used\':0,\\n             \'existing_experiments_modified\':False}\\n    write_json(directory/\'summary.json\',summary)\\n    files={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in directory.iterdir() if p.is_file()}\\n    write_json(directory/\'manifest.json\',{\'implementation\':\'claim_style_extension_v1\',\\n        \'inputs\':{\'training_rows\':len(train),\'validation_rows\':len(validation),\'training_split_sha256\':None if synthetic_run else EXPECTED[\'train\'],\\n                  \'validation_split_sha256\':None if synthetic_run else EXPECTED[\'validation\']},\\n        \'code_hashes\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob(\'*.py\')},\\n        \'files\':files,\'no_model_or_api_calls\':True,\'test_rows_used\':0})\\n    print(json.dumps({\'run_kind\':summary[\'run_kind\'],\'train_rows\':len(train),\'validation_rows\':len(validation),\\n                      \'selected_prototype_rewrites\':phase_b[\'selected_rewrites\'],\\n                      \'sft_candidates\':training[\'summary\'][\'sft_variants_staged\'],\\n                      \'training_ready\':False,\'gpu_jobs_submitted\':0,\'output_dir\':str(directory)}))\\n    return summary\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--dry-run\',action=\'store_true\',help=\'Use only synthetic examples; CPU, no model/API\')\\n    parser.add_argument(\'--output-dir\',type=Path,required=True)\\n    args=parser.parse_args(argv)\\n    train,validation=synthetic() if args.dry_run else (read_split(ROOT/\'data/fama_corpus_v1/train.jsonl\',\'train\'),\\n                                                   read_split(ROOT/\'data/fama_corpus_v1/validation.jsonl\',\'validation\'))\\n    execute(train,validation,args.output_dir,args.dry_run)\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/research/signal.py": "\\"\\"\\"Training-only feature profiles and distributions, without ML dependencies.\\"\\"\\"\\nimport collections\\nimport difflib\\nimport math\\nimport re\\nimport statistics\\n\\nfrom .claims import sentences, tokens, risk_flags\\n\\nFEATURES = (\\"words_per_sentence\\", \\"semicolon_rate\\", \\"author_voice_rate\\", \\"initial_if_rate\\", \\"parenthesis_rate\\")\\n\\n\\ndef features(text):\\n    words = tokens(text)\\n    parts = sentences(text)\\n    n = max(1, len(words))\\n    return {\\"words_per_sentence\\": len(words)/max(1, len(parts)),\\n            \\"semicolon_rate\\": 100*text.count(\';\')/n,\\n            \\"author_voice_rate\\": 100*len(re.findall(r\\"\\\\b(?:we|our)\\\\b\\", text, re.I))/n,\\n            \\"initial_if_rate\\": sum(p.lower().startswith(\'if \') for p in parts)/max(1,len(parts)),\\n            \\"parenthesis_rate\\": 100*text.count(\'(\')/n}\\n\\n\\ndef overlap(a, b):\\n    return difflib.SequenceMatcher(None, tokens(a), tokens(b), autojunk=False).ratio()\\n\\n\\ndef distribution(values, bins=None):\\n    ordered = sorted(values)\\n    def quantile(q):\\n        position = (len(ordered)-1)*q\\n        lo, hi = math.floor(position), math.ceil(position)\\n        return ordered[lo]+(ordered[hi]-ordered[lo])*(position-lo)\\n    report = {\\"n\\": len(values), \\"min\\": ordered[0], \\"p10\\": quantile(.1), \\"p25\\": quantile(.25),\\n              \\"median\\": quantile(.5), \\"p75\\": quantile(.75), \\"p90\\": quantile(.9), \\"max\\": ordered[-1],\\n              \\"mean\\": statistics.mean(values)}\\n    if bins:\\n        report[\\"histogram\\"] = [{\\"lower\\": a, \\"upper\\": b, \\"count\\": sum(a<=v<b or (i==len(bins)-2 and v==b) for v in values)}\\n                               for i,(a,b) in enumerate(zip(bins,bins[1:]))]\\n    return report\\n\\n\\ndef fit_profile(rows):\\n    if not rows or any(row[\'split\']!=\'train\' for row in rows):\\n        raise ValueError(\'Style features can only be fitted on training references\')\\n    source = [features(row[\'plain_english\']) for row in rows]\\n    target = [features(row[\'completion\']) for row in rows]\\n    profile = {\\"fit_split\\": \\"train\\", \\"training_rows\\": len(rows), \\"features\\": {},\\n               \\"limitations\\": \\"Surface style proxies, not an author classifier; topic/length confounds remain\\"}\\n    for key in FEATURES:\\n        a,b=[r[key] for r in source],[r[key] for r in target]\\n        profile[\'features\'][key] = {\\"source_mean\\": statistics.mean(a), \\"reference_mean\\": statistics.mean(b),\\n                                    \\"reference_std\\": max(statistics.pstdev(b), .1),\\n                                    \\"mean_paired_delta\\": statistics.mean(y-x for x,y in zip(a,b)),\\n                                    \\"positive_pair_fraction\\": sum(y>x for x,y in zip(a,b))/len(a),\\n                                    \\"negative_pair_fraction\\": sum(y<x for x,y in zip(a,b))/len(a)}\\n    # Rules are eligible only when their direction follows the observed training delta.\\n    f=profile[\'features\'];ops=[]\\n    if f[\'words_per_sentence\'][\'mean_paired_delta\']>.5:ops.append(\'join_independent_sentences\')\\n    if f[\'words_per_sentence\'][\'mean_paired_delta\']<-.5:ops.append(\'split_independent_clauses\')\\n    if f[\'initial_if_rate\'][\'mean_paired_delta\']>.01:ops.append(\'condition_to_front\')\\n    if f[\'initial_if_rate\'][\'mean_paired_delta\']<-.01:ops.append(\'condition_to_end\')\\n    if f[\'author_voice_rate\'][\'mean_paired_delta\']<-.05:ops.append(\'neutralize_author_voice\')\\n    profile[\'eligible_operations\']=ops\\n    return profile\\n\\n\\ndef distance(text, profile):\\n    observed=features(text)\\n    return statistics.mean(abs(observed[key]-value[\'reference_mean\'])/value[\'reference_std\']\\n                           for key,value in profile[\'features\'].items())\\n\\n\\ndef diagnose(rows, profile):\\n    result=[]\\n    for row in rows:\\n        a,b=row[\'plain_english\'],row[\'completion\'];fa,fb=features(a),features(b)\\n        result.append({\\"id\\":row[\'id\'],\\"paper_key\\":row[\'paper_key\'],\\"lexical_overlap\\":overlap(a,b),\\n                       \\"word_length_ratio\\":len(tokens(b))/max(1,len(tokens(a))),\\n                       \\"feature_deltas\\":{key:fb[key]-fa[key] for key in FEATURES},\\n                       \\"style_vector_magnitude\\":statistics.mean(abs(fb[key]-fa[key])/profile[\'features\'][key][\'reference_std\'] for key in FEATURES),\\n                       \\"alignment_risk_flags\\":risk_flags(a,b),\\n                       \\"alignment_status\\":\\"risk_screen_only_not_entailment\\"})\\n    summary={\\"rows\\":len(rows),\\"lexical_overlap\\":distribution([r[\'lexical_overlap\'] for r in result],[0,.2,.4,.6,.8,.85,1]),\\n             \\"word_length_ratio\\":distribution([r[\'word_length_ratio\'] for r in result]),\\n             \\"style_vector_magnitude\\":distribution([r[\'style_vector_magnitude\'] for r in result]),\\n             \\"feature_delta_distributions\\":{key:distribution([r[\'feature_deltas\'][key] for r in result]) for key in FEATURES},\\n             \\"alignment_risk_counts\\":dict(collections.Counter(flag for r in result for flag in r[\'alignment_risk_flags\'])),\\n             \\"pairs_with_surface_alignment_flags\\":sum(bool(r[\'alignment_risk_flags\']) for r in result),\\n             \\"exact_normalized_copies\\":sum(tokens(row[\'plain_english\'])==tokens(row[\'completion\']) for row in rows),\\n             \\"overlap_at_least_0_85\\":sum(r[\'lexical_overlap\']>=.85 for r in result),\\n             \\"per_paper\\":{key:{\\"rows\\":len(group),\\"median_lexical_overlap\\":statistics.median(r[\'lexical_overlap\'] for r in group),\\n                               \\"median_style_magnitude\\":statistics.median(r[\'style_vector_magnitude\'] for r in group),\\n                               \\"mean_feature_deltas\\":{feature:statistics.mean(r[\'feature_deltas\'][feature] for r in group) for feature in FEATURES}}\\n                          for key in sorted({r[\'paper_key\'] for r in result})\\n                          for group in [[r for r in result if r[\'paper_key\']==key]]},\\n             \\"semantic_alignment_caveat\\":\\"Counts/lexical features flag risk; differing surface markers can be equivalent and matching markers can hide drift.\\"}\\n    return result, summary\\n", "src/fama_style/rhetoric/__init__.py": "\\"\\"\\"Separate, CPU-first rhetorical structure and oracle inference experiment.\\"\\"\\"\\n", "src/fama_style/rhetoric/claim_constrained.py": "\\"\\"\\"Matched 2x2 claim-constraint and output-format inference diagnostic.\\"\\"\\"\\nimport argparse\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport statistics\\nimport time\\n\\nfrom . import concise_plan as base\\nfrom . import sentence_slots as slot_base\\nfrom .concise_plan import (MODEL_ID,MODEL_REVISION,NLI_MODEL,NLI_REVISION,CONTEXT_LIMIT,\\n    file_sha,read_lines,load_pack,synthetic_pack)\\nfrom .experiment import secret_token,with_heartbeat,write_json,write_lines,fresh\\nfrom .semantics import assess,calibration_controls as original_controls\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research.claims import sentences,sha,risk_flags,ledger\\nfrom fama_style.research.signal import distance,overlap\\n\\nPROTOCOL=\'claim_constrained_realization_v1\'\\nCONDITIONS=(\'A_count_paragraph\',\'B_claims_paragraph\',\'C_count_slots\',\'D_claims_slots\')\\nMAX_NEW_TOKENS=768\\n\\n\\ndef constraint_ledger(source):\\n    \\"\\"\\"Source-only evidence, not an independent semantic parse or repaired input.\\"\\"\\"\\n    extracted=ledger(source);claims=[]\\n    bounds=((\'upper_bound\',r\\"\\\\b(?:does not exceed|do not exceed|never more than|at most|no more than)\\\\b\\"),\\n            (\'lower_bound\',r\\"\\\\b(?:does not fall below|do not fall below|at least|no less than)\\\\b\\"))\\n    for claim in extracted[\'claims\']:\\n        text=claim[\'evidence\'][\'text\'];protected=[]\\n        for kind in (\'quantities\',\'negation\',\'modality\',\'scope\',\'condition_cues\',\'causality\'):\\n            protected.extend({\'kind\':kind,\'evidence\':x[\'text\']} for x in claim[kind])\\n        protected.extend({\'kind\':label,\'evidence\':m.group()} for label,pattern in bounds for m in re.finditer(pattern,text,re.I))\\n        claims.append({\'claim_id\':claim[\'claim_id\'],\'source_evidence\':text,\'protected_evidence\':protected})\\n    return {\'claims\':claims,\'status\':\'sentence_spans_and_surface_cues_not_atomic_or_independently_verified\'}\\n\\n\\ndef jobs_for(pack):\\n    jobs=[]\\n    for index,row in enumerate(pack[\'rows\']):\\n        order=CONDITIONS[index%4:]+CONDITIONS[:index%4]\\n        count=row[\'plan\'][\'sentence_count\'];common=make_prompt(row[\'source\'])\\n        common+=f\'\\\\n\\\\nPresentation control: write exactly {count} complete sentences. Preserve source claim order. Faithfulness takes priority over sentence count.\'\\n        for condition in order:\\n            prompt=common\\n            if condition in (\'B_claims_paragraph\',\'D_claims_slots\'):\\n                prompt+=\'\\\\n\\\\nSource-only preservation ledger (evidence, not additional facts):\\\\n\'+json.dumps(constraint_ledger(row[\'source\']),ensure_ascii=False,separators=(\',\',\':\'))\\n                prompt+=\'\\\\nPreserve every ledger entry, including conditions, attribution, uncertainty and supporting details. Keep upper bounds as upper bounds and lower bounds as lower bounds; do not invert inequality direction or its negation. Preserve ranges, denominators, quantities and qualifiers together. Possibility is not certainty, association is not causation, and an observed frequency is not a universal claim. Do not invent explanations, mechanisms, biases, recommendations or conclusions. The ledger is a fallible surface aid; the complete source remains authoritative. Reorganize wording only; do not add sentences with unsupported claims to reach the count. Return no ledger or analysis.\'\\n            if condition.endswith(\'_slots\'):\\n                prompt+=\'\\\\nOutput-format override: return only a JSON object with the single key \\"sentences\\", whose value is an array of exactly \'+str(count)+\' strings. Each string must contain exactly one complete sentence ending in a period, question mark or exclamation mark. No Markdown fences, commentary or other keys. The strings in order will be joined into one paragraph.\'\\n            jobs.append({\'id\':row[\'id\'],\'condition\':condition,\'prompt\':prompt,\'prompt_sha256\':sha(prompt),\'source_sha256\':sha(row[\'source\'])})\\n    return jobs\\n\\n\\ndef realize(raw,condition,count):\\n    mapped=\'C_sentence_slots\' if condition.endswith(\'_slots\') else \'A_ordinary\'\\n    strict=slot_base.realize(raw,mapped,count);normalized=raw;wrapper_removed=False\\n    # Predeclared normalization: strip exactly one complete outer JSON fence only.\\n    if mapped==\'C_sentence_slots\' and raw.startswith(\'```json\\\\n\') and raw.endswith(\'\\\\n```\'):\\n        normalized=raw[len(\'```json\\\\n\'):-len(\'\\\\n```\')];wrapper_removed=True\\n    result=slot_base.realize(normalized,mapped,count)\\n    result.update(raw_format_valid=strict[\'format_valid\'],raw_format_errors=strict[\'format_errors\'],outer_fence_removed=wrapper_removed)\\n    if result[\'slot_count\'] is not None:result[\'semantic_input\']=\'joined_slots\' if result[\'format_valid\'] else \'joined_slots_format_failed\'\\n    return result\\n\\n\\ndef calibration_controls(predict):\\n    controls=original_controls(predict)\\n    pairs=[(\'upper_bound_reversal\',\'Independence is useful only when dependence does not exceed the threshold.\',\\n            \'Independence is useful only when dependence does not fall below the threshold.\'),\\n           (\'quantity_cap\',\'The maximum portfolio share is 4 percent.\',\'The maximum portfolio share is 8 percent.\'),\\n           (\'uncertainty\',\'The pattern may occur in some years.\',\'The pattern occurs in all years.\'),\\n           (\'causal_direction\',\'Higher costs cause lower trading volume.\',\'Lower trading volume causes higher costs.\')]\\n    results={name:assess(a,b,predict) for name,a,b in pairs}\\n    controls[\'original_controls_passed\']=controls[\'passed\']\\n    controls[\'targeted_mutation_checks\']=results\\n    controls[\'targeted_mutations_rejected\']=sum(r[\'status\']==\'proxy_flagged\' for r in results.values())\\n    controls[\'passed\']=controls[\'passed\'] and controls[\'targeted_mutations_rejected\']==len(pairs)\\n    controls[\'meaning\']=\'Synthetic sanity checks including targeted semantic mutations; passing does not verify economic claims or checker accuracy.\'\\n    return controls\\n\\n\\ndef audit_tokens(job,ids):\\n    if sha(job[\'prompt\'])!=job[\'prompt_sha256\']:raise ValueError(\'Changed prompt\')\\n    if not ids or len(ids)+MAX_NEW_TOKENS>CONTEXT_LIMIT:raise ValueError(\'Context overflow; no truncation allowed\')\\n    return {\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'input_tokens\':len(ids),\'prompt_sha256\':job[\'prompt_sha256\'],\'input_ids_sha256\':sha(json.dumps(ids))}\\n\\n\\ndef evaluate(pack,jobs,outputs,semantic=None,pair_checks=None,calibration=None,synthetic=False):\\n    expected={(j[\'id\'],j[\'condition\']):j for j in jobs};seen=set();records=[]\\n    if jobs!=jobs_for(pack) or len(outputs)!=len(expected):raise ValueError(\'Changed jobs or incomplete outputs\')\\n    rows={r[\'id\']:r for r in pack[\'rows\']}\\n    for out in outputs:\\n        key=(out[\'id\'],out[\'condition\']);job=expected.get(key)\\n        if key in seen or job is None or out[\'prompt_sha256\']!=job[\'prompt_sha256\']:raise ValueError(\'Duplicate/swapped output\')\\n        if not out.get(\'output\',\'\').strip() or out.get(\'hit_output_limit\') is not False or out.get(\'synthetic\') is not synthetic:\\n            raise ValueError(\'Empty, cutoff or mislabeled output\')\\n        seen.add(key);row=rows[out[\'id\']];n=row[\'plan\'][\'sentence_count\'];r=realize(out[\'output\'],out[\'condition\'],n);text=r[\'text\']\\n        nli=(semantic or {}).get(key);valid=r[\'format_valid\'];passed=bool(nli and nli[\'status\']==\'proxy_consistent\' and calibration and calibration[\'passed\'])\\n        records.append({\'id\':out[\'id\'],\'condition\':out[\'condition\'],**r,\'desired_sentence_count\':n,\\n            \'sentence_count_absolute_error\':abs(r[\'sentence_count\']-n),\\n            \'nli\':nli,\'nli_status\':nli[\'status\'] if nli else \'not_run\',\\n            \'format_count_semantic_proxy_pass\':valid and r[\'count_match\'] and passed if calibration and calibration[\'passed\'] else None,\\n            \'near_copy\':overlap(row[\'source\'],text)>=.9 if valid else None,\\n            \'style_gain_proxy\':distance(row[\'source\'],pack[\'style_profile\'])-distance(text,pack[\'style_profile\']) if valid else None,\\n            \'semantic_surface_flags\':risk_flags(row[\'source\'],text),\\n            \'generation_seconds\':out.get(\'generation_seconds\',0),\'input_tokens\':out.get(\'input_tokens\',0),\'output_tokens\':out.get(\'output_tokens\',0)})\\n    aggregate={}\\n    for c in CONDITIONS:\\n        group=[r for r in records if r[\'condition\']==c];valid=[r for r in group if r[\'format_valid\']]\\n        aggregate[c]={\'rows\':len(group),\'format_valid\':len(valid),\'count_matches\':sum(r[\'format_valid\'] and r[\'count_match\'] for r in group),\\n            \'format_invalid\':len(group)-len(valid),\\n            \'raw_format_valid\':sum(r[\'raw_format_valid\'] for r in group),\\n            \'outer_fences_removed\':sum(r[\'outer_fence_removed\'] for r in group),\\n            \'nli_unknown_valid_only\':sum(r[\'nli_status\']==\'unknown_over_context\' for r in valid),\\n            \'mean_count_error_valid_only\':statistics.mean(r[\'sentence_count_absolute_error\'] for r in valid) if valid else None,\\n            \'nli_proxy_passes_valid_only\':sum(r[\'nli_status\']==\'proxy_consistent\' for r in valid) if calibration and calibration[\'passed\'] else None,\\n            \'format_count_semantic_proxy_passes\':sum(r[\'format_count_semantic_proxy_pass\'] for r in group) if calibration and calibration[\'passed\'] else None,\\n            \'uncalibrated_nli_consistent_valid_only\':sum(r[\'nli_status\']==\'proxy_consistent\' for r in valid),\\n            \'near_copies_valid_only\':sum(r[\'near_copy\'] for r in valid),\\n            \'mean_style_gain_valid_only\':statistics.mean(r[\'style_gain_proxy\'] for r in valid) if valid else None,\\n            **{k:sum(r[k] or 0 for r in group) for k in (\'generation_seconds\',\'input_tokens\',\'output_tokens\')}}\\n    return records,{\'protocol\':PROTOCOL,\'execution_kind\':\'synthetic_wiring_check\' if synthetic else \'model_inference\',\'conditions\':aggregate,\\n        \'pair_source_reference_nli\':pair_checks,\'nli_controls\':calibration,\'training_called\':False,\'held_out_rows_used\':0,\\n        \'oracle_is_deployable\':False,\'semantic_accuracy\':None,\'full_mapping_training_ready\':False,\'repair_attempts\':0,\'normalization\':\'Exactly one full outer json code fence; report raw and normalized compliance separately\',\\n        \'limitations\':\'Four previously inspected training pairs; matched constraints x format; source-only heuristic ledgers; no generalization claim. Sentence boundaries and style are surface proxies. NLI can miss drift. Invalid formats remain failures in all-row denominators; valid-only metrics must report their denominators.\'}\\n\\n\\ndef run_training(pack,output,uploaded_pack_sha):\\n    \\"\\"\\"Inference only; lazy ML imports follow the repository entry-point convention.\\"\\"\\"\\n    fresh(output);jobs=jobs_for(pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'uploaded_pack_identity.json\',{\'sha256\':uploaded_pack_sha,\'format\':\'uploaded file bytes\'})\\n    status={\'complete\':False,\'protocol\':PROTOCOL,\'training_called\':False,\'held_out_rows_used\':0,\'completed_jobs\':0,\'total_jobs\':len(jobs)}\\n    started=time.monotonic();outputs=[]\\n    def phase(name,**extra):\\n        status.update(phase=name,**extra);write_json(output/\'run_status.json\',status);print(\'Phase: \'+name,flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        import os\\n        import gc\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\';os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor,AutoTokenizer,AutoModelForSequenceClassification,\\n            Gemma3ForConditionalGeneration,BitsAndBytesConfig,LogitsProcessor,LogitsProcessorList)\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Expose one Kaggle GPU before importing torch\')\\n        torch.cuda.init();torch.cuda.set_device(0);torch.manual_seed(42);torch.cuda.reset_peak_memory_stats(0)\\n        if \'T4\' not in torch.cuda.get_device_name(0):raise RuntimeError(\'Select Kaggle T4 x2 for the pinned diagnostic\')\\n        phase(\'checking_secret\');token=secret_token()\\n        phase(\'loading_pinned_processor\')\\n        processor=with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token),\'Processor load\')\\n        def tokenize(job):\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[]\\n        for job in jobs:audits.append(audit_tokens(job,tokenize(job)[\'input_ids\'][0].tolist()))\\n        write_json(output/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)),\'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):\\n            raise RuntimeError(\'Unexpected model placement\')\\n        model.eval();model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):\\n                    raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        by_key={(r[\'id\'],r[\'condition\']):r for r in audits}\\n        for job in jobs:\\n            phase(\'generation\',completed_jobs=len(outputs))\\n            batch=tokenize(job);ids=batch[\'input_ids\'][0].tolist();audit=audit_tokens(job,ids)\\n            if audit!=by_key[(job[\'id\'],job[\'condition\'])]:raise ValueError(\'Tokenized prompt changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()};before=time.monotonic()\\n            with torch.inference_mode():\\n                generated=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,\\n                    use_cache=True,pad_token_id=processor.tokenizer.pad_token_id,remove_invalid_values=False,\\n                    logits_processor=LogitsProcessorList([FiniteScores()])),\'Generation\')\\n            new=generated[0,len(ids):];eos=model.generation_config.eos_token_id;eos=eos if isinstance(eos,list) else [eos]\\n            text=processor.tokenizer.decode(new,skip_special_tokens=True).strip()\\n            result={\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\'output\':text,\\n                \'input_tokens\':len(ids),\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            outputs.append(result);write_lines(output/\'generations.private.jsonl\',outputs)\\n            if not text or result[\'hit_output_limit\']:raise RuntimeError(\'Empty/cutoff output; partial results remain incomplete\')\\n            phase(\'generation\',completed_jobs=len(outputs))\\n        gemma_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del model,processor,batch,generated,new,tokenize\\n        gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'loading_independent_nli_on_gpu\');nli_started=time.monotonic()\\n        tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n        nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,\\n            token=False).to(\'cuda:0\'),\'NLI model load\')\\n        nli.eval();nli.requires_grad_(False)\\n        labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n        if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n        limit=min(512,int(getattr(nli.config,\'max_position_embeddings\',512)),int(tokenizer.model_max_length))\\n        cache={};cache_hits=0\\n        def predict(premise,hypothesis):\\n            nonlocal cache_hits\\n            key=(sha(premise),sha(hypothesis))\\n            if key in cache:cache_hits+=1;return cache[key]\\n            batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n            if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n            with torch.inference_mode():\\n                probs=torch.softmax(nli(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n            cache[key]={labels[i]:float(p) for i,p in enumerate(probs)}\\n            return cache[key]\\n        calibration=with_heartbeat(lambda:calibration_controls(predict),\'NLI controls\');write_json(output/\'nli_controls.json\',calibration)\\n        semantic={};pair_checks={}\\n        for index,row in enumerate(pack[\'rows\'],1):\\n            phase(\'pair_fidelity_assessment\',completed_pairs=index-1)\\n            pair_checks[row[\'id\']]=assess(row[\'source\'],row[\'reference\'],predict)\\n        source_by_id={row[\'id\']:row[\'source\'] for row in pack[\'rows\']}\\n        for index,result in enumerate(outputs,1):\\n            phase(\'output_semantic_assessment\',completed_nli_outputs=index-1)\\n            semantic[(result[\'id\'],result[\'condition\'])]=with_heartbeat(lambda:assess(source_by_id[result[\'id\']],realize(result[\'output\'],result[\'condition\'],next(r[\'plan\'][\'sentence_count\'] for r in pack[\'rows\'] if r[\'id\']==result[\'id\']))[\'text\'],predict),\'NLI assessment\')\\n        records,summary=evaluate(pack,jobs,outputs,semantic,pair_checks,calibration)\\n        write_lines(output/\'metrics.private.jsonl\',records)\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=NLI_REVISION,\\n            nli_device=\'cuda:0 after Gemma unload\',nli_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,\\n            nli_cache_entries=len(cache),nli_cache_hits=cache_hits,gemma_memory=gemma_memory,\\n            nli_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)},\\n            generation={\'do_sample\':False,\'seed\':42,\'max_new_tokens\':MAX_NEW_TOKENS,\'context_limit\':CONTEXT_LIMIT,\'truncate\':False},\\n            versions={name:importlib.metadata.version(name) for name in (\'torch\',\'transformers\',\'accelerate\',\'bitsandbytes\',\'huggingface-hub\')})\\n        write_json(output/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True,completed_jobs=len(outputs),completed_nli_outputs=len(outputs),completed_pairs=len(pack[\'rows\']))\\n        write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'training_called\':False,\'input_pack_sha256\':uploaded_pack_sha,\\n            \'files\':{p.name:file_sha(p) for p in output.iterdir() if p.is_file()}})\\n    except Exception as error:\\n        failed_at=status.get(\'phase\');locations=[];tb=error.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:locations.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,failed_at=failed_at,error_code=\'concise_plan_inference_failed\',local_line_numbers=locations)\\n        raise RuntimeError(\'Concise-plan inference failed; inspect run_status.json and saved partial files. No training was called.\') from None\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    actions=parser.add_mutually_exclusive_group()\\n    actions.add_argument(\'--dry-run\',action=\'store_true\');actions.add_argument(\'--execute-inference\',action=\'store_true\')\\n    parser.add_argument(\'--pack\',type=Path);parser.add_argument(\'--expected-pack-sha\');parser.add_argument(\'--output-dir\',type=Path,required=True)\\n    args=parser.parse_args(argv)\\n    if args.dry_run:\\n        fresh(args.output_dir);pack=synthetic_pack();jobs=jobs_for(pack);rows={r[\'id\']:r for r in pack[\'rows\']}\\n        outputs=[]\\n        for j in jobs:\\n            ref=rows[j[\'id\']][\'reference\'];text=json.dumps({\'sentences\':sentences(ref)}) if j[\'condition\'].endswith(\'_slots\') else ref\\n            outputs.append({\'id\':j[\'id\'],\'condition\':j[\'condition\'],\'prompt_sha256\':j[\'prompt_sha256\'],\'output\':text,\'hit_output_limit\':False,\'synthetic\':True})\\n        records,summary=evaluate(pack,jobs,outputs,synthetic=True)\\n        write_json(args.output_dir/\'synthetic_evaluation.json\',summary);write_lines(args.output_dir/\'synthetic_jobs.jsonl\',jobs)\\n        print(\'Synthetic wiring check complete: 16 jobs; no model calls.\');return\\n    if args.pack is None or args.expected_pack_sha is None:parser.error(\'--pack and --expected-pack-sha required\')\\n    pack=load_pack(args.pack,args.expected_pack_sha)\\n    if args.execute_inference:run_training(pack,args.output_dir,args.expected_pack_sha)\\n    else:\\n        fresh(args.output_dir);write_json(args.output_dir/\'preflight.json\',{\'protocol\':PROTOCOL,\'pack_sha256\':args.expected_pack_sha,\'pairs\':4,\'jobs\':16,\'training_called\':False})\\n        print(\'Preflight passed; no Secret access or model loading.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/rhetoric/concise_plan.py": "\\"\\"\\"Eight-generation, training-only oracle-plan realization diagnostic. No trainer.\\"\\"\\"\\nimport argparse\\nimport collections\\nimport hashlib\\nimport importlib.metadata\\nimport itertools\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport statistics\\nimport time\\n\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research.claims import sentences,sha,risk_flags,tokens\\nfrom fama_style.research.signal import fit_profile,distance,overlap\\nfrom .extract import extract\\nfrom .experiment import secret_token,with_heartbeat,write_json,write_lines,fresh\\nfrom .semantics import NLI_MODEL,assess,calibration_controls\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'concise_plan_realization_v1\'\\nTRAIN_SHA=\'a108416c1935615a7e01bb4d3fa86e0818b961bfb16f15d3883e6905654ee248\'\\nMODEL_ID=\'google/gemma-3-4b-it\'\\nMODEL_REVISION=\'093f9f388b31de276ce2de164bdc2081324b9767\'\\nNLI_REVISION=\'6f5cf0a2b59cabb106aca4c287eed12e357e90eb\'\\nCONDITIONS=(\'A_ordinary\',\'B_concise_oracle_plan\')\\nWORD_BANDS={\'short\':(1,16),\'medium\':(17,32),\'long\':(33,48),\'extended\':(49,72)}\\nSHAPES=(\'unconstrained\',\'dependent_clause\',\'coordinated_segments\')\\nMAX_NEW_TOKENS=512\\nCONTEXT_LIMIT=4096\\n\\n\\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\\ndef read_lines(path):return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]\\ndef word_count(text):return len(re.findall(r\\"\\\\b[\\\\w’\'-]+\\\\b\\",text))\\n\\n\\ndef word_band(text):\\n    n=word_count(text)\\n    for band,(low,high) in WORD_BANDS.items():\\n        if low<=n<=high:return band\\n    raise ValueError(\'Reference sentence outside supported length bands\')\\n\\n\\ndef plan_from_annotation(annotation,reference):\\n    units=annotation[\'rhetorical_taxonomy\'][\'sentence_structure\']\\n    if \' \'.join(\' \'.join(s[\'sentence_text\'].split()) for s in units)!=\' \'.join(reference.split()):\\n        raise ValueError(\'Reference annotation coverage mismatch\')\\n    if len(sentences(reference))!=len(units):raise ValueError(\'Sentence counter and annotated reference disagree\')\\n    layouts=[]\\n    for unit in units:\\n        clauses=unit[\'clauses\']\\n        # Positive annotations only: a single spanning clause does not imply simplicity.\\n        shape=\'dependent_clause\' if any(c[\'depends_on_clause_ids\'] for c in clauses) else (\\n            \'coordinated_segments\' if any(\'coordinat\' in c[\'role\'][\'name\'].lower() for c in clauses) else \'unconstrained\')\\n        layouts.append({\'word_band\':word_band(unit[\'sentence_text\']),\'shape_hint\':shape})\\n    plan={\'sentence_count\':len(units),\'sentences\':layouts,\'max_semicolons\':reference.count(\';\')}\\n    validate_plan(plan)\\n    return plan\\n\\n\\ndef validate_plan(plan):\\n    if set(plan)!={\'sentence_count\',\'sentences\',\'max_semicolons\'}:raise ValueError(\'Unexpected free-form plan fields\')\\n    if type(plan[\'sentence_count\']) is not int or not 2<=plan[\'sentence_count\']<=6:raise ValueError(\'Unsupported sentence count\')\\n    if type(plan[\'max_semicolons\']) is not int or not 0<=plan[\'max_semicolons\']<=2:raise ValueError(\'Unsupported semicolon budget\')\\n    if len(plan[\'sentences\'])!=plan[\'sentence_count\']:raise ValueError(\'Plan slot count mismatch\')\\n    for slot in plan[\'sentences\']:\\n        if set(slot)!={\'word_band\',\'shape_hint\'} or slot[\'word_band\'] not in WORD_BANDS or slot[\'shape_hint\'] not in SHAPES:\\n            raise ValueError(\'Unknown or lexicalized plan slot\')\\n\\n\\ndef render_plan(plan):\\n    validate_plan(plan)\\n    lines=[f\'Write exactly {plan[\\"sentence_count\\"]} complete sentences in one paragraph.\',\\n           f\'Use at most {plan[\\"max_semicolons\\"]} semicolons.\']\\n    for index,slot in enumerate(plan[\'sentences\'],1):\\n        low,high=WORD_BANDS[slot[\'word_band\']];hint=\'\'\\n        if slot[\'shape_hint\']==\'dependent_clause\':hint=\' Prefer a dependent construction only where the source already supports that relationship.\'\\n        elif slot[\'shape_hint\']==\'coordinated_segments\':hint=\' Prefer linked grammatical segments only where the source already supplies their relationship.\'\\n        lines.append(f\'Sentence {index}: aim for {low}–{high} words.\'+hint)\\n    lines.append(\'These are presentation controls. Preserve the source claim order, all claims and qualifications. \'\\n                 \'Do not add a causal connection, condition, comparison, example or citation to satisfy a pattern. \'\\n                 \'Faithfulness takes priority over any length or syntax hint; do not pad the text. Return only the paragraph.\')\\n    return \'\\\\n\'.join(lines)\\n\\n\\ndef select_four(candidates):\\n    \\"\\"\\"Frozen low-cost contrast selection; no model outputs or held-out examples.\\"\\"\\"\\n    if any(row[\'split\']!=\'train\' for row in candidates):raise ValueError(\'Only training candidates allowed\')\\n    choices=[]\\n    for group in itertools.combinations(candidates,4):\\n        if len({r[\'paper_key\'] for r in group})!=4:continue\\n        deltas=[r[\'plan\'][\'sentence_count\']-r[\'source_sentence_count\'] for r in group]\\n        if sum(d>0 for d in deltas)!=2 or sum(d<0 for d in deltas)!=2:continue\\n        score=(sum(word_count(r[\'source\'])+word_count(r[\'reference\']) for r in group),\\n               sum(abs(d) for d in deltas),tuple(sorted(r[\'id\'] for r in group)))\\n        choices.append((score,group))\\n    if not choices:raise ValueError(\'Cannot select four distinct-paper pairs with two count increases and two decreases\')\\n    return sorted(min(choices,key=lambda value:value[0])[1],key=lambda row:row[\'id\'])\\n\\n\\ndef prepare_pack(output):\\n    fresh(output)\\n    training=ROOT/\'data/fama_corpus_v1/train.jsonl\'\\n    if file_sha(training)!=TRAIN_SHA:raise ValueError(\'Frozen training file changed\')\\n    train=read_lines(training)\\n    if len(train)!=247 or any(r[\'split\']!=\'train\' for r in train):raise ValueError(\'Expected frozen 247 training rows\')\\n    base=ROOT/\'research_runs/taxonomy_discovery_v1\';package=base/\'package_v1\'\\n    originals=base/\'response_audit_v1/original_annotations.checked.private.jsonl\'\\n    neutral=base/\'response_audit_v1/neutral_proposals.checked.private.jsonl\'\\n    references=base/\'response_audit_v2/annotations_B.normalized.private.jsonl\'\\n    keypath=package/\'PRIVATE_MAPPING_DO_NOT_UPLOAD.json\'\\n    sources=[training,originals,neutral,references,keypath]\\n    before={str(path.relative_to(ROOT)):file_sha(path) for path in sources}\\n    for folder,names in ((package,[keypath.name]),(originals.parent,[originals.name,neutral.name]),(references.parent,[references.name])):\\n        manifest=json.loads((folder/\'manifest.json\').read_text())\\n        for name in names:\\n            if file_sha(folder/name)!=manifest[\'files\'][name]:raise ValueError(\'Previously audited annotation changed\')\\n    by_id={r[\'id\']:r for r in train};a={r[\'paragraph_id\']:r for r in read_lines(originals)}\\n    b={r[\'paragraph_id\']:r for r in read_lines(references)};proposals={r[\'pair_id\']:r for r in read_lines(neutral)}\\n    candidates=[];excluded=collections.Counter()\\n    for key in json.loads(keypath.read_text()):\\n        proposal=proposals[key[\'pair_id\']];row=by_id[key[\'training_id\']]\\n        if proposal[\'decision\']!=\'retain\':excluded[\'unaccepted_rewrite_proposal\']+=1;continue\\n        if proposal[\'proposed_neutral_text\']!=row[\'plain_english\']:raise ValueError(\'Retained proposal differs from frozen source\')\\n        if word_count(row[\'plain_english\'])>140 or word_count(row[\'completion\'])>160:excluded[\'length_cap\']+=1;continue\\n        source_units=b[key[\'reference_id\']][\'rhetorical_taxonomy\'][\'sentence_structure\']\\n        if len(source_units)!=len(sentences(row[\'plain_english\'])):excluded[\'source_sentence_counter_disagreement\']+=1;continue\\n        try:plan=plan_from_annotation(a[key[\'original_item_id\']],row[\'completion\'])\\n        except ValueError:excluded[\'unsupported_reference_plan\']+=1;continue\\n        if plan[\'sentence_count\']==len(source_units):excluded[\'no_sentence_count_contrast\']+=1;continue\\n        candidates.append({\'id\':row[\'id\'],\'paper_key\':row[\'paper_key\'],\'split\':\'train\',\'source\':row[\'plain_english\'],\\n            \'reference\':row[\'completion\'],\'source_sentence_count\':len(source_units),\'plan\':plan,\\n            \'reference_annotation_sha256\':sha(json.dumps(a[key[\'original_item_id\']],sort_keys=True,ensure_ascii=False)),\\n            \'annotation_uncertainty\':a[key[\'original_item_id\']][\'review_notes\'],\\n            \'neutral_reference_review\':\'retained_by_ChatGPT_not_independently_verified\'})\\n    selected=select_four(candidates)\\n    pack={\'protocol\':PROTOCOL,\'parent_training_sha256\':TRAIN_SHA,\'rows\':selected,\'style_profile\':fit_profile(train),\\n        \'selection\':{\'eligible_candidates\':len(candidates),\'exclusion_counts\':dict(excluded),\\n            \'rule\':\'Four distinct training papers; two sentence-count increases, two decreases; minimize source+reference word budget; deterministic ties\',\\n            \'selection_used_model_outputs\':False,\'held_out_rows_read\':0},\\n        \'annotation_input_hashes\':before,\'scope\':\'Small training-only oracle diagnostic; no deployment or generalization result; not a test of the entire 132-category taxonomy\'}\\n    path=output/\'concise_plan_pack.json\';write_json(path,pack)\\n    for name,h in before.items():\\n        if file_sha(ROOT/name)!=h:raise ValueError(\'Source changed during preparation\')\\n    write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'pack_sha256\':file_sha(path),\'inputs_unchanged\':True,\\n        \'input_hashes\':before,\'code_sha256\':file_sha(Path(__file__).resolve()),\'model_calls\':0,\'gpu_training_called\':False})\\n    print(json.dumps({\'status\':\'pack_prepared_not_executed\',\'pairs\':4,\'jobs\':8,\'pack_sha256\':file_sha(path),\'output_dir\':str(output)}))\\n    return pack\\n\\n\\ndef load_pack(path,expected_sha):\\n    if not re.fullmatch(\'[0-9a-f]{64}\',expected_sha) or file_sha(path)!=expected_sha:raise ValueError(\'Input pack hash mismatch\')\\n    pack=json.loads(path.read_text())\\n    if pack[\'protocol\']!=PROTOCOL or pack[\'parent_training_sha256\']!=TRAIN_SHA:raise ValueError(\'Wrong pack protocol or parent corpus\')\\n    rows=pack[\'rows\']\\n    if len(rows)!=4 or len({r[\'id\'] for r in rows})!=4 or len({r[\'paper_key\'] for r in rows})!=4:raise ValueError(\'Wrong pair/paper counts\')\\n    for row in rows:\\n        if row[\'split\']!=\'train\' or not row[\'source\'].strip() or not row[\'reference\'].strip():raise ValueError(\'Invalid training row\')\\n        validate_plan(row[\'plan\'])\\n        if row[\'source_sentence_count\']!=len(sentences(row[\'source\'])):raise ValueError(\'Source sentence count changed\')\\n        if len(sentences(row[\'reference\']))!=row[\'plan\'][\'sentence_count\']:raise ValueError(\'Reference/plan count mismatch\')\\n        if [word_band(part) for part in sentences(row[\'reference\'])]!=[slot[\'word_band\'] for slot in row[\'plan\'][\'sentences\']]:\\n            raise ValueError(\'Reference/plan length bands changed\')\\n    deltas=[r[\'plan\'][\'sentence_count\']-r[\'source_sentence_count\'] for r in rows]\\n    if sum(d>0 for d in deltas)!=2 or sum(d<0 for d in deltas)!=2:raise ValueError(\'Count-contrast balance changed\')\\n    return pack\\n\\n\\ndef jobs_for(pack):\\n    result=[]\\n    for index,row in enumerate(pack[\'rows\']):\\n        order=CONDITIONS if index%2==0 else CONDITIONS[::-1]\\n        for condition in order:\\n            prompt=make_prompt(row[\'source\'])\\n            if condition==CONDITIONS[1]:prompt+=\'\\\\n\\\\nPresentation plan:\\\\n\'+render_plan(row[\'plan\'])\\n            result.append({\'id\':row[\'id\'],\'condition\':condition,\'prompt\':prompt,\'prompt_sha256\':sha(prompt),\'source_sha256\':sha(row[\'source\'])})\\n    return result\\n\\n\\ndef plan_metrics(text,plan):\\n    parts=sentences(text);count_match=len(parts)==plan[\'sentence_count\'];lengths=[];shapes=[]\\n    for part,slot in zip(parts,plan[\'sentences\']):\\n        low,high=WORD_BANDS[slot[\'word_band\']];lengths.append(low<=word_count(part)<=high)\\n        if slot[\'shape_hint\']!=\'unconstrained\':\\n            syntax=extract(part)[\'structure\'][\'sentences\'][0][\'syntax\']\\n            shapes.append(syntax in ((\'subordinated\',\'mixed\') if slot[\'shape_hint\']==\'dependent_clause\' else (\'coordinated\',\'mixed\')))\\n    # Missing/extra sentences never receive a perfect length score from a partial zip.\\n    return {\'sentence_count\':len(parts),\'desired_sentence_count\':plan[\'sentence_count\'],\'sentence_count_match\':count_match,\\n        \'sentence_count_absolute_error\':abs(len(parts)-plan[\'sentence_count\']),\\n        \'word_band_match\':sum(lengths)/max(len(parts),plan[\'sentence_count\']),\\n        \'semicolon_budget_match\':text.count(\';\')<=plan[\'max_semicolons\'],\\n        \'shape_hint_surface_match\':statistics.mean(shapes) if count_match and shapes else None,\\n        \'shape_measurement\':\'Limited surface syntax proxy, not verified parse or preserved semantic relation\'}\\n\\n\\ndef evaluate(pack,jobs,outputs,semantic=None,pair_checks=None,calibration=None,synthetic=False):\\n    expected={(job[\'id\'],job[\'condition\']):job for job in jobs};seen=set();records=[]\\n    if len(expected)!=len(jobs) or len(outputs)!=len(expected):raise ValueError(\'Duplicate jobs or incomplete generation set\')\\n    by_id={r[\'id\']:r for r in pack[\'rows\']}\\n    for output in outputs:\\n        key=(output[\'id\'],output[\'condition\']);job=expected.get(key)\\n        if key in seen or job is None or output[\'prompt_sha256\']!=job[\'prompt_sha256\']:raise ValueError(\'Duplicate/swapped generation\')\\n        if not output.get(\'output\',\'\').strip() or output.get(\'hit_output_limit\') is not False or output.get(\'synthetic\') is not synthetic:\\n            raise ValueError(\'Empty, truncated or misreported generation\')\\n        seen.add(key);row=by_id[output[\'id\']];text=output[\'output\'];nli=(semantic or {}).get(key)\\n        records.append({\'id\':row[\'id\'],\'condition\':output[\'condition\'],**plan_metrics(text,row[\'plan\']),\\n            \'source_similarity\':overlap(row[\'source\'],text),\'near_copy\':overlap(row[\'source\'],text)>=.90,\\n            \'exact_copy\':\' \'.join(row[\'source\'].split())==\' \'.join(text.split()),\\n            \'reference_similarity_diagnostic_only\':overlap(row[\'reference\'],text),\\n            \'semantic_surface_flags\':risk_flags(row[\'source\'],text),\'nli\':nli,\\n            \'nli_status\':nli[\'status\'] if nli else \'not_run\',\\n            \'style_gain_proxy\':distance(row[\'source\'],pack[\'style_profile\'])-distance(text,pack[\'style_profile\']),\\n            \'words_per_sentence\':word_count(text)/max(1,len(sentences(text))),\\n            \'input_tokens\':output.get(\'input_tokens\'),\'output_tokens\':output.get(\'output_tokens\'),\\n            \'generation_seconds\':output.get(\'generation_seconds\')})\\n    aggregate={}\\n    for condition in CONDITIONS:\\n        group=[r for r in records if r[\'condition\']==condition]\\n        aggregate[condition]={\'rows\':len(group),**{key:statistics.mean(r[key] for r in group) for key in\\n            (\'sentence_count_match\',\'sentence_count_absolute_error\',\'word_band_match\',\'semicolon_budget_match\',\'source_similarity\',\'near_copy\',\'exact_copy\',\'style_gain_proxy\',\'words_per_sentence\')},\\n            \'nli_proxy_passes\':sum(r[\'nli_status\']==\'proxy_consistent\' for r in group) if calibration and calibration[\'passed\'] else None,\\n            \'nli_unknown\':sum(r[\'nli_status\']==\'unknown_over_context\' for r in group),\\n            \'generation_seconds\':sum(r[\'generation_seconds\'] or 0 for r in group),\\n            \'input_tokens\':sum(r[\'input_tokens\'] or 0 for r in group),\'output_tokens\':sum(r[\'output_tokens\'] or 0 for r in group)}\\n    lookup={(r[\'id\'],r[\'condition\']):r for r in records};paired=[]\\n    for row in pack[\'rows\']:\\n        a,b=(lookup[(row[\'id\'],condition)] for condition in CONDITIONS)\\n        paired.append({\'id\':row[\'id\'],\'count_error_reduction\':a[\'sentence_count_absolute_error\']-b[\'sentence_count_absolute_error\'],\\n            \'style_gain_difference\':b[\'style_gain_proxy\']-a[\'style_gain_proxy\'],\\n            \'ordinary_nli\':a[\'nli_status\'],\'planned_nli\':b[\'nli_status\']})\\n    return records,{\'protocol\':PROTOCOL,\'execution_kind\':\'synthetic_wiring_check\' if synthetic else \'model_inference\',\\n        \'conditions\':aggregate,\'paired_differences\':paired,\'pair_source_reference_nli\':pair_checks,\\n        \'nli_controls\':calibration,\'training_called\':False,\'held_out_rows_used\':0,\'oracle_is_deployable\':False,\\n        \'semantic_accuracy\':None,\'full_mapping_training_ready\':False,\\n        \'limitations\':\'Four training pairs; joint count/length/clause/semicolon controls; heuristic adherence/style proxies; NLI is not proof. Pair-fidelity flags are reported without post-hoc sample removal.\'}\\n\\n\\ndef audit_tokens(job,ids):\\n    if sha(job[\'prompt\'])!=job[\'prompt_sha256\']:raise ValueError(\'Changed prompt\')\\n    if not ids or len(ids)+MAX_NEW_TOKENS>CONTEXT_LIMIT:raise ValueError(\'Context overflow; no truncation allowed\')\\n    return {\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'input_tokens\':len(ids),\\n        \'prompt_sha256\':job[\'prompt_sha256\'],\'input_ids_sha256\':sha(json.dumps(ids))}\\n\\n\\ndef run_training(pack,output,uploaded_pack_sha):\\n    \\"\\"\\"Inference only; lazy ML imports follow the repository entry-point convention.\\"\\"\\"\\n    fresh(output);jobs=jobs_for(pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'uploaded_pack_identity.json\',{\'sha256\':uploaded_pack_sha,\'format\':\'uploaded file bytes\'})\\n    status={\'complete\':False,\'protocol\':PROTOCOL,\'training_called\':False,\'held_out_rows_used\':0,\'completed_jobs\':0,\'total_jobs\':len(jobs)}\\n    started=time.monotonic();outputs=[]\\n    def phase(name,**extra):\\n        status.update(phase=name,**extra);write_json(output/\'run_status.json\',status);print(\'Phase: \'+name,flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        import os\\n        import gc\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\';os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor,AutoTokenizer,AutoModelForSequenceClassification,\\n            Gemma3ForConditionalGeneration,BitsAndBytesConfig,LogitsProcessor,LogitsProcessorList)\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Expose one Kaggle GPU before importing torch\')\\n        torch.cuda.init();torch.cuda.set_device(0);torch.manual_seed(42);torch.cuda.reset_peak_memory_stats(0)\\n        if \'T4\' not in torch.cuda.get_device_name(0):raise RuntimeError(\'Select Kaggle T4 x2 for the pinned diagnostic\')\\n        phase(\'checking_secret\');token=secret_token()\\n        phase(\'loading_pinned_processor\')\\n        processor=with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token),\'Processor load\')\\n        def tokenize(job):\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[]\\n        for job in jobs:audits.append(audit_tokens(job,tokenize(job)[\'input_ids\'][0].tolist()))\\n        write_json(output/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)),\'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):\\n            raise RuntimeError(\'Unexpected model placement\')\\n        model.eval();model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):\\n                    raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        by_key={(r[\'id\'],r[\'condition\']):r for r in audits}\\n        for job in jobs:\\n            phase(\'generation\',completed_jobs=len(outputs))\\n            batch=tokenize(job);ids=batch[\'input_ids\'][0].tolist();audit=audit_tokens(job,ids)\\n            if audit!=by_key[(job[\'id\'],job[\'condition\'])]:raise ValueError(\'Tokenized prompt changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()};before=time.monotonic()\\n            with torch.inference_mode():\\n                generated=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,\\n                    use_cache=True,pad_token_id=processor.tokenizer.pad_token_id,remove_invalid_values=False,\\n                    logits_processor=LogitsProcessorList([FiniteScores()])),\'Generation\')\\n            new=generated[0,len(ids):];eos=model.generation_config.eos_token_id;eos=eos if isinstance(eos,list) else [eos]\\n            text=processor.tokenizer.decode(new,skip_special_tokens=True).strip()\\n            result={\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\'output\':text,\\n                \'input_tokens\':len(ids),\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            outputs.append(result);write_lines(output/\'generations.private.jsonl\',outputs)\\n            if not text or result[\'hit_output_limit\']:raise RuntimeError(\'Empty/cutoff output; partial results remain incomplete\')\\n            phase(\'generation\',completed_jobs=len(outputs))\\n        gemma_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del model,processor,batch,generated,new,tokenize\\n        gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'loading_independent_nli_on_gpu\');nli_started=time.monotonic()\\n        tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n        nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,\\n            token=False).to(\'cuda:0\'),\'NLI model load\')\\n        nli.eval();nli.requires_grad_(False)\\n        labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n        if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n        limit=min(512,int(getattr(nli.config,\'max_position_embeddings\',512)),int(tokenizer.model_max_length))\\n        cache={};cache_hits=0\\n        def predict(premise,hypothesis):\\n            nonlocal cache_hits\\n            key=(sha(premise),sha(hypothesis))\\n            if key in cache:cache_hits+=1;return cache[key]\\n            batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n            if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n            with torch.inference_mode():\\n                probs=torch.softmax(nli(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n            cache[key]={labels[i]:float(p) for i,p in enumerate(probs)}\\n            return cache[key]\\n        calibration=with_heartbeat(lambda:calibration_controls(predict),\'NLI controls\');write_json(output/\'nli_controls.json\',calibration)\\n        semantic={};pair_checks={}\\n        for index,row in enumerate(pack[\'rows\'],1):\\n            phase(\'pair_fidelity_assessment\',completed_pairs=index-1)\\n            pair_checks[row[\'id\']]=assess(row[\'source\'],row[\'reference\'],predict)\\n        source_by_id={row[\'id\']:row[\'source\'] for row in pack[\'rows\']}\\n        for index,result in enumerate(outputs,1):\\n            phase(\'output_semantic_assessment\',completed_nli_outputs=index-1)\\n            semantic[(result[\'id\'],result[\'condition\'])]=with_heartbeat(lambda:assess(source_by_id[result[\'id\']],result[\'output\'],predict),\'NLI assessment\')\\n        records,summary=evaluate(pack,jobs,outputs,semantic,pair_checks,calibration)\\n        write_lines(output/\'metrics.private.jsonl\',records)\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=NLI_REVISION,\\n            nli_device=\'cuda:0 after Gemma unload\',nli_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,\\n            nli_cache_entries=len(cache),nli_cache_hits=cache_hits,gemma_memory=gemma_memory,\\n            nli_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)},\\n            generation={\'do_sample\':False,\'seed\':42,\'max_new_tokens\':MAX_NEW_TOKENS,\'context_limit\':CONTEXT_LIMIT,\'truncate\':False},\\n            versions={name:importlib.metadata.version(name) for name in (\'torch\',\'transformers\',\'accelerate\',\'bitsandbytes\',\'huggingface-hub\')})\\n        write_json(output/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True,completed_jobs=len(outputs),completed_nli_outputs=len(outputs),completed_pairs=len(pack[\'rows\']))\\n        write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'training_called\':False,\'input_pack_sha256\':uploaded_pack_sha,\\n            \'files\':{p.name:file_sha(p) for p in output.iterdir() if p.is_file()}})\\n    except Exception as error:\\n        failed_at=status.get(\'phase\');locations=[];tb=error.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:locations.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,failed_at=failed_at,error_code=\'concise_plan_inference_failed\',local_line_numbers=locations)\\n        raise RuntimeError(\'Concise-plan inference failed; inspect run_status.json and saved partial files. No training was called.\') from None\\n\\n\\ndef synthetic_pack():\\n    examples=[(\'Firms report revenue and investors read it.\',\'Firms report revenue. Investors read it.\'),\\n        (\'Firms report sales, investors read reports, and analysts examine costs.\',\'Firms report sales. Investors read reports. Analysts examine costs.\'),\\n        (\'Firms publish reports. Investors read them. Analysts examine costs.\',\'Firms publish reports; investors read them. Analysts examine costs.\'),\\n        (\'Firms report revenue. Investors read reports. Analysts compare costs. Managers plan investment.\',\\n         \'Firms report revenue; investors read reports. Analysts compare costs. Managers plan investment.\')]\\n    rows=[];profile_rows=[]\\n    for index,(source,reference) in enumerate(examples):\\n        plan={\'sentence_count\':len(sentences(reference)),\\n            \'sentences\':[{\'word_band\':word_band(part),\'shape_hint\':\'unconstrained\'} for part in sentences(reference)],\'max_semicolons\':reference.count(\';\')}\\n        rows.append({\'id\':f\'synthetic-{index}\',\'paper_key\':f\'SYN-{index}\',\'split\':\'train\',\'source\':source,\'reference\':reference,\\n                     \'source_sentence_count\':len(sentences(source)),\'plan\':plan})\\n        profile_rows.append({\'split\':\'train\',\'plain_english\':source,\'completion\':reference})\\n    return {\'protocol\':PROTOCOL,\'parent_training_sha256\':\'synthetic_not_frozen_corpus\',\'rows\':rows,\'style_profile\':fit_profile(profile_rows)}\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    actions=parser.add_mutually_exclusive_group()\\n    actions.add_argument(\'--prepare-pack\',action=\'store_true\');actions.add_argument(\'--dry-run\',action=\'store_true\')\\n    actions.add_argument(\'--execute-inference\',action=\'store_true\')\\n    parser.add_argument(\'--pack\',type=Path);parser.add_argument(\'--expected-pack-sha\');parser.add_argument(\'--output-dir\',type=Path,required=True)\\n    args=parser.parse_args(argv)\\n    if args.prepare_pack:prepare_pack(args.output_dir);return\\n    if args.dry_run:\\n        fresh(args.output_dir);pack=synthetic_pack();jobs=jobs_for(pack);sources={r[\'id\']:r[\'source\'] for r in pack[\'rows\']}\\n        outputs=[{\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\\n                  \'output\':sources[job[\'id\']],\'hit_output_limit\':False,\'synthetic\':True} for job in jobs]\\n        records,summary=evaluate(pack,jobs,outputs,synthetic=True)\\n        write_json(args.output_dir/\'synthetic_evaluation.json\',summary);write_lines(args.output_dir/\'synthetic_jobs.jsonl\',jobs)\\n        print(json.dumps({\'status\':\'synthetic_wiring_check_complete\',\'jobs\':len(jobs),\'model_calls\':0,\'training_called\':False}));return\\n    if args.pack is None or args.expected_pack_sha is None:parser.error(\'--pack and --expected-pack-sha are required\')\\n    pack=load_pack(args.pack,args.expected_pack_sha)\\n    if args.execute_inference:\\n        run_training(pack,args.output_dir,args.expected_pack_sha)\\n    else:\\n        fresh(args.output_dir);write_json(args.output_dir/\'preflight.json\',{\'protocol\':PROTOCOL,\'pack_sha256\':args.expected_pack_sha,\'pairs\':4,\'jobs\':8,\'training_called\':False})\\n        print(\'Preflight passed; no Secret access, model download or inference performed.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/rhetoric/experiment.py": "\\"\\"\\"Prepare rhetorical pairs and run an explicitly requested inference-only oracle.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport threading\\nimport time\\n\\nfrom fama_style.research.run_extension import read_split, synthetic, EXPECTED\\nfrom fama_style.research.signal import fit_profile\\nfrom fama_style.research.claims import sha\\nfrom .extract import paired, inspect_legacy\\nfrom .mapping import probe\\nfrom .oracle import jobs_for, generate_jobs, validate_outputs, score, PROTOCOL\\nfrom .schema import VERSION\\nfrom .semantics import NLI_MODEL, assess, calibration_controls\\n\\nROOT = Path(__file__).resolve().parents[3]\\nMODEL_ID = \'google/gemma-3-4b-it\'\\nMODEL_REVISION = \'093f9f388b31de276ce2de164bdc2081324b9767\'\\nMAX_NEW_TOKENS = 512\\nCONTEXT_LIMIT = 4096\\n\\n\\ndef write_json(path, value):\\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\\\n\', encoding=\'utf-8\')\\n    temporary.replace(path)\\n\\n\\ndef write_lines(path, rows):\\n    path.write_text(\'\'.join(json.dumps(row,ensure_ascii=False)+\'\\\\n\' for row in rows), encoding=\'utf-8\')\\n\\n\\ndef fresh(path):\\n    if path.exists() and any(path.iterdir()): raise ValueError(\'Refusing to overwrite experiment outputs\')\\n    path.mkdir(parents=True, exist_ok=True)\\n\\n\\ndef manifest(directory, synthetic_run):\\n    # Hash only outputs/code; do not load any reserved-test records.\\n    paths = sorted(Path(__file__).resolve().parent.glob(\'*.py\'))\\n    paths += sorted((ROOT/\'src/fama_style/research\').glob(\'*.py\'))\\n    paths += [ROOT/\'src/fama_style/prompting/corpus.py\']\\n    write_json(directory/\'manifest.json\', {\'protocol\': PROTOCOL, \'schema\': VERSION,\\n        \'synthetic\': synthetic_run, \'test_rows_used\': 0,\\n        \'input_split_hashes\': None if synthetic_run else EXPECTED,\\n        \'code_hashes\': {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths},\\n        \'files\': {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(directory.iterdir())\\n                  if p.is_file() and p.name!=\'manifest.json\'}})\\n\\n\\ndef prepare(train, validation, output_dir, dry_run=False):\\n    if any(row[\'split\']!=\'train\' for row in train) or any(row[\'split\']!=\'validation\' for row in validation):\\n        raise ValueError(\'Unexpected split in rhetorical experiment\')\\n    if len({r[\'id\'] for r in train+validation})!=len(train)+len(validation): raise ValueError(\'Duplicate or overlapping split IDs\')\\n    fresh(output_dir)\\n    pairs = [{\'id\': row[\'id\'], \'paper_key\': row[\'paper_key\'], \'split\':\'train\',\\n              \'annotation\': paired(row[\'plain_english\'], row[\'completion\'])} for row in train]\\n    write_lines(output_dir/\'paired_schemas.private.jsonl\', pairs)\\n    analysis = probe(pairs)\\n    write_json(output_dir/\'learnability_summary.json\', analysis)\\n    write_json(output_dir/\'legacy_inventory.json\', inspect_legacy(train))\\n    jobs = jobs_for(validation)\\n    write_lines(output_dir/\'oracle_jobs.private.jsonl\', jobs)\\n    protocol = {\'protocol\': PROTOCOL, \'status\':\'prepared_not_executed\' if not dry_run else \'synthetic_wiring_check\',\\n        \'training_pairs\': len(train), \'validation_examples\': len(validation), \'generation_jobs\': len(jobs),\\n        \'schema_version\': VERSION, \'model_id\': MODEL_ID, \'model_revision\': MODEL_REVISION,\\n        \'decoding\': {\'do_sample\': False, \'max_new_tokens\':MAX_NEW_TOKENS, \'use_cache\':True, \'seed\':42},\\n        \'context_limit\':CONTEXT_LIMIT, \'truncate\':False, \'condition_order\':\'cyclic A/B/C per validation index\',\\n        \'claim_information\':\'B/C share source evidence units; A is the ordinary frozen prompt\',\\n        \'primary_comparison\':\'C versus B; A anchors the ordinary rewrite baseline\',\\n        \'oracle_reference_wording_in_generator\':False, \'test_rows_used\':0, \'human_review_required\':False,\\n        \'gpu_training_allowed\':False, \'inference_executed\':False,\\n        \'semantic_assessment\': \'Independent bidirectional document and sentence-unit NLI after Gemma is unloaded; synthetic controls must pass; over-context pairs abstain\',\\n        \'factorial_followup\':{\'status\':\'blocked_until_oracle_support\', \'cells\':[\\n            {\'weights\':\'base\',\'mapping\':\'none\'}, {\'weights\':\'base\',\'mapping\':\'learned\'},\\n            {\'weights\':\'frozen_qlora\',\'mapping\':\'none\'}, {\'weights\':\'frozen_qlora\',\'mapping\':\'learned\'}],\\n            \'controls\':\'same source examples, model revision, quantization, budgets, prompts within mapping condition; existing adapter was not plan-conditioned\'}}\\n    write_json(output_dir/\'oracle_protocol.json\', protocol)\\n    profile = fit_profile(train)\\n    write_json(output_dir/\'style_profile.json\', profile)\\n    if dry_run:\\n        sources = {row[\'id\']: row[\'plain_english\'] for row in validation}\\n        outputs = generate_jobs(jobs, lambda job: {\'output\':sources[job[\'id\']], \'hit_output_limit\':False, \'synthetic\':True,\\n                                                    \'input_tokens\':None, \'output_tokens\':None, \'generation_seconds\':None})\\n        records, summary = score(validation, jobs, outputs, profile, synthetic=True)\\n        write_lines(output_dir/\'synthetic_outputs.private.jsonl\', outputs)\\n        write_json(output_dir/\'synthetic_wiring_summary.json\', summary)\\n    manifest(output_dir, dry_run)\\n    return jobs, profile, analysis\\n\\n\\ndef secret_token():\\n    if Path(\'/kaggle\').exists():\\n        try:\\n            from kaggle_secrets import UserSecretsClient\\n            value = UserSecretsClient().get_secret(\'HF_TOKEN\')\\n        except Exception:\\n            raise RuntimeError(\'Enable the Kaggle Secret named HF_TOKEN\') from None\\n    else:\\n        value = os.environ.get(\'HF_TOKEN\')\\n    if not isinstance(value,str) or not value.strip(): raise RuntimeError(\'Configure HF_TOKEN outside code and repository\')\\n    return value\\n\\n\\ndef with_heartbeat(call, label):\\n    stop = threading.Event(); started = time.monotonic()\\n    def tick():\\n        while not stop.wait(30): print(f\'{label}: working ({int(time.monotonic()-started)} seconds)\', flush=True)\\n    thread = threading.Thread(target=tick, daemon=True); thread.start()\\n    try: return call()\\n    finally: stop.set(); thread.join(timeout=1)\\n\\n\\ndef audit_input_ids(job, ids):\\n    if sha(job[\'prompt\'])!=job[\'prompt_sha256\']: raise ValueError(\'Changed prompt\')\\n    if not ids or len(ids)+MAX_NEW_TOKENS>CONTEXT_LIMIT: raise ValueError(\'Prompt/context overflow; no truncation allowed\')\\n    return {\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'input_tokens\':len(ids),\\n            \'prompt_sha256\':job[\'prompt_sha256\'],\'input_ids_sha256\':sha(json.dumps(ids))}\\n\\n\\ndef run_training(validation, jobs, profile, output_dir):\\n    \\"\\"\\"Inference only; name follows the repository convention for lazy ML imports.\\"\\"\\"\\n    fresh(output_dir)\\n    status = {\'complete\':False, \'training_called\':False, \'test_rows_used\':0, \'protocol\':PROTOCOL}\\n    started = time.monotonic()\\n    def phase(name, **extra):\\n        status.update(phase=name, **extra); write_json(output_dir/\'run_status.json\',status)\\n        print(\'Phase: \'+name, flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\'\\n        os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\'\\n        os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor, AutoTokenizer, AutoModelForSequenceClassification,\\n            BitsAndBytesConfig, Gemma3ForConditionalGeneration, LogitsProcessor, LogitsProcessorList)\\n        from huggingface_hub import HfApi\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:\\n            raise RuntimeError(\'Use Kaggle T4 x2; expose only its first GPU before importing torch\')\\n        torch.cuda.init(); torch.cuda.set_device(0); torch.manual_seed(42); torch.cuda.reset_peak_memory_stats(0)\\n        token = secret_token()\\n        phase(\'loading_pinned_processor\')\\n        processor = with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token), \'Processor load\')\\n        def inputs_for(job):\\n            if sha(job[\'prompt\'])!=job[\'prompt_sha256\']: raise ValueError(\'Changed prompt\')\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[]\\n        for job in jobs:\\n            batch=inputs_for(job)\\n            audits.append(audit_input_ids(job,batch[\'input_ids\'][0].tolist()))\\n        write_json(output_dir/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)), \'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):\\n            raise RuntimeError(\'Unexpected model tensor placement\')\\n        model.eval(); model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any() | torch.isposinf(scores).any() | ~torch.isfinite(scores).any(dim=-1).all()):\\n                    raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        audit_by_key={(a[\'id\'],a[\'condition\']):a for a in audits}\\n        def generate(job):\\n            batch=inputs_for(job); n=int(batch[\'input_ids\'].shape[-1]); audit=audit_by_key[(job[\'id\'],job[\'condition\'])]\\n            if sha(json.dumps(batch[\'input_ids\'][0].tolist()))!=audit[\'input_ids_sha256\']: raise ValueError(\'Prompt tokenization changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()}\\n            phase(\'generation\',completed_jobs=generate.completed,total_jobs=len(jobs))\\n            before=time.monotonic()\\n            with torch.inference_mode():\\n                out=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,\\n                    use_cache=True,pad_token_id=processor.tokenizer.pad_token_id,\\n                    logits_processor=LogitsProcessorList([FiniteScores()]),remove_invalid_values=False), \'Generation\')\\n            new=out[0,n:]; eos=model.generation_config.eos_token_id; eos=eos if isinstance(eos,list) else [eos]\\n            result={\'output\':processor.tokenizer.decode(new,skip_special_tokens=True).strip(),\\n                    \'input_tokens\':n,\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                    \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            generate.completed+=1\\n            return result\\n        generate.completed=0\\n        outputs=generate_jobs(jobs,generate,lambda rows:write_lines(output_dir/\'generations.private.jsonl\',rows))\\n        validate_outputs(jobs,outputs)\\n        gpu_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del generate,model,processor,batch\\n        torch.cuda.empty_cache()\\n        phase(\'loading_independent_nli_on_cpu\')\\n        nli_started=time.monotonic()\\n        revision=HfApi(token=False).model_info(NLI_MODEL,timeout=30).sha\\n        if not isinstance(revision,str) or not re.fullmatch(\'[0-9a-f]{40}\',revision): raise ValueError(\'Unresolved NLI revision\')\\n        tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=revision,token=False), \'NLI tokenizer load\')\\n        nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=revision,token=False), \'NLI model load\')\\n        nli.eval(); nli.requires_grad_(False); torch.set_num_threads(min(4,os.cpu_count() or 1))\\n        labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n        if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}: raise ValueError(\'Unexpected NLI labels\')\\n        limit=min(512,int(getattr(nli.config,\'max_position_embeddings\',512)),int(tokenizer.model_max_length))\\n        def predict(premise,hypothesis):\\n            batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n            if int(batch[\'input_ids\'].shape[-1])>limit:return None\\n            with torch.inference_mode(): probabilities=torch.softmax(nli(**batch).logits[0],dim=-1).tolist()\\n            return {labels[i]:float(p) for i,p in enumerate(probabilities)}\\n        calibration=with_heartbeat(lambda:calibration_controls(predict),\'NLI synthetic controls\')\\n        write_json(output_dir/\'nli_controls.json\',calibration)\\n        sources={row[\'id\']:row[\'plain_english\'] for row in validation}; semantic={}\\n        for index,row in enumerate(outputs,1):\\n            semantic[(row[\'id\'],row[\'condition\'])]=with_heartbeat(lambda:assess(sources[row[\'id\']],row[\'output\'],predict),\'NLI assessment\')\\n            phase(\'independent_semantic_assessment\',completed_nli_outputs=index,total_jobs=len(outputs))\\n        records,summary=score(validation,jobs,outputs,profile,semantic=semantic,calibration=calibration)\\n        write_lines(output_dir/\'metrics.private.jsonl\',records)\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=revision,\\n            nli_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,memory=gpu_memory,\\n            versions={name:importlib.metadata.version(name) for name in (\'torch\',\'transformers\',\'bitsandbytes\',\'accelerate\',\'huggingface-hub\')})\\n        write_json(output_dir/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True)\\n        manifest(output_dir,False)\\n    except Exception as exc:\\n        # Remote exception text/locals can contain authenticated URLs. Do not persist them.\\n        frames=[]; tb=exc.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:frames.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,error_code=\'oracle_inference_failed\',local_line_numbers=frames)\\n        raise RuntimeError(\'Oracle inference failed; inspect run_status.json and saved partial outputs. No training was called.\') from None\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    action=parser.add_mutually_exclusive_group()\\n    action.add_argument(\'--dry-run\',action=\'store_true\',help=\'Synthetic CPU wiring check, no model/Secret/network\')\\n    action.add_argument(\'--execute-inference\',action=\'store_true\',help=\'Explicitly run the 63-job Kaggle inference experiment; never training\')\\n    parser.add_argument(\'--train\',type=Path,default=ROOT/\'data/fama_corpus_v1/train.jsonl\')\\n    parser.add_argument(\'--validation\',type=Path,default=ROOT/\'data/fama_corpus_v1/validation.jsonl\')\\n    parser.add_argument(\'--output-dir\',required=True,type=Path)\\n    args=parser.parse_args(argv)\\n    if args.dry_run:\\n        train,validation=synthetic()\\n        for index,row in enumerate(train):row[\'paper_key\']=f\'SYN-{index}\'\\n    else:train,validation=read_split(args.train,\'train\'),read_split(args.validation,\'validation\')\\n    fresh(args.output_dir)\\n    jobs,profile,analysis=prepare(train,validation,args.output_dir/\'preparation\',args.dry_run)\\n    if args.execute_inference:run_training(validation,jobs,profile,args.output_dir/\'inference\')\\n    print(json.dumps({\'status\':\'synthetic_check_complete\' if args.dry_run else \'inference_complete\' if args.execute_inference else \'prepared_not_executed\',\\n                      \'training_pairs\':len(train),\'validation_examples\':len(validation),\'jobs\':len(jobs),\'training_called\':False,\'test_rows_used\':0}))\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/rhetoric/extract.py": "\\"\\"\\"Independent surface annotation, paired hypotheses, and constrained operations.\\"\\"\\"\\nimport collections\\nimport json\\nimport re\\n\\nfrom fama_style.research.claims import ledger, sha, tokens, sentences, conditional, variants, verify, risk_flags\\nfrom fama_style.research.signal import overlap\\nfrom .schema import VERSION, EXTRACTOR, validate_links, validate\\n\\nPROTECTED_FEATURES = {\'hedging\', \'citation\'}\\n\\n\\ndef count_bin(n):\\n    return \'one\' if n == 1 else \'two\' if n == 2 else \'three_plus\'\\n\\n\\ndef rate_bin(n, total):\\n    return \'none\' if not n else \'low\' if n/max(1, total) <= .025 else \'high\'\\n\\n\\ndef role(text):\\n    \\"\\"\\"Cue observations, never assertions that a relation is semantically valid.\\"\\"\\"\\n    low = text.lower()\\n    if re.search(r\'\\\\b(?:is defined as|we define|denotes)\\\\b\', low): return \'definition\'\\n    if re.match(r\'(?:however|nevertheless|in contrast|but)\\\\b\', low): return \'contrast\'\\n    if re.match(r\'(?:therefore|thus|hence|in sum)\\\\b\', low): return \'conclusion\'\\n    if low.startswith((\'if \', \'unless \', \'when \')): return \'condition\'\\n    if re.search(r\'\\\\b(?:we find|results show|table \\\\d|we estimate|we test)\\\\b\', low): return \'reporting\'\\n    return \'assertion\'\\n\\n\\ndef extract(text):\\n    if not isinstance(text, str) or not text.strip(): raise ValueError(\'Empty paragraph\')\\n    claims = ledger(text)\\n    parts = sentences(text)\\n    units, moves = [], []\\n    for index, part in enumerate(parts, 1):\\n        subordinated = bool(re.search(r\'\\\\b(if|unless|when|although|because|which|that|while|whereas)\\\\b\', part, re.I))\\n        coordinated = bool(re.search(r\';|,\\\\s+(?:and|but|or)\\\\b\', part, re.I))\\n        syntax = \'mixed\' if subordinated and coordinated else \'subordinated\' if subordinated else \'coordinated\' if coordinated else \'simple\'\\n        position = \'none\'\\n        if re.search(r\'\\\\b(if|unless|when|provided that)\\\\b\', part, re.I):\\n            position = \'front\' if re.match(r\'(if|unless|when|provided that)\\\\b\', part, re.I) else \'end\' if conditional(part) else \'embedded\'\\n        n = len(tokens(part)); unit = f\'s{index}\'; refs = [f\'c{index}\']\\n        units.append({\'unit\': unit, \'claims\': refs, \'syntax\': syntax,\\n                      \'condition_position\': position, \'voice\': \'present\' if re.search(r\'\\\\b(we|our|us)\\\\b\', part, re.I) else \'absent\',\\n                      \'length\': \'short\' if n <= 16 else \'medium\' if n <= 32 else \'long\'})\\n        moves.append({\'unit\': unit, \'claims\': refs, \'role\': role(part)})\\n    roles = {move[\'role\'] for move in moves}\\n    specials = roles & {\'contrast\', \'condition\', \'reporting\'}\\n    organization = \'mixed\' if len(specials)>1 else \'contrastive\' if \'contrast\' in roles else \'conditional\' if \'condition\' in roles else \'reporting\' if \'reporting\' in roles else \'single\' if len(parts)==1 else \'sequence\'\\n    words = tokens(text); n = len(words)\\n    structure = {\'version\': VERSION,\\n        \'paragraph\': {\'organization\': organization, \'sentence_count\': count_bin(len(parts)), \'moves\': moves},\\n        \'sentences\': units,\\n        \'language\': {\'author_voice\': \'present\' if any(unit[\'voice\']==\'present\' for unit in units) else \'absent\',\\n            \'semicolon\': rate_bin(text.count(\';\'), n), \'parenthesis\': rate_bin(text.count(\'(\'), n),\\n            \'hedging\': rate_bin(len(re.findall(r\'\\\\b(may|might|could|possibly|perhaps)\\\\b\', text, re.I)), n),\\n            \'citation\': rate_bin(len(re.findall(r\'\\\\b(?:18|19|20)\\\\d{2}\\\\b\', text)), n),\\n            \'nominalization\': rate_bin(sum(bool(re.search(r\'(tion|sion|ment|ance|ence|ity)$\', word)) for word in words), n)}}\\n    validate_links(structure, claims)\\n    return {\'structure\': structure, \'semantic_ledger\': claims,\\n        \'provenance\': {\'specification\': VERSION, \'extractor\': EXTRACTOR, \'input_sha256\': sha(text),\\n                       \'annotation_method\': \'deterministic_surface_rules\', \'independent_of_paired_text\': True},\\n        \'checks\': {\'claim_links_valid\': True, \'semantic_parse_verified\': False,\\n                   \'atomic_claim_coverage\': \'unresolved\', \'unsupported_relation_creation_allowed\': False}}\\n\\n\\ndef flatten(structure):\\n    validate(structure)\\n    sentence = structure[\'sentences\']\\n    def majority(key):\\n        counts = collections.Counter(row[key] for row in sentence)\\n        return sorted(counts, key=lambda item:(-counts[item], item))[0]\\n    return {\'organization\': structure[\'paragraph\'][\'organization\'], \'sentence_count\': structure[\'paragraph\'][\'sentence_count\'],\\n            \'syntax\': majority(\'syntax\'), \'condition_position\': majority(\'condition_position\'),\\n            \'sentence_length\': majority(\'length\'), **structure[\'language\']}\\n\\n\\ndef paired(source, target):\\n    \\"\\"\\"Extract separately; alignment is an explicitly unverified lexical hypothesis.\\"\\"\\"\\n    a, b = extract(source), extract(target)\\n    hypotheses = []\\n    for target_claim in b[\'semantic_ledger\'][\'claims\']:\\n        ranked = sorted(((overlap(claim[\'evidence\'][\'text\'], target_claim[\'evidence\'][\'text\']), claim[\'claim_id\'])\\n                         for claim in a[\'semantic_ledger\'][\'claims\']), reverse=True)\\n        score, claim_id = ranked[0]\\n        hypotheses.append({\'target_claim\': target_claim[\'claim_id\'], \'source_claim\': claim_id if score >= .35 else None,\\n                           \'similarity\': score, \'status\': \'lexical_hypothesis_not_semantically_validated\'})\\n    fa, fb = flatten(a[\'structure\']), flatten(b[\'structure\'])\\n    refs = [claim[\'claim_id\'] for claim in a[\'semantic_ledger\'][\'claims\']]\\n    changes = [{\'operation\': \'preserve_source_feature\' if key in PROTECTED_FEATURES else \'set_template_feature\',\\n                \'feature\': key, \'from\': fa[key], \'to\': fa[key] if key in PROTECTED_FEATURES else fb[key],\\n                \'observed_target\': fb[key], \'status\': \'blocked_semantic_sensitive\' if key in PROTECTED_FEATURES else \'structural_hypothesis\',\\n                \'source_claim_refs\': refs, \'scope\': \'planning_only\', \'introduces_semantic_relation\': False}\\n               for key in fa if fa[key] != fb[key]]\\n    # Programs are replayable surface controls, not a claim that they realize the target.\\n    programs = [{\'operation\': item[\'operation\'], \'source_claim_refs\': refs,\\n                 \'output_sha256\': sha(item[\'text\']), \'verification\': verify(source, item[\'text\']),\\n                 \'target_feature_matches\': sum(value == flatten(extract(item[\'text\'])[\'structure\'])[key] for key,value in fb.items())}\\n                for item in variants(source) if item[\'operation\']!=\'identity\']\\n    return {\'source\': a, \'target\': b, \'claim_alignment\': hypotheses, \'template_changes\': changes,\\n            \'executable_surface_controls\': programs,\\n            \'consistency\': {\'source_target_surface_flags\': risk_flags(source, target),\\n                            \'semantic_alignment_verified\': False, \'claim_reordering_allowed\': False}}\\n\\n\\ndef execute_operation(text, operation, claim_refs, expected_source_sha):\\n    \\"\\"\\"An operation can touch only this source\'s known claims and bounded grammar.\\"\\"\\"\\n    if sha(text) != expected_source_sha: raise ValueError(\'Changed operation source\')\\n    known = {c[\'claim_id\'] for c in ledger(text)[\'claims\']}\\n    if not claim_refs or set(claim_refs) != known: raise ValueError(\'Invalid operation claim scope\')\\n    candidate = next((item for item in variants(text) if item[\'operation\']==operation), None)\\n    if candidate is None or verify(text, candidate[\'text\'])[\'status\']!=\'surface_preserved\':\\n        raise ValueError(\'Unsupported semantic or structural operation\')\\n    return candidate[\'text\']\\n\\n\\ndef inspect_legacy(rows):\\n    shapes = collections.Counter(); labels = collections.Counter()\\n    for row in rows:\\n        value = row.get(\'rhetorical_taxonomy\')\\n        shape = \'missing\' if value is None else \',\'.join(sorted(value)) if isinstance(value, dict) else \'unexpected\'\\n        shapes[shape] += 1\\n        if isinstance(value, dict):\\n            # Inventory only pre-existing finite objectives, not private descriptions.\\n            top = value.get(\'level_1\', value.get(\'R1\', {}))\\n            objective = top.get(\'discourse_objective\') if isinstance(top, dict) else None\\n            if isinstance(objective, str) and re.fullmatch(r\'[A-Z_]{1,32}\', objective): labels[objective] += 1\\n    return {\'rows\': len(rows), \'formats\': dict(shapes), \'objective_counts\': dict(labels),\\n            \'reused_as_source_labels\': False, \'migration\': \'preserved unchanged; discourse objective/moves/relations do not match new three-level axes\'}\\n", "src/fama_style/rhetoric/ledger_ablation.py": "\\"\\"\\"Two-case, 2x2 ledger-format/grouping ablation; inference only, post-hoc.\\"\\"\\"\\nimport argparse\\nfrom collections import Counter\\nimport copy\\nimport gc\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport statistics\\nimport time\\n\\nfrom . import soft_guidance_experiment as parent\\nfrom .soft_guidance import jobs_for as parent_jobs\\nfrom .claim_constrained import constraint_ledger\\nfrom .concise_plan import TRAIN_SHA,MODEL_ID,MODEL_REVISION,NLI_MODEL,NLI_REVISION,file_sha,read_lines\\nfrom .experiment import fresh,write_json,write_lines,secret_token,with_heartbeat\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research import claim_verifier_v3 as verifier\\nfrom fama_style.research.claims import sha,sentences,tokens\\nfrom fama_style.research.signal import distance,overlap,features\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'ledger_grouping_ablation_v1\'\\nMAX_NEW_TOKENS=768\\nCONTEXT_LIMIT=4096\\nVERIFIER_FOLDER=ROOT/\'docs/claim_verifier_calibration_v3\'\\nCONDITIONS=(\'A_sentence_only\',\'B_sentence_grouping\',\'C_cue_rich\',\'D_cue_rich_grouping\')\\nFACTORS={c:{\'ledger\':\'sentence_evidence\' if c.startswith((\'A_\',\'B_\')) else \'cue_rich\',\\n    \'grouping\':c.startswith((\'B_\',\'D_\'))} for c in CONDITIONS}\\naudit_tokens=parent.audit_tokens\\nparagraph_format=parent.paragraph_format\\nruntime_paths=parent.runtime_paths\\n\\n\\ndef jobs_for(pack):\\n    rows=pack[\'rows\']\\n    if len(rows)!=2 or len({r[\'id\'] for r in rows})!=2 or any(r[\'split\']!=\'validation\' for r in rows):raise ValueError(\'Expected two unique development rows\')\\n    jobs=[]\\n    for index,row in enumerate(rows):\\n        rich=constraint_ledger(row[\'source\']);simple=copy.deepcopy(rich)\\n        for claim in simple[\'claims\']:del claim[\'protected_evidence\']\\n        rich_json=json.dumps(rich,ensure_ascii=False,separators=(\',\',\':\'))\\n        simple_json=json.dumps(simple,ensure_ascii=False,separators=(\',\',\':\'))\\n        for c in CONDITIONS if index==0 else CONDITIONS[::-1]:\\n            factors=FACTORS[c];prompt=row[\'parent_prompts\'][\'grouping\' if factors[\'grouping\'] else \'no_grouping\']\\n            if prompt.count(rich_json)!=1:raise ValueError(\'Original prompt ledger mismatch\')\\n            if not prompt.startswith(make_prompt(row[\'source\'])):raise ValueError(\'Original source prefix mismatch\')\\n            if factors[\'ledger\']==\'sentence_evidence\':prompt=prompt.replace(rich_json,simple_json,1)\\n            jobs.append({\'id\':row[\'id\'],\'condition\':c,\'prompt\':prompt,\'prompt_sha256\':sha(prompt),\'source_sha256\':sha(row[\'source\']),**factors,\'oracle\':False})\\n    return jobs\\n\\n\\ndef prepare_pack(output):\\n    base=ROOT/\'research_runs/soft_guidance_validation_v2\';parent_path=base/\'input_pack/soft_guidance_pack.json\'\\n    release=json.loads((base/\'release_verification.json\').read_text());prior=parent.load_pack(parent_path,release[\'pack_sha256\'])\\n    run=base/\'kaggle_version_1/soft_guidance_validation_v2\';review_path=base/\'version_1_review/verification.json\';review=json.loads(review_path.read_text())\\n    if not review[\'checks\'] or not all(review[\'checks\'].values()):raise ValueError(\'Prior run review failed\')\\n    for name,digest in review[\'artifact_hashes\'].items():\\n        if file_sha(run/name)!=digest:raise ValueError(\'Reviewed result changed\')\\n    metrics=read_lines(run/\'metrics.private.jsonl\');ranked=sorted((r for r in metrics if r[\'condition\']==\'B_claims_soft_plan\'),key=lambda r:(r[\'style_gain_proxy\'],r[\'id\']))[:2]\\n    chosen={r[\'id\'] for r in ranked};original_jobs={(j[\'id\'],j[\'condition\']):j for j in parent_jobs(prior)}\\n    outputs={(r[\'id\'],r[\'condition\']):r for r in read_lines(run/\'generations.private.jsonl\')}\\n    rows=[]\\n    for row in prior[\'rows\']:\\n        if row[\'id\'] not in chosen:continue\\n        rows.append({**row,\'parent_prompts\':{\'no_grouping\':original_jobs[(row[\'id\'],\'A_claims_only\')][\'prompt\'],\'grouping\':original_jobs[(row[\'id\'],\'B_claims_soft_plan\')][\'prompt\']},\\n            \'previous_rich_output_sha256\':{\'C_cue_rich\':sha(outputs[(row[\'id\'],\'A_claims_only\')][\'output\']),\'D_cue_rich_grouping\':sha(outputs[(row[\'id\'],\'B_claims_soft_plan\')][\'output\'])}})\\n    pack={k:copy.deepcopy(prior[k]) for k in (\'parent_training_sha256\',\'validation_sha256\',\'style_profile\',\'execution_gate\',\'verifier_evidence\',\'generation\')}\\n    pack.update(protocol=PROTOCOL,rows=rows,parent_pack_sha256=file_sha(parent_path),\\n        selection={\'post_hoc\':True,\'rule\':\'Two lowest B style-gain proxies in reviewed 21-example run; ascending score then ID; both exposed failures\',\\n            \'review_sha256\':file_sha(review_path),\'parent_metrics_sha256\':file_sha(run/\'metrics.private.jsonl\'),\'held_out_test_rows_read\':0},\\n        scope=\'Post-hoc two-case mechanism diagnostic; same master instruction, source evidence, model, decoding and grouping texts. Only protected_evidence fields are removed in the simple-ledger arms.\')\\n    jobs=jobs_for(pack);pack[\'runtime_code_files\']={str(p.relative_to(ROOT)):file_sha(p) for p in runtime_paths()};pack[\'jobs_sha256\']=sha(json.dumps(jobs,sort_keys=True,ensure_ascii=False))\\n    fresh(output);path=output/\'ledger_ablation_pack.json\';write_json(path,pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'status\':\'prepared_not_executed\',\'pairs\':2,\'jobs\':8,\'pack_sha256\':file_sha(path),\\n        \'training_called\':False,\'model_calls\':0,\'reserved_test_rows_read\':0,\'parent_pack_sha256\':file_sha(parent_path),\\n        \'files\':{p.name:file_sha(p) for p in output.iterdir() if p.is_file()}})\\n    print(json.dumps({\'status\':\'prepared_not_executed\',\'pairs\':2,\'jobs\':8,\'pack_sha256\':file_sha(path)}));return pack\\n\\n\\ndef load_pack(path,expected_sha):\\n    if not re.fullmatch(\'[0-9a-f]{64}\',expected_sha) or file_sha(path)!=expected_sha:raise ValueError(\'Input pack hash mismatch\')\\n    pack=json.loads(path.read_text())\\n    if pack[\'protocol\']!=PROTOCOL or pack[\'parent_training_sha256\']!=TRAIN_SHA or pack[\'validation_sha256\']!=parent.VALIDATION_SHA:raise ValueError(\'Wrong frozen protocol/data\')\\n    if pack[\'execution_gate\'][\'ready\'] is not True or pack[\'style_profile\'][\'fit_split\']!=\'train\' or pack[\'style_profile\'][\'training_rows\']!=247:raise ValueError(\'Wrong gate/profile\')\\n    for name,h in pack[\'runtime_code_files\'].items():\\n        if file_sha(ROOT/name)!=h:raise ValueError(\'Frozen implementation changed\')\\n    protocol,_=verifier.validate_protocol(VERIFIER_FOLDER);evidence=pack[\'verifier_evidence\']\\n    if evidence[\'protocol_sha256\']!=file_sha(VERIFIER_FOLDER/\'protocol.json\') or evidence[\'code_files\']!=protocol[\'code_files\'] or evidence[\'fixture_files\']!=protocol[\'fixture_files\']:raise ValueError(\'Verifier changed\')\\n    if not verifier.gate(evidence[\'held_out_metrics\'],protocol[\'readiness_gate\'])[\'passed\']:raise ValueError(\'Verifier gate failed\')\\n    if pack[\'generation\']!={\'model_id\':MODEL_ID,\'revision\':MODEL_REVISION,\'max_new_tokens\':768,\'context_limit\':4096,\'do_sample\':False,\'seed\':42,\'truncate\':False}:raise ValueError(\'Changed generation controls\')\\n    if pack[\'jobs_sha256\']!=sha(json.dumps(jobs_for(pack),sort_keys=True,ensure_ascii=False)):raise ValueError(\'Prepared prompts changed\')\\n    return pack\\n\\n\\ndef artifact_checks(source,text):\\n    cues={x[\'evidence\'].lower() for c in constraint_ledger(source)[\'claims\'] for x in c[\'protected_evidence\'] if x[\'kind\']!=\'quantities\' and len(x[\'evidence\'].split())<=3}\\n    quoted=[]\\n    for cue in sorted(cues):\\n        pattern=r\\"(?<!\\\\w)[‘\\\\\\"\']\\"+re.escape(cue)+r\\"[’\\\\\\"\'](?!\\\\w)\\"\\n        if re.search(pattern,text,re.I) and not re.search(pattern,source,re.I):quoted.append(cue)\\n    terms=(\'modality\',\'qualifier\',\'quantifier\',\'ledger\',\'annotation\',\'source evidence\',\'claim_id\',\'protected_evidence\')\\n    labels=[t for t in terms if re.search(r\'\\\\b\'+re.escape(t)+r\'\\\\b\',text,re.I) and not re.search(r\'\\\\b\'+re.escape(t)+r\'\\\\b\',source,re.I)]\\n    year_phrases=re.findall(r\'\\\\b(?:quantitative|statistical|numerical)\\\\s+evidence\\\\s+(?:of|for|from)\\\\s+(?:the\\\\s+year\\\\s+)?(?:18|19|20)\\\\d{2}\\\\b\',text,re.I)\\n    year_phrases=[p for p in year_phrases if p.lower() not in source.lower()]\\n    return {\'artifact_warning\':bool(quoted or labels or year_phrases),\'newly_quoted_protected_cues\':quoted,\'new_metalinguistic_terms\':labels,\\n        \'new_numeric_evidence_phrases\':year_phrases,\'semantic_proof\':False,\'meaning\':\'Predeclared surface artifact screen, not confirmed added-claim detection\'}\\n\\n\\ndef evaluate(pack,jobs,outputs,semantic=None,pair_checks=None,canary_passed=False,synthetic=False):\\n    if jobs!=jobs_for(pack) or len(outputs)!=8:raise ValueError(\'Changed jobs or incomplete generation set\')\\n    expected={(j[\'id\'],j[\'condition\']):j for j in jobs};seen=set();rows={r[\'id\']:r for r in pack[\'rows\']};records=[]\\n    if canary_passed and not synthetic and set(semantic or {})!=set(expected):raise ValueError(\'Incomplete verifier results\')\\n    for out in outputs:\\n        key=(out[\'id\'],out[\'condition\']);job=expected.get(key)\\n        if key in seen or job is None or out[\'prompt_sha256\']!=job[\'prompt_sha256\']:raise ValueError(\'Duplicate/swapped output\')\\n        if not out.get(\'output\',\'\').strip() or out.get(\'hit_output_limit\') is not False or out.get(\'synthetic\') is not synthetic:raise ValueError(\'Empty/cutoff/mislabeled output\')\\n        seen.add(key);row=rows[out[\'id\']];text=out[\'output\'];v=(semantic or {}).get(key);valid=paragraph_format(text);status=v[\'status\'] if v else \'not_run\'\\n        if v and status not in (\'model_consistent\',\'risk_signal\',\'abstain\'):raise ValueError(\'Unknown verifier status\')\\n        usable=canary_passed and not synthetic and v is not None\\n        records.append({\'id\':out[\'id\'],\'condition\':out[\'condition\'],**FACTORS[out[\'condition\']],**artifact_checks(row[\'source\'],text),\\n            \'format_valid\':valid,\'verifier_status\':status,\'verifier\':v,\'output_sha256\':sha(text),\\n            \'source_sentence_count\':len(sentences(row[\'source\'])),\'output_sentence_count\':len(sentences(text)),\\n            \'source_features\':features(row[\'source\']),\'output_features\':features(text),\'reference_features_diagnostic_only\':features(row[\'reference\']),\\n            \'style_gain_proxy\':distance(row[\'source\'],pack[\'style_profile\'])-distance(text,pack[\'style_profile\']),\\n            \'source_similarity\':overlap(row[\'source\'],text),\'near_copy\':overlap(row[\'source\'],text)>=.9,\\n            \'exact_copy\':\' \'.join(row[\'source\'].split())==\' \'.join(text.split()),\\n            \'valid_semantic_proxy_pass\':valid and status==\'model_consistent\' if usable else None,\\n            \'previous_rich_output_hash_matches\':sha(text)==row.get(\'previous_rich_output_sha256\',{}).get(out[\'condition\']) if out[\'condition\'].startswith((\'C_\',\'D_\')) else None,\\n            \'input_tokens\':out.get(\'input_tokens\',0),\'output_tokens\':out.get(\'output_tokens\',0),\'generation_seconds\':out.get(\'generation_seconds\',0)})\\n    conditions={}\\n    for c in CONDITIONS:\\n        group=[r for r in records if r[\'condition\']==c]\\n        conditions[c]={\'rows\':2,\'artifact_warning_outputs\':sum(r[\'artifact_warning\'] for r in group),\'single_sentence_outputs\':sum(r[\'output_sentence_count\']==1 for r in group),\\n            \'mean_words_per_sentence\':statistics.mean(r[\'output_features\'][\'words_per_sentence\'] for r in group),\\n            \'mean_style_gain_proxy\':statistics.mean(r[\'style_gain_proxy\'] for r in group),\'mean_source_similarity\':statistics.mean(r[\'source_similarity\'] for r in group),\\n            \'near_copies\':sum(r[\'near_copy\'] for r in group),\'valid_formats\':sum(r[\'format_valid\'] for r in group),\\n            \'verifier_status_counts\':dict(Counter(r[\'verifier_status\'] for r in group)),\\n            \'valid_semantic_proxy_passes\':sum(bool(r[\'valid_semantic_proxy_pass\']) for r in group) if canary_passed and not synthetic else None,\\n            \'rich_control_hash_matches\':sum(r[\'previous_rich_output_hash_matches\'] is True for r in group) if c.startswith((\'C_\',\'D_\')) else None,\\n            **{k:sum(r[k] or 0 for r in group) for k in (\'input_tokens\',\'output_tokens\',\'generation_seconds\')}}\\n    lookup={(r[\'id\'],r[\'condition\']):r for r in records};contrasts=[]\\n    for row in pack[\'rows\']:\\n        for label,a,b in ((\'remove_cues_without_grouping\',\'C_cue_rich\',\'A_sentence_only\'),(\'remove_cues_with_grouping\',\'D_cue_rich_grouping\',\'B_sentence_grouping\'),\\n                          (\'add_grouping_sentence_evidence\',\'A_sentence_only\',\'B_sentence_grouping\'),(\'add_grouping_cue_rich\',\'C_cue_rich\',\'D_cue_rich_grouping\')):\\n            before,after=lookup[(row[\'id\'],a)],lookup[(row[\'id\'],b)]\\n            contrasts.append({\'id\':row[\'id\'],\'contrast\':label,\'before\':a,\'after\':b,\\n                \'artifact_before\':before[\'artifact_warning\'],\'artifact_after\':after[\'artifact_warning\'],\\n                \'words_per_sentence_delta\':after[\'output_features\'][\'words_per_sentence\']-before[\'output_features\'][\'words_per_sentence\'],\\n                \'style_gain_delta\':after[\'style_gain_proxy\']-before[\'style_gain_proxy\'],\\n                \'verifier_before\':before[\'verifier_status\'],\'verifier_after\':after[\'verifier_status\']})\\n    return records,{\'protocol\':PROTOCOL,\'execution_kind\':\'synthetic_wiring_check\' if synthetic else \'model_inference\',\\n        \'conditions\':conditions,\'within_case_contrasts\':contrasts,\'runtime_canary_passed\':canary_passed,\'pair_source_reference_verifier\':pair_checks,\\n        \'training_called\':False,\'reserved_test_rows_used\':0,\'development_validation_rows_used\':2,\'training_ready\':False,\\n        \'selection_post_hoc\':True,\'semantic_accuracy\':None,\'author_style_accuracy\':None,\\n        \'limitations\':\'Two already-observed failures, eight first attempts; causal prompt-factor diagnostic, not held-out performance. Artifact markers, NLI and style metrics are imperfect proxies. No repairs or sentence quotas.\'}\\n\\n\\ndef run_training(pack,output,uploaded_pack_sha):\\n    \\"\\"\\"Inference only. Check public verifier canary before accessing the Gemma Secret.\\"\\"\\"\\n    fresh(output);jobs=jobs_for(pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'uploaded_pack_identity.json\',{\'sha256\':uploaded_pack_sha,\'format\':\'uploaded file bytes\'})\\n    status={\'complete\':False,\'protocol\':PROTOCOL,\'training_called\':False,\'reserved_test_rows_used\':0,\'completed_jobs\':0,\'total_jobs\':len(jobs)}\\n    started=time.monotonic();outputs=[]\\n    def phase(name,**extra):\\n        status.update(phase=name,**extra);write_json(output/\'run_status.json\',status);print(\'Phase: \'+name,flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        import os\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\';os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor,AutoTokenizer,AutoModelForSequenceClassification,\\n            Gemma3ForConditionalGeneration,BitsAndBytesConfig,LogitsProcessor,LogitsProcessorList)\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Expose one Kaggle GPU before importing torch\')\\n        torch.cuda.init();torch.cuda.set_device(0);torch.manual_seed(42)\\n        if \'T4\' not in torch.cuda.get_device_name(0):raise RuntimeError(\'Select Kaggle T4 x2\')\\n        cache={};cache_hits=0\\n        def load_nli():\\n            tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n            nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,\\n                token=False,dtype=torch.float32).to(\'cuda:0\'),\'NLI model load\')\\n            nli.eval();nli.requires_grad_(False);labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n            if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n            limit=min(512,int(tokenizer.model_max_length),int(getattr(nli.config,\'max_position_embeddings\',512)))\\n            def predict(premise,hypothesis):\\n                nonlocal cache_hits\\n                key=(sha(premise),sha(hypothesis))\\n                if key in cache:cache_hits+=1;return cache[key]\\n                batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n                if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n                with torch.inference_mode():\\n                    probs=torch.softmax(nli(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n                cache[key]={labels[i]:float(v) for i,v in enumerate(probs)};return cache[key]\\n            return tokenizer,nli,predict\\n        phase(\'runtime_verifier_canary\');torch.cuda.reset_peak_memory_stats(0);canary_started=time.monotonic()\\n        nli_tokenizer,nli,predict=load_nli()\\n        canary=verifier.evaluate(VERIFIER_FOLDER,output/\'runtime_calibration\',predict,\'pinned_nli_inference\')\\n        canary_seconds=time.monotonic()-canary_started\\n        if canary[\'readiness_gate\'][\'passed\'] is not True:raise RuntimeError(\'Runtime verifier canary failed; no Gemma generation performed\')\\n        del nli_tokenizer,nli,predict;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'checking_secret\');token=secret_token()\\n        processor=with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token),\'Processor load\')\\n        def tokenize(job):\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[audit_tokens(job,tokenize(job)[\'input_ids\'][0].tolist()) for job in jobs];write_json(output/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)),\'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):raise RuntimeError(\'Unexpected model placement\')\\n        model.eval();model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        by_key={(r[\'id\'],r[\'condition\']):r for r in audits}\\n        for job in jobs:\\n            phase(\'generation\',completed_jobs=len(outputs));batch=tokenize(job);ids=batch[\'input_ids\'][0].tolist()\\n            if audit_tokens(job,ids)!=by_key[(job[\'id\'],job[\'condition\'])]:raise ValueError(\'Tokenized prompt changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()};before=time.monotonic()\\n            with torch.inference_mode():\\n                generated=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,use_cache=True,\\n                    pad_token_id=processor.tokenizer.pad_token_id,remove_invalid_values=False,\\n                    logits_processor=LogitsProcessorList([FiniteScores()])),\'Generation\')\\n            new=generated[0,len(ids):];eos=model.generation_config.eos_token_id;eos=eos if isinstance(eos,list) else [eos]\\n            text=processor.tokenizer.decode(new,skip_special_tokens=True).strip()\\n            result={\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\'output\':text,\\n                \'input_tokens\':len(ids),\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            outputs.append(result);write_lines(output/\'generations.private.jsonl\',outputs)\\n            if not text or result[\'hit_output_limit\']:raise RuntimeError(\'Empty/cutoff output; partial results remain incomplete\')\\n        gemma_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del model,processor,batch,generated,new,tokenize;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'loading_verifier_after_gemma_unload\');nli_started=time.monotonic();nli_tokenizer,nli,predict=load_nli()\\n        semantic={};pair_checks={};sources={r[\'id\']:r[\'source\'] for r in pack[\'rows\']}\\n        for index,row in enumerate(pack[\'rows\'],1):\\n            phase(\'pair_fidelity_assessment\',completed_pairs=index-1)\\n            pair_checks[row[\'id\']]=with_heartbeat(lambda:verifier.verify(row[\'source\'],row[\'reference\'],predict),\'Pair verifier\')\\n        for index,result in enumerate(outputs,1):\\n            phase(\'output_semantic_assessment\',completed_verifier_outputs=index-1)\\n            semantic[(result[\'id\'],result[\'condition\'])]=with_heartbeat(lambda:verifier.verify(sources[result[\'id\']],result[\'output\'],predict),\'Output verifier\')\\n        records,summary=evaluate(pack,jobs,outputs,semantic,pair_checks,canary_passed=True)\\n        write_lines(output/\'metrics.private.jsonl\',records)\\n        write_lines(output/\'nli_probability_cache.private.jsonl\',[{\'premise_sha256\':a,\'hypothesis_sha256\':b,\'probabilities\':p} for (a,b),p in cache.items()])\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=NLI_REVISION,\\n            canary_seconds=canary_seconds,verifier_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,\\n            nli_cache_entries=len(cache),nli_cache_hits=cache_hits,gemma_memory=gemma_memory,\\n            nli_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)},\\n            generation=pack[\'generation\'],versions={n:importlib.metadata.version(n) for n in (\'torch\',\'transformers\',\'accelerate\',\'bitsandbytes\',\'huggingface-hub\')})\\n        write_json(output/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True,completed_jobs=8,completed_pairs=2,completed_verifier_outputs=8)\\n        write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'training_called\':False,\'input_pack_sha256\':uploaded_pack_sha,\\n            \'files\':{str(p.relative_to(output)):file_sha(p) for p in output.rglob(\'*\') if p.is_file()}})\\n    except Exception as error:\\n        failed_at=status.get(\'phase\');locations=[];tb=error.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:locations.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,failed_at=failed_at,error_code=\'ledger_ablation_inference_failed\',local_line_numbers=locations)\\n        raise RuntimeError(\'Ledger ablation inference failed; inspect run_status.json and saved partial files. No training was called.\') from None\\n\\n\\ndef synthetic_pack():\\n    prior=parent.synthetic_pack();original={(j[\'id\'],j[\'condition\']):j for j in parent_jobs(prior)}\\n    rows=[]\\n    for r in prior[\'rows\'][:2]:\\n        rows.append({**r,\'parent_prompts\':{\'no_grouping\':original[(r[\'id\'],\'A_claims_only\')][\'prompt\'],\'grouping\':original[(r[\'id\'],\'B_claims_soft_plan\')][\'prompt\']},\\n            \'previous_rich_output_sha256\':{\'C_cue_rich\':sha(r[\'source\']),\'D_cue_rich_grouping\':sha(r[\'source\'])}})\\n    return {\'protocol\':PROTOCOL,\'rows\':rows,\'style_profile\':prior[\'style_profile\']}\\n\\n\\ndef main(argv=None):\\n    p=argparse.ArgumentParser(description=__doc__);actions=p.add_mutually_exclusive_group()\\n    actions.add_argument(\'--prepare-pack\',action=\'store_true\');actions.add_argument(\'--dry-run\',action=\'store_true\');actions.add_argument(\'--execute-inference\',action=\'store_true\')\\n    p.add_argument(\'--pack\',type=Path);p.add_argument(\'--expected-pack-sha\');p.add_argument(\'--output-dir\',type=Path,required=True);a=p.parse_args(argv)\\n    if a.prepare_pack:prepare_pack(a.output_dir);return\\n    if a.dry_run:\\n        pack=synthetic_pack();jobs=jobs_for(pack);rows={r[\'id\']:r for r in pack[\'rows\']};fresh(a.output_dir)\\n        outputs=[{\'id\':j[\'id\'],\'condition\':j[\'condition\'],\'prompt_sha256\':j[\'prompt_sha256\'],\'output\':rows[j[\'id\']][\'source\'],\'hit_output_limit\':False,\'synthetic\':True} for j in jobs]\\n        records,summary=evaluate(pack,jobs,outputs,synthetic=True)\\n        write_json(a.output_dir/\'synthetic_evaluation.json\',summary);write_lines(a.output_dir/\'synthetic_jobs.jsonl\',jobs)\\n        print(\'Synthetic wiring check: 8 jobs; no model calls or training.\');return\\n    if a.pack is None or a.expected_pack_sha is None:p.error(\'--pack and --expected-pack-sha required\')\\n    pack=load_pack(a.pack,a.expected_pack_sha)\\n    if a.execute_inference:run_training(pack,a.output_dir,a.expected_pack_sha)\\n    else:\\n        fresh(a.output_dir);write_json(a.output_dir/\'preflight.json\',{\'protocol\':PROTOCOL,\'pack_sha256\':a.expected_pack_sha,\'pairs\':2,\'jobs\':8,\'model_calls\':0,\'training_called\':False})\\n        print(\'Preflight passed; no Secret access, model download or inference.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/rhetoric/mapping.py": "\\"\\"\\"Training-only learnability probes; no mapping is deployed before oracle evidence.\\"\\"\\"\\nimport collections\\nimport math\\nimport statistics\\n\\nfrom .extract import flatten, PROTECTED_FEATURES\\n\\n\\ndef entropy(values):\\n    counts = collections.Counter(values); n = sum(counts.values())\\n    return -sum(count/n * math.log2(count/n) for count in counts.values()) if n else 0.0\\n\\n\\ndef mode(values):\\n    counts = collections.Counter(values)\\n    return sorted(counts, key=lambda label: (-counts[label], label))[0]\\n\\n\\ndef probe(pairs):\\n    if not pairs or any(row[\'split\']!=\'train\' for row in pairs):\\n        raise ValueError(\'Only training pairs may inform mapping probes\')\\n    rows = [{\'paper\': row[\'paper_key\'], \'x\': flatten(row[\'annotation\'][\'source\'][\'structure\']),\\n             \'y\': flatten(row[\'annotation\'][\'target\'][\'structure\'])} for row in pairs]\\n    keys = [key for key in rows[0][\'x\'] if key not in PROTECTED_FEATURES]\\n    if len({row[\'paper\'] for row in rows}) < 2: raise ValueError(\'Need independent paper folds\')\\n    predictions = {name: [] for name in (\'source_identity\', \'fixed_prior\', \'retrieval\', \'smoothed_conditional\')}\\n    per_paper = {}\\n    for paper in sorted({row[\'paper\'] for row in rows}):\\n        fit = [row for row in rows if row[\'paper\']!=paper]\\n        held = [row for row in rows if row[\'paper\']==paper]\\n        local = {name: [] for name in predictions}\\n        for row in held:\\n            nearest = min(enumerate(fit), key=lambda item:(sum(row[\'x\'][key]!=item[1][\'x\'][key] for key in keys), item[0]))[1]\\n            pred = {name:{} for name in predictions}\\n            for key in keys:\\n                prior = collections.Counter(item[\'y\'][key] for item in fit)\\n                conditioned = collections.Counter(item[\'y\'][key] for item in fit if item[\'x\'][key]==row[\'x\'][key])\\n                # Five prior-equivalent observations regularize rare source buckets.\\n                labels = sorted(prior)\\n                conditional = max(labels, key=lambda label: conditioned[label] + 5*prior[label]/len(fit))\\n                pred[\'source_identity\'][key] = row[\'x\'][key]\\n                pred[\'fixed_prior\'][key] = mode(item[\'y\'][key] for item in fit)\\n                pred[\'retrieval\'][key] = nearest[\'y\'][key]\\n                pred[\'smoothed_conditional\'][key] = conditional\\n            for name in pred:\\n                matches = {key: pred[name][key]==row[\'y\'][key] for key in keys}\\n                predictions[name].append(matches); local[name].append(matches)\\n        per_paper[paper] = {name: statistics.mean(sum(record.values())/len(keys) for record in records)\\n                            for name, records in local.items()}\\n    scores = {name: {\'macro_field_accuracy\': statistics.mean(sum(row.values())/len(keys) for row in records),\\n                     \'per_field_accuracy\': {key: statistics.mean(row[key] for row in records) for key in keys}}\\n              for name, records in predictions.items()}\\n    return {\'rows\': len(rows), \'papers\': len(per_paper), \'fit_split\': \'train\',\\n        \'source_signatures\': len({tuple(row[\'x\'][key] for key in keys) for row in rows}),\\n        \'target_signatures\': len({tuple(row[\'y\'][key] for key in keys) for row in rows}),\\n        \'protected_features_excluded_from_prediction\': sorted(PROTECTED_FEATURES),\\n        \'per_field\': {key: {\'source_counts\': dict(collections.Counter(row[\'x\'][key] for row in rows)),\\n            \'target_counts\': dict(collections.Counter(row[\'y\'][key] for row in rows)),\\n            \'source_entropy_bits\': entropy([row[\'x\'][key] for row in rows]),\\n            \'target_entropy_bits\': entropy([row[\'y\'][key] for row in rows]),\\n            \'changed_fraction\': statistics.mean(row[\'x\'][key]!=row[\'y\'][key] for row in rows)} for key in keys},\\n        \'leave_one_training_paper_out\': scores, \'per_paper_macro_accuracy\': per_paper,\\n        \'protocol\': \'Exploratory categorical probes; validation and reserved test excluded; no deployable mapping fitted\',\\n        \'recommendation\': \'Test the oracle first. If supported, start with fixed prior; require paper-grouped gains before adding retrieval or smoothed conditional edits. No neural mapper yet.\',\\n        \'caveat\': \'Bucket prediction accuracy is not generation quality or semantic preservation; field imbalance and correlated controls inflate accuracy.\'}\\n", "src/fama_style/rhetoric/oracle.py": "\\"\\"\\"Content-free oracle templates, balanced prompts and automatic diagnostic scoring.\\"\\"\\"\\nimport copy\\nimport difflib\\nimport json\\nimport statistics\\n\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research.claims import sha, verify, risk_flags, tokens\\nfrom fama_style.research.signal import overlap, distance\\nfrom .extract import extract, flatten\\nfrom .schema import validate, validate_links\\n\\nCONDITIONS = (\'A_plain\', \'B_source_structure\', \'C_oracle_target_structure\')\\nPROTOCOL = \'oracle_rhetoric_v1\'\\nPRESENTATION_KEYS = {\'organization\', \'sentence_count\', \'syntax\', \'condition_position\', \'sentence_length\',\\n                     \'author_voice\', \'semicolon\', \'parenthesis\', \'nominalization\'}\\n\\n\\ndef structural_template(structure):\\n    \\"\\"\\"Copy only validated finite labels; omit hashes, claims and reference spans.\\"\\"\\"\\n    validate(structure)\\n    return {\'paragraph\': {\'organization\': structure[\'paragraph\'][\'organization\'],\\n                          \'sentence_count\': structure[\'paragraph\'][\'sentence_count\'],\\n                          \'move_roles\': [item[\'role\'] for item in structure[\'paragraph\'][\'moves\']]},\\n            \'sentences\': [{key: item[key] for key in (\'syntax\', \'condition_position\', \'voice\', \'length\')}\\n                          for item in structure[\'sentences\']],\\n            \'language\': {key: value for key,value in structure[\'language\'].items() if key not in (\'hedging\',\'citation\')}}\\n\\n\\ndef adherence(actual, desired):\\n    \\"\\"\\"Score finite feature buckets and the requested paragraph/sentence sequences.\\"\\"\\"\\n    a, b = flatten(actual), flatten(desired)\\n    buckets = statistics.mean(a[key]==b[key] for key in PRESENTATION_KEYS)\\n    moves = difflib.SequenceMatcher(None, [r[\'role\'] for r in actual[\'paragraph\'][\'moves\']],\\n                                    [r[\'role\'] for r in desired[\'paragraph\'][\'moves\']], autojunk=False).ratio()\\n    def patterns(structure):\\n        return [tuple(row[key] for key in (\'syntax\',\'condition_position\',\'voice\',\'length\')) for row in structure[\'sentences\']]\\n    syntax = difflib.SequenceMatcher(None, patterns(actual), patterns(desired), autojunk=False).ratio()\\n    return {\'bucket_match\':buckets, \'move_sequence_match\':moves, \'sentence_sequence_match\':syntax,\\n            \'combined\':statistics.mean([buckets,moves,syntax])}\\n\\n\\ndef prompt_for(source, condition, source_annotation, target_structure=None):\\n    if condition not in CONDITIONS: raise ValueError(\'Unknown oracle condition\')\\n    validate_links(source_annotation[\'structure\'], source_annotation[\'semantic_ledger\'])\\n    if source_annotation[\'provenance\'][\'input_sha256\'] != sha(source): raise ValueError(\'Changed source annotation\')\\n    if condition == \'A_plain\': return make_prompt(source)\\n    structure = source_annotation[\'structure\'] if condition == \'B_source_structure\' else target_structure\\n    template = structural_template(structure)\\n    # Same source evidence in B/C. Never bind target-side IDs to source claims by position.\\n    claims = [{\'claim_id\': c[\'claim_id\'], \'source_evidence\': c[\'evidence\'][\'text\']}\\n              for c in source_annotation[\'semantic_ledger\'][\'claims\']]\\n    constraints = (\\n        \'Use the structural template as a presentation guide, not new factual evidence. \'\\n        \'The source-evidence units below must all be preserved; they may contain multiple atomic claims. \'\\n        \'Keep their actors, qualifications and semantic relationships fixed. \'\\n        \'Move-role and syntax labels are fallible observations, not permission to invent a definition, \'\\n        \'contrast, result, citation, technical term or causal connection. \'\\n        \'Ignore a template feature if realizing it would add or change a claim. \'\\n        \'Sentence slots are not claim identities: combine or split presentation only when meaning is preserved. \'\\n        \'Do not output the plan or claim IDs. Return only the rewritten paragraph.\'\\n    )\\n    return make_prompt(source) + \'\\\\n\\\\n\' + constraints + \'\\\\nSource evidence units:\\\\n\' + json.dumps(claims, ensure_ascii=False) + \'\\\\nStructural template:\\\\n\' + json.dumps(template, sort_keys=True)\\n\\n\\ndef jobs_for(validation):\\n    if not validation or any(row[\'split\']!=\'validation\' for row in validation): raise ValueError(\'Oracle requires validation only\')\\n    if len({row[\'id\'] for row in validation})!=len(validation): raise ValueError(\'Duplicate validation identity\')\\n    jobs = []\\n    for index, row in enumerate(validation):\\n        a, b = extract(row[\'plain_english\']), extract(row[\'completion\'])\\n        # Rotate A/B/C execution order to balance load/order effects; all greedy controls identical.\\n        order = CONDITIONS[index%3:] + CONDITIONS[:index%3]\\n        for condition in order:\\n            prompt = prompt_for(row[\'plain_english\'], condition, a, b[\'structure\'])\\n            jobs.append({\'id\': row[\'id\'], \'split\': \'validation\', \'condition\': condition,\\n                         \'prompt\': prompt, \'prompt_sha256\': sha(prompt), \'source_sha256\': sha(row[\'plain_english\']),\\n                         \'oracle\': condition == CONDITIONS[2]})\\n    return jobs\\n\\n\\ndef validate_outputs(jobs, outputs, allow_synthetic=False):\\n    expected = {(job[\'id\'], job[\'condition\']): job for job in jobs}\\n    if len(expected)!=len(jobs) or len(outputs)!=len(jobs): raise ValueError(\'Incomplete oracle outputs\')\\n    seen = set()\\n    for row in outputs:\\n        key = (row[\'id\'], row[\'condition\']); job = expected.get(key)\\n        if key in seen or job is None or row.get(\'prompt_sha256\')!=job[\'prompt_sha256\']:\\n            raise ValueError(\'Duplicate, changed or mismatched oracle result\')\\n        if (not isinstance(row.get(\'output\'), str) or not row[\'output\'].strip() or row.get(\'hit_output_limit\') is not False\\n                or row.get(\'synthetic\') is not allow_synthetic): raise ValueError(\'Invalid or truncated oracle output\')\\n        seen.add(key)\\n\\n\\ndef generate_jobs(jobs, generator, checkpoint=None):\\n    \\"\\"\\"The generator sees a prompt-only job, never paired targets or annotations.\\"\\"\\"\\n    results = []\\n    for job in jobs:\\n        if sha(job[\'prompt\']) != job[\'prompt_sha256\']: raise ValueError(\'Changed prompt\')\\n        response = generator(copy.deepcopy(job))\\n        result = {**response, \'id\': job[\'id\'], \'condition\': job[\'condition\'],\\n                  \'prompt_sha256\': job[\'prompt_sha256\']}\\n        results.append(result)\\n        if checkpoint: checkpoint(results)\\n        if (not isinstance(result.get(\'output\'),str) or not result[\'output\'].strip()\\n                or result.get(\'hit_output_limit\') is not False):\\n            raise ValueError(\'Empty or truncated output; partial record saved, experiment incomplete\')\\n    return results\\n\\n\\ndef score(validation, jobs, outputs, profile, synthetic=False, semantic=None, calibration=None):\\n    validate_outputs(jobs, outputs, synthetic)\\n    if any(row[\'split\']!=\'validation\' for row in validation): raise ValueError(\'Invalid scoring split\')\\n    by_id = {row[\'id\']: row for row in validation}\\n    records = []\\n    for output in outputs:\\n        row = by_id[output[\'id\']]; source, target = row[\'plain_english\'], row[\'completion\']\\n        generated = extract(output[\'output\'])\\n        actual = flatten(generated[\'structure\']); target_structure = extract(target)[\'structure\']; oracle = flatten(target_structure)\\n        source_structure = extract(source)[\'structure\']\\n        requested = source_structure if output[\'condition\']==CONDITIONS[1] else target_structure if output[\'condition\']==CONDITIONS[2] else None\\n        oracle_adherence = adherence(generated[\'structure\'],target_structure)\\n        verification = verify(source, output[\'output\'])\\n        flags = risk_flags(source, output[\'output\'])\\n        independent = (semantic or {}).get((row[\'id\'], output[\'condition\']))\\n        src_tokens, out_tokens = set(tokens(source)), set(tokens(output[\'output\']))\\n        records.append({\'id\': row[\'id\'], \'condition\': output[\'condition\'],\\n            \'source_similarity\': overlap(source, output[\'output\']),\\n            \'exact_copy\': \' \'.join(source.split())==\' \'.join(output[\'output\'].split()),\\n            \'near_copy\': overlap(source, output[\'output\'])>=.90,\\n            \'semantic_surface_flags\': flags, \'bounded_verification\': verification[\'status\'],\\n            \'semantic_consistency\': independent[\'status\'] if independent else \'unresolved\',\\n            \'independent_semantic_assessment_executed\': independent is not None,\\n            \'independent_nli\': independent,\\n            \'source_token_set_coverage\': len(src_tokens & out_tokens)/max(1,len(src_tokens)),\\n            \'output_token_set_support\': len(src_tokens & out_tokens)/max(1,len(out_tokens)),\\n            \'oracle_feature_matches\': {key: actual[key]==oracle[key] for key in oracle},\\n            \'oracle_structure_match\': oracle_adherence[\'combined\'], \'oracle_adherence_components\':oracle_adherence,\\n            \'requested_plan_match\': None if requested is None else adherence(generated[\'structure\'],requested)[\'combined\'],\\n            \'style_gain_proxy\': distance(source, profile)-distance(output[\'output\'], profile),\\n            \'output_input_token_ratio\': len(tokens(output[\'output\']))/max(1,len(tokens(source))),\\n            \'input_tokens\': output.get(\'input_tokens\'), \'output_tokens\': output.get(\'output_tokens\'),\\n            \'generation_seconds\': output.get(\'generation_seconds\')})\\n    aggregate = {}\\n    for condition in CONDITIONS:\\n        group = [row for row in records if row[\'condition\']==condition]\\n        aggregate[condition] = {\'rows\': len(group),\\n            **{key: statistics.mean(row[key] for row in group) for key in\\n               (\'source_similarity\', \'exact_copy\', \'near_copy\', \'oracle_structure_match\', \'style_gain_proxy\', \'output_input_token_ratio\')},\\n            \'rows_with_semantic_surface_flags\': sum(bool(row[\'semantic_surface_flags\']) for row in group),\\n            \'semantic_consistency_rate\': None,\\n            \'requested_plan_match\': None if condition==CONDITIONS[0] else statistics.mean(row[\'requested_plan_match\'] for row in group),\\n            \'nli_proxy_consistency_rate\': (statistics.mean(row[\'semantic_consistency\']==\'proxy_consistent\' for row in group)\\n                                            if calibration and calibration.get(\'passed\') else None),\\n            \'generation_seconds\': sum(row[\'generation_seconds\'] or 0 for row in group),\\n            \'input_tokens\': sum(row[\'input_tokens\'] or 0 for row in group),\\n            \'output_tokens\': sum(row[\'output_tokens\'] or 0 for row in group)}\\n    lookup = {(row[\'id\'], row[\'condition\']): row for row in records}\\n    paired = {}\\n    for comparison in (CONDITIONS[0], CONDITIONS[1]):\\n        changes = [lookup[(row[\'id\'],CONDITIONS[2])][\'oracle_structure_match\']-lookup[(row[\'id\'],comparison)][\'oracle_structure_match\'] for row in validation]\\n        paired[\'C_minus_\'+comparison] = {\'mean_oracle_structure_gain\': statistics.mean(changes),\\n            \'improved\': sum(value>0 for value in changes), \'worsened\': sum(value<0 for value in changes), \'tied\': sum(value==0 for value in changes)}\\n    a, b, c = (aggregate[key] for key in CONDITIONS)\\n    semantic_ready = not synthetic and bool(calibration and calibration.get(\'passed\')) and all(row[\'independent_semantic_assessment_executed\'] for row in records)\\n    support = semantic_ready and c[\'nli_proxy_consistency_rate\']>=.9 and c[\'nli_proxy_consistency_rate\']>=max(a[\'nli_proxy_consistency_rate\'],b[\'nli_proxy_consistency_rate\']) and c[\'near_copy\']<=min(a[\'near_copy\'],b[\'near_copy\']) and c[\'oracle_structure_match\']-b[\'oracle_structure_match\']>=.05 and c[\'style_gain_proxy\']>max(a[\'style_gain_proxy\'],b[\'style_gain_proxy\'])\\n    return records, {\'protocol\': PROTOCOL, \'execution_kind\': \'synthetic_wiring_check\' if synthetic else \'model_inference\',\\n        \'validation_rows\': len(validation), \'conditions\': aggregate, \'paired_differences\': paired,\\n        \'oracle_is_deployment_result\': False, \'test_rows_used\': 0, \'training_called\': False,\\n        \'oracle_supported_by_automated_proxies\': bool(support), \'semantic_claims_verified\': False,\\n        \'nli_control_results\': calibration,\\n        \'mapping_gate\': \'provisionally_supported_by_proxies\' if support else \'not_supported_by_current_proxies\' if semantic_ready else \'unresolved_semantic_assessment_or_synthetic_run\',\\n        \'limitations\': \'Adherence and style share surface proxies. Token overlap is not entailment. C uses reference-derived side information; B/C share source evidence while A is the ordinary prompt.\'}\\n", "src/fama_style/rhetoric/schema.py": "\\"\\"\\"Closed, content-free structural vocabulary. Claims live in a separate ledger.\\"\\"\\"\\nimport re\\n\\nVERSION = \'rhetoric_v1\'\\nEXTRACTOR = \'surface_rules_v1\'\\nROLES = [\'assertion\', \'definition\', \'reporting\', \'contrast\', \'condition\', \'conclusion\', \'unknown\']\\nORGANIZATIONS = [\'single\', \'sequence\', \'conditional\', \'contrastive\', \'reporting\', \'mixed\']\\nBINS = [\'none\', \'low\', \'high\']\\nCOUNTS = [\'one\', \'two\', \'three_plus\']\\nLENGTHS = [\'short\', \'medium\', \'long\']\\nSYNTAX = [\'simple\', \'coordinated\', \'subordinated\', \'mixed\']\\nPOSITIONS = [\'none\', \'front\', \'end\', \'embedded\', \'mixed\']\\nVOICES = [\'absent\', \'present\']\\n\\n\\ndef obj(properties):\\n    return {\'type\': \'object\', \'properties\': properties, \'required\': list(properties), \'additionalProperties\': False}\\n\\n\\ndef enum(values):\\n    return {\'type\': \'string\', \'enum\': values}\\n\\n\\ndef array(item):\\n    return {\'type\': \'array\', \'items\': item, \'minItems\': 1}\\n\\n\\nUNIT = {\'type\': \'string\', \'pattern\': r\'^s[1-9][0-9]*$\'}\\nCLAIMS = array({\'type\': \'string\', \'pattern\': r\'^c[1-9][0-9]*$\'})\\nSTRUCTURE_SCHEMA = obj({\\n    \'version\': enum([VERSION]),\\n    \'paragraph\': obj({\'organization\': enum(ORGANIZATIONS), \'sentence_count\': enum(COUNTS),\\n        \'moves\': array(obj({\'unit\': UNIT, \'role\': enum(ROLES), \'claims\': CLAIMS}))}),\\n    \'sentences\': array(obj({\'unit\': UNIT, \'claims\': CLAIMS, \'syntax\': enum(SYNTAX),\\n        \'condition_position\': enum(POSITIONS), \'voice\': enum(VOICES), \'length\': enum(LENGTHS)})),\\n    \'language\': obj({\'author_voice\': enum(VOICES), \'semicolon\': enum(BINS),\\n        \'parenthesis\': enum(BINS), \'hedging\': enum(BINS), \'citation\': enum(BINS),\\n        \'nominalization\': enum(BINS)}),\\n})\\nJSON_SCHEMA = {\'$schema\': \'https://json-schema.org/draft/2020-12/schema\',\\n               \'$id\': \'urn:fama-style:rhetoric:v1\', **STRUCTURE_SCHEMA}\\n\\n\\ndef validate(value, schema=STRUCTURE_SCHEMA):\\n    \\"\\"\\"Validate this closed subset without requiring a local JSON Schema dependency.\\"\\"\\"\\n    kind = schema[\'type\']\\n    if kind == \'object\':\\n        if not isinstance(value, dict) or set(value) != set(schema[\'properties\']):\\n            raise ValueError(\'Unexpected or missing structural fields\')\\n        for key, child in schema[\'properties\'].items():\\n            validate(value[key], child)\\n    elif kind == \'array\':\\n        if not isinstance(value, list) or len(value) < schema[\'minItems\']:\\n            raise ValueError(\'Invalid structural array\')\\n        for item in value:\\n            validate(item, schema[\'items\'])\\n    elif kind == \'string\':\\n        if not isinstance(value, str):\\n            raise ValueError(\'Invalid structural label\')\\n        if \'enum\' in schema and value not in schema[\'enum\']:\\n            raise ValueError(\'Unknown structural label\')\\n        if \'pattern\' in schema and not re.fullmatch(schema[\'pattern\'], value):\\n            raise ValueError(\'Invalid claim/unit identity\')\\n    return value\\n\\n\\ndef validate_links(structure, ledger):\\n    validate(structure)\\n    ids = {claim[\'claim_id\'] for claim in ledger[\'claims\']}\\n    units = [row[\'unit\'] for row in structure[\'sentences\']]\\n    if len(set(units)) != len(units):\\n        raise ValueError(\'Duplicate structural unit\')\\n    if [row[\'unit\'] for row in structure[\'paragraph\'][\'moves\']] != units:\\n        raise ValueError(\'Paragraph moves and sentence units disagree\')\\n    if set(ref for row in structure[\'sentences\'] for ref in row[\'claims\']) != ids:\\n        raise ValueError(\'Uncovered or invented claim IDs\')\\n    for move, sentence in zip(structure[\'paragraph\'][\'moves\'], structure[\'sentences\']):\\n        if move[\'claims\'] != sentence[\'claims\']:\\n            raise ValueError(\'Move claim binding disagrees with sentence binding\')\\n    return True\\n", "src/fama_style/rhetoric/semantics.py": "\\"\\"\\"Independent NLI diagnostic interface, with automatic controls and abstention.\\"\\"\\"\\nimport math\\n\\nfrom fama_style.research.claims import sentences\\n\\nNLI_MODEL = \'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli\'\\n\\n\\ndef checked_prediction(predict, premise, hypothesis):\\n    result = predict(premise, hypothesis)\\n    if result is None: return None  # Context overflow must abstain, never truncate.\\n    labels = {\'entailment\', \'neutral\', \'contradiction\'}\\n    if set(result)!=labels or any(not isinstance(p, (float,int)) or not math.isfinite(p) or not 0<=p<=1 for p in result.values()):\\n        raise ValueError(\'Invalid NLI probability mapping\')\\n    if abs(sum(result.values())-1)>.01: raise ValueError(\'Invalid NLI probability sum\')\\n    return result\\n\\n\\ndef assess(source, output, predict):\\n    pairs = [(\'whole_forward\', source, output), (\'whole_reverse\', output, source)]\\n    pairs += [(f\'source_unit_{i}\', output, text) for i,text in enumerate(sentences(source),1)]\\n    pairs += [(f\'output_unit_{i}\', source, text) for i,text in enumerate(sentences(output),1)]\\n    records = [{\'direction\': direction, \'probabilities\': checked_prediction(predict, premise, hypothesis)}\\n               for direction, premise, hypothesis in pairs]\\n    known = [row[\'probabilities\'] for row in records if row[\'probabilities\'] is not None]\\n    complete = len(known)==len(records)\\n    minimum = min((p[\'entailment\'] for p in known), default=None)\\n    maximum = max((p[\'contradiction\'] for p in known), default=None)\\n    return {\'status\': \'unknown_over_context\' if not complete else \'proxy_consistent\' if minimum>=.8 and maximum<=.1 else \'proxy_flagged\',\\n            \'minimum_entailment\': minimum, \'maximum_contradiction\': maximum, \'checks\': records,\\n            \'semantic_proof\': False, \'sentence_units_are_atomic_claims\': False}\\n\\n\\ndef calibration_controls(predict):\\n    # Authored synthetic controls; no validation or reserved-test references.\\n    equivalent = [(\'Firms report revenue.\', \'Firms report revenue.\'),\\n                  (\'If demand falls, firms reduce output.\', \'Firms reduce output if demand falls.\'),\\n                  (\'Firms report revenue. Investors read reports.\', \'Firms report revenue; investors read reports.\')]\\n    changed = [(\'Firms pay investors.\', \'Investors pay firms.\'),\\n               (\'Costs cause losses.\', \'Losses cause costs.\'),\\n               (\'If demand falls, firms reduce output.\', \'Firms reduce output.\'),\\n               (\'Some firms invest.\', \'All firms invest.\'),\\n               (\'Firms may reduce output.\', \'Firms must reduce output.\'),\\n               (\'Firms do not invest.\', \'Firms invest.\'),\\n               (\'Revenue rises by 5 percent.\', \'Revenue rises by 10 percent.\')]\\n    positives = [assess(a,b,predict)[\'status\']==\'proxy_consistent\' for a,b in equivalent]\\n    negatives = [assess(a,b,predict)[\'status\']==\'proxy_flagged\' for a,b in changed]\\n    passed = all(positives) and all(negatives)\\n    return {\'equivalent_controls\': len(positives), \'accepted_equivalents\': sum(positives),\\n            \'semantic_mutations\': len(negatives), \'rejected_mutations\': sum(negatives),\\n            \'passed\': passed, \'thresholds\': {\'entailment_minimum\': .8, \'contradiction_maximum\': .1},\\n            \'meaning\': \'Small synthetic sanity gate only; not domain calibration or verified economic entailment\',\\n            \'human_review_used\': False}\\n", "src/fama_style/rhetoric/sentence_slots.py": "\\"\\"\\"Count-only prose versus sentence-slot JSON: inference only, first attempts.\\"\\"\\"\\nimport argparse\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport statistics\\nimport time\\n\\nfrom . import concise_plan as base\\nfrom .concise_plan import (MODEL_ID,MODEL_REVISION,NLI_MODEL,NLI_REVISION,CONTEXT_LIMIT,\\n    file_sha,read_lines,load_pack,synthetic_pack)\\nfrom .experiment import secret_token,with_heartbeat,write_json,write_lines,fresh\\nfrom .semantics import assess,calibration_controls\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research.claims import sentences,sha,risk_flags\\nfrom fama_style.research.signal import distance,overlap\\n\\nPROTOCOL=\'sentence_slot_realization_v1\'\\nCONDITIONS=(\'A_ordinary\',\'B_count_only\',\'C_sentence_slots\')\\nMAX_NEW_TOKENS=768\\n\\n\\ndef jobs_for(pack):\\n    jobs=[]\\n    for index,row in enumerate(pack[\'rows\']):\\n        # Rotate order rather than always giving a condition the same position.\\n        order=CONDITIONS[index%3:]+CONDITIONS[:index%3]\\n        for condition in order:\\n            prompt=make_prompt(row[\'source\'])\\n            count=row[\'plan\'][\'sentence_count\']\\n            if condition!=\'A_ordinary\':\\n                prompt+=f\'\\\\n\\\\nPresentation control: write exactly {count} complete sentences. Preserve source claim order, all claims and qualifications. Do not add claims to fill slots. Faithfulness takes priority.\'\\n            if condition==\'C_sentence_slots\':\\n                prompt+=\'\\\\nOutput-format override: return only a JSON object with the single key \\"sentences\\", whose value is an array of exactly \'+str(count)+\' strings. Each string must contain exactly one complete sentence ending in a period, question mark or exclamation mark. No Markdown fences, commentary or other keys. The strings in order will be joined into one paragraph.\'\\n            jobs.append({\'id\':row[\'id\'],\'condition\':condition,\'prompt\':prompt,\'prompt_sha256\':sha(prompt),\'source_sha256\':sha(row[\'source\'])})\\n    return jobs\\n\\n\\ndef realize(raw,condition,count):\\n    \\"\\"\\"Strict format validation. No repair, regex extraction or silent salvage.\\"\\"\\"\\n    errors=[];items=None;text=raw\\n    if condition==\'C_sentence_slots\':\\n        try:\\n            def unique(pairs):\\n                result={}\\n                for key,value in pairs:\\n                    if key in result:raise ValueError(\'duplicate_key\')\\n                    result[key]=value\\n                return result\\n            obj=json.loads(raw,object_pairs_hook=unique)\\n            if not isinstance(obj,dict) or set(obj)!={\'sentences\'}:raise ValueError(\'wrong_keys\')\\n            items=obj[\'sentences\']\\n            if not isinstance(items,list) or not items or any(not isinstance(s,str) or not s.strip() for s in items):\\n                raise ValueError(\'invalid_sentence_array\')\\n            text=\' \'.join(s.strip() for s in items)\\n            if len(items)!=count:errors.append(\'wrong_slot_count\')\\n            if any(len(sentences(s.strip()))!=1 or not re.search(r\'[.!?][\\"”’\\\\\']?$\',s.strip()) or \'\\\\n\' in s for s in items):\\n                errors.append(\'slot_not_one_complete_surface_sentence\')\\n        except (ValueError,TypeError):\\n            errors.append(\'invalid_json_contract\')\\n    else:\\n        if raw.lstrip().startswith((\'```\',\'{\',\'[\')):errors.append(\'non_paragraph_format\')\\n    return {\'text\':text,\'format_valid\':not errors,\'format_errors\':errors,\'slot_count\':len(items) if isinstance(items,list) else None,\\n            \'sentence_count\':len(sentences(text)),\'count_match\':len(sentences(text))==count,\\n            \'semantic_input\':\'joined_slots\' if items is not None and not errors else (\'raw_invalid_format\' if errors else \'paragraph\')}\\n\\n\\ndef audit_tokens(job,ids):\\n    if sha(job[\'prompt\'])!=job[\'prompt_sha256\']:raise ValueError(\'Changed prompt\')\\n    if not ids or len(ids)+MAX_NEW_TOKENS>CONTEXT_LIMIT:raise ValueError(\'Context overflow; no truncation allowed\')\\n    return {\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'input_tokens\':len(ids),\'prompt_sha256\':job[\'prompt_sha256\'],\'input_ids_sha256\':sha(json.dumps(ids))}\\n\\n\\ndef evaluate(pack,jobs,outputs,semantic=None,pair_checks=None,calibration=None,synthetic=False):\\n    expected={(j[\'id\'],j[\'condition\']):j for j in jobs};seen=set();records=[]\\n    if jobs!=jobs_for(pack) or len(outputs)!=len(expected):raise ValueError(\'Changed jobs or incomplete outputs\')\\n    rows={r[\'id\']:r for r in pack[\'rows\']}\\n    for out in outputs:\\n        key=(out[\'id\'],out[\'condition\']);job=expected.get(key)\\n        if key in seen or job is None or out[\'prompt_sha256\']!=job[\'prompt_sha256\']:raise ValueError(\'Duplicate/swapped output\')\\n        if not out.get(\'output\',\'\').strip() or out.get(\'hit_output_limit\') is not False or out.get(\'synthetic\') is not synthetic:\\n            raise ValueError(\'Empty, cutoff or mislabeled output\')\\n        seen.add(key);row=rows[out[\'id\']];n=row[\'plan\'][\'sentence_count\'];r=realize(out[\'output\'],out[\'condition\'],n);text=r[\'text\']\\n        nli=(semantic or {}).get(key);valid=r[\'format_valid\'];passed=bool(nli and nli[\'status\']==\'proxy_consistent\' and calibration and calibration[\'passed\'])\\n        records.append({\'id\':out[\'id\'],\'condition\':out[\'condition\'],**r,\'desired_sentence_count\':n,\\n            \'sentence_count_absolute_error\':abs(r[\'sentence_count\']-n),\\n            \'nli\':nli,\'nli_status\':nli[\'status\'] if nli else \'not_run\',\\n            \'format_count_semantic_proxy_pass\':valid and r[\'count_match\'] and passed,\\n            \'near_copy\':overlap(row[\'source\'],text)>=.9 if valid else None,\\n            \'style_gain_proxy\':distance(row[\'source\'],pack[\'style_profile\'])-distance(text,pack[\'style_profile\']) if valid else None,\\n            \'semantic_surface_flags\':risk_flags(row[\'source\'],text),\\n            \'generation_seconds\':out.get(\'generation_seconds\',0),\'input_tokens\':out.get(\'input_tokens\',0),\'output_tokens\':out.get(\'output_tokens\',0)})\\n    aggregate={}\\n    for c in CONDITIONS:\\n        group=[r for r in records if r[\'condition\']==c];valid=[r for r in group if r[\'format_valid\']]\\n        aggregate[c]={\'rows\':len(group),\'format_valid\':len(valid),\'count_matches\':sum(r[\'format_valid\'] and r[\'count_match\'] for r in group),\\n            \'format_invalid\':len(group)-len(valid),\\n            \'nli_unknown_valid_only\':sum(r[\'nli_status\']==\'unknown_over_context\' for r in valid),\\n            \'mean_count_error_valid_only\':statistics.mean(r[\'sentence_count_absolute_error\'] for r in valid) if valid else None,\\n            \'nli_proxy_passes_valid_only\':sum(r[\'nli_status\']==\'proxy_consistent\' for r in valid) if calibration and calibration[\'passed\'] else None,\\n            \'format_count_semantic_proxy_passes\':sum(r[\'format_count_semantic_proxy_pass\'] for r in group),\\n            \'near_copies_valid_only\':sum(r[\'near_copy\'] for r in valid),\\n            \'mean_style_gain_valid_only\':statistics.mean(r[\'style_gain_proxy\'] for r in valid) if valid else None,\\n            **{k:sum(r[k] or 0 for r in group) for k in (\'generation_seconds\',\'input_tokens\',\'output_tokens\')}}\\n    return records,{\'protocol\':PROTOCOL,\'execution_kind\':\'synthetic_wiring_check\' if synthetic else \'model_inference\',\'conditions\':aggregate,\\n        \'pair_source_reference_nli\':pair_checks,\'nli_controls\':calibration,\'training_called\':False,\'held_out_rows_used\':0,\\n        \'oracle_is_deployable\':False,\'semantic_accuracy\':None,\'full_mapping_training_ready\':False,\'repair_attempts\':0,\\n        \'limitations\':\'Four previously inspected training pairs; count-only and output-format interventions; no generalization claim. Sentence boundaries and style are surface proxies. NLI can miss drift. Invalid formats remain failures in all-row denominators; valid-only metrics must report their denominators.\'}\\n\\n\\ndef run_training(pack,output,uploaded_pack_sha):\\n    \\"\\"\\"Inference only; lazy ML imports follow the repository entry-point convention.\\"\\"\\"\\n    fresh(output);jobs=jobs_for(pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'uploaded_pack_identity.json\',{\'sha256\':uploaded_pack_sha,\'format\':\'uploaded file bytes\'})\\n    status={\'complete\':False,\'protocol\':PROTOCOL,\'training_called\':False,\'held_out_rows_used\':0,\'completed_jobs\':0,\'total_jobs\':len(jobs)}\\n    started=time.monotonic();outputs=[]\\n    def phase(name,**extra):\\n        status.update(phase=name,**extra);write_json(output/\'run_status.json\',status);print(\'Phase: \'+name,flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        import os\\n        import gc\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\';os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor,AutoTokenizer,AutoModelForSequenceClassification,\\n            Gemma3ForConditionalGeneration,BitsAndBytesConfig,LogitsProcessor,LogitsProcessorList)\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Expose one Kaggle GPU before importing torch\')\\n        torch.cuda.init();torch.cuda.set_device(0);torch.manual_seed(42);torch.cuda.reset_peak_memory_stats(0)\\n        if \'T4\' not in torch.cuda.get_device_name(0):raise RuntimeError(\'Select Kaggle T4 x2 for the pinned diagnostic\')\\n        phase(\'checking_secret\');token=secret_token()\\n        phase(\'loading_pinned_processor\')\\n        processor=with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token),\'Processor load\')\\n        def tokenize(job):\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[]\\n        for job in jobs:audits.append(audit_tokens(job,tokenize(job)[\'input_ids\'][0].tolist()))\\n        write_json(output/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)),\'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):\\n            raise RuntimeError(\'Unexpected model placement\')\\n        model.eval();model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):\\n                    raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        by_key={(r[\'id\'],r[\'condition\']):r for r in audits}\\n        for job in jobs:\\n            phase(\'generation\',completed_jobs=len(outputs))\\n            batch=tokenize(job);ids=batch[\'input_ids\'][0].tolist();audit=audit_tokens(job,ids)\\n            if audit!=by_key[(job[\'id\'],job[\'condition\'])]:raise ValueError(\'Tokenized prompt changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()};before=time.monotonic()\\n            with torch.inference_mode():\\n                generated=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,\\n                    use_cache=True,pad_token_id=processor.tokenizer.pad_token_id,remove_invalid_values=False,\\n                    logits_processor=LogitsProcessorList([FiniteScores()])),\'Generation\')\\n            new=generated[0,len(ids):];eos=model.generation_config.eos_token_id;eos=eos if isinstance(eos,list) else [eos]\\n            text=processor.tokenizer.decode(new,skip_special_tokens=True).strip()\\n            result={\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\'output\':text,\\n                \'input_tokens\':len(ids),\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            outputs.append(result);write_lines(output/\'generations.private.jsonl\',outputs)\\n            if not text or result[\'hit_output_limit\']:raise RuntimeError(\'Empty/cutoff output; partial results remain incomplete\')\\n            phase(\'generation\',completed_jobs=len(outputs))\\n        gemma_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del model,processor,batch,generated,new,tokenize\\n        gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'loading_independent_nli_on_gpu\');nli_started=time.monotonic()\\n        tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n        nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,\\n            token=False).to(\'cuda:0\'),\'NLI model load\')\\n        nli.eval();nli.requires_grad_(False)\\n        labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n        if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n        limit=min(512,int(getattr(nli.config,\'max_position_embeddings\',512)),int(tokenizer.model_max_length))\\n        cache={};cache_hits=0\\n        def predict(premise,hypothesis):\\n            nonlocal cache_hits\\n            key=(sha(premise),sha(hypothesis))\\n            if key in cache:cache_hits+=1;return cache[key]\\n            batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n            if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n            with torch.inference_mode():\\n                probs=torch.softmax(nli(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n            cache[key]={labels[i]:float(p) for i,p in enumerate(probs)}\\n            return cache[key]\\n        calibration=with_heartbeat(lambda:calibration_controls(predict),\'NLI controls\');write_json(output/\'nli_controls.json\',calibration)\\n        semantic={};pair_checks={}\\n        for index,row in enumerate(pack[\'rows\'],1):\\n            phase(\'pair_fidelity_assessment\',completed_pairs=index-1)\\n            pair_checks[row[\'id\']]=assess(row[\'source\'],row[\'reference\'],predict)\\n        source_by_id={row[\'id\']:row[\'source\'] for row in pack[\'rows\']}\\n        for index,result in enumerate(outputs,1):\\n            phase(\'output_semantic_assessment\',completed_nli_outputs=index-1)\\n            semantic[(result[\'id\'],result[\'condition\'])]=with_heartbeat(lambda:assess(source_by_id[result[\'id\']],realize(result[\'output\'],result[\'condition\'],next(r[\'plan\'][\'sentence_count\'] for r in pack[\'rows\'] if r[\'id\']==result[\'id\']))[\'text\'],predict),\'NLI assessment\')\\n        records,summary=evaluate(pack,jobs,outputs,semantic,pair_checks,calibration)\\n        write_lines(output/\'metrics.private.jsonl\',records)\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=NLI_REVISION,\\n            nli_device=\'cuda:0 after Gemma unload\',nli_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,\\n            nli_cache_entries=len(cache),nli_cache_hits=cache_hits,gemma_memory=gemma_memory,\\n            nli_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)},\\n            generation={\'do_sample\':False,\'seed\':42,\'max_new_tokens\':MAX_NEW_TOKENS,\'context_limit\':CONTEXT_LIMIT,\'truncate\':False},\\n            versions={name:importlib.metadata.version(name) for name in (\'torch\',\'transformers\',\'accelerate\',\'bitsandbytes\',\'huggingface-hub\')})\\n        write_json(output/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True,completed_jobs=len(outputs),completed_nli_outputs=len(outputs),completed_pairs=len(pack[\'rows\']))\\n        write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'training_called\':False,\'input_pack_sha256\':uploaded_pack_sha,\\n            \'files\':{p.name:file_sha(p) for p in output.iterdir() if p.is_file()}})\\n    except Exception as error:\\n        failed_at=status.get(\'phase\');locations=[];tb=error.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:locations.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,failed_at=failed_at,error_code=\'concise_plan_inference_failed\',local_line_numbers=locations)\\n        raise RuntimeError(\'Concise-plan inference failed; inspect run_status.json and saved partial files. No training was called.\') from None\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    actions=parser.add_mutually_exclusive_group()\\n    actions.add_argument(\'--dry-run\',action=\'store_true\');actions.add_argument(\'--execute-inference\',action=\'store_true\')\\n    parser.add_argument(\'--pack\',type=Path);parser.add_argument(\'--expected-pack-sha\');parser.add_argument(\'--output-dir\',type=Path,required=True)\\n    args=parser.parse_args(argv)\\n    if args.dry_run:\\n        fresh(args.output_dir);pack=synthetic_pack();jobs=jobs_for(pack);rows={r[\'id\']:r for r in pack[\'rows\']}\\n        outputs=[]\\n        for j in jobs:\\n            ref=rows[j[\'id\']][\'reference\'];text=json.dumps({\'sentences\':sentences(ref)}) if j[\'condition\']==\'C_sentence_slots\' else ref\\n            outputs.append({\'id\':j[\'id\'],\'condition\':j[\'condition\'],\'prompt_sha256\':j[\'prompt_sha256\'],\'output\':text,\'hit_output_limit\':False,\'synthetic\':True})\\n        records,summary=evaluate(pack,jobs,outputs,synthetic=True)\\n        write_json(args.output_dir/\'synthetic_evaluation.json\',summary);write_lines(args.output_dir/\'synthetic_jobs.jsonl\',jobs)\\n        print(\'Synthetic wiring check complete: 12 jobs; no model calls.\');return\\n    if args.pack is None or args.expected_pack_sha is None:parser.error(\'--pack and --expected-pack-sha required\')\\n    pack=load_pack(args.pack,args.expected_pack_sha)\\n    if args.execute_inference:run_training(pack,args.output_dir,args.expected_pack_sha)\\n    else:\\n        fresh(args.output_dir);write_json(args.output_dir/\'preflight.json\',{\'protocol\':PROTOCOL,\'pack_sha256\':args.expected_pack_sha,\'pairs\':4,\'jobs\':12,\'training_called\':False})\\n        print(\'Preflight passed; no Secret access or model loading.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/rhetoric/soft_guidance.py": "\\"\\"\\"CPU preparation of a gated 21-example soft-plan comparison; no GPU runner.\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research.claims import sentences,sha\\nfrom fama_style.research.signal import fit_profile\\nfrom .claim_constrained import constraint_ledger\\nfrom .concise_plan import TRAIN_SHA,file_sha,read_lines\\nfrom .experiment import fresh,write_json,write_lines\\nfrom .extract import extract\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'soft_guidance_validation_v1\'\\nVALIDATION_SHA=\'f1c75c29fa93f8458e7a3b2f8996bbd13ff023f92e4a74f1d086d22af5e546b8\'\\nCONDITIONS=(\'A_claims_only\',\'B_claims_soft_plan\')\\n\\n\\ndef soft_plan(source,profile):\\n    if profile.get(\'fit_split\')!=\'train\':raise ValueError(\'Only training-fitted priors allowed\')\\n    f=profile[\'features\'][\'words_per_sentence\']\\n    merge=(f[\'mean_paired_delta\']>.5 and f[\'positive_pair_fraction\']>=.75 and len(sentences(source))>1)\\n    structure=extract(source)[\'structure\']\\n    return {\'paragraph_structure\':{\'operation\':\'retain_source_argument_order\',\\n                \'observed_source_organization\':structure[\'paragraph\'][\'organization\'],\\n                \'source_unit_order\':[\'c\'+str(i) for i in range(1,len(sentences(source))+1)]},\\n            \'sentence_structure\':{\'operation\':\'optional_adjacent_grouping\' if merge else \'retain_or_rephrase\',\\n                \'hard_sentence_count\':None,\'force_grouping\':False,\\n                \'evidence\':\'training_reference_sentence_length_direction\' if merge else \'no_eligible_count_free_style_operation\'},\\n            \'language_style\':{\'operation\':\'no_learned_lexical_prescription\',\\n                \'reason\':\'Frozen surface profile does not establish safe distinctive lexical transformations\'},\\n            \'introduces_semantic_relation\':False}\\n\\n\\ndef jobs_for(pack):\\n    rows=pack[\'rows\']\\n    if len(rows)!=21 or len({r[\'id\'] for r in rows})!=21 or any(r[\'split\']!=\'validation\' for r in rows):raise ValueError(\'Expected 21 unique development-validation rows\')\\n    jobs=[]\\n    for index,row in enumerate(rows):\\n        base=make_prompt(row[\'source\'])+\'\\\\n\\\\nSource-only preservation ledger:\\\\n\'+json.dumps(constraint_ledger(row[\'source\']),ensure_ascii=False,separators=(\',\',\':\'))\\n        base+=\'\\\\nPreserve all source claims, actors, quantities and their assignments, negation, bounds, uncertainty, conditions, qualifications, attribution and causal direction. The ledger is a fallible aid; the complete source is authoritative. Do not invent explanations or conclusions. Choose the sentence count freely. Return only the paragraph.\'\\n        for condition in CONDITIONS if index%2==0 else CONDITIONS[::-1]:\\n            prompt=base\\n            if condition==CONDITIONS[1]:\\n                prompt+=\'\\\\n\\\\nOptional source-bound presentation plan:\\\\n\'+json.dumps(soft_plan(row[\'source\'],pack[\'style_profile\']),separators=(\',\',\':\'))\\n                prompt+=\'\\\\nYou may group adjacent statements into a more integrated sentence only where the source already supports their relationship. Do not add a causal link or condition. Keep qualifications attached to their claims. Do not merge, pad or formalize wording merely to satisfy a target pattern. The plan is optional; faithfulness has priority.\'\\n            jobs.append({\'id\':row[\'id\'],\'split\':\'validation\',\'condition\':condition,\'prompt\':prompt,\\n                \'source_sha256\':sha(row[\'source\']),\'prompt_sha256\':sha(prompt),\'oracle\':False})\\n    return jobs\\n\\n\\ndef prepare(training,validation,benchmark_summary,output):\\n    if file_sha(training)!=TRAIN_SHA or file_sha(validation)!=VALIDATION_SHA:raise ValueError(\'Frozen train/validation bytes changed\')\\n    train,val=read_lines(training),read_lines(validation)\\n    if len(train)!=247 or any(r[\'split\']!=\'train\' for r in train):raise ValueError(\'Wrong training split\')\\n    if len(val)!=21 or any(r[\'split\']!=\'validation\' for r in val):raise ValueError(\'Wrong validation split\')\\n    if {r[\'id\'] for r in train}&{r[\'id\'] for r in val} or {r[\'paper_key\'] for r in train}&{r[\'paper_key\'] for r in val}:raise ValueError(\'Train/validation overlap\')\\n    benchmark=json.loads(benchmark_summary.read_text());manifest=json.loads((benchmark_summary.parent/\'manifest.json\').read_text())\\n    if benchmark.get(\'protocol\')!=\'claim_checker_benchmark_v1\' or manifest[\'files\'][\'summary.json\']!=file_sha(benchmark_summary):raise ValueError(\'Unverified benchmark result\')\\n    ready=benchmark[\'readiness_gate\'][\'passed\']\\n    pack={\'protocol\':PROTOCOL,\'parent_training_sha256\':TRAIN_SHA,\'validation_sha256\':VALIDATION_SHA,\\n        \'style_profile\':fit_profile(train),\\n        \'rows\':[{\'id\':r[\'id\'],\'paper_key\':r[\'paper_key\'],\'split\':\'validation\',\'source\':r[\'plain_english\'],\'reference\':r[\'completion\']} for r in val],\\n        \'execution_gate\':{\'ready\':ready,\'reason\':\'Frozen checker passed synthetic gate\' if ready else \'Frozen checker failed synthetic gate; do not generate more comparison text yet\',\\n            \'benchmark_summary_sha256\':file_sha(benchmark_summary)},\\n        \'scope\':\'Non-oracle fixed soft prior, not a learned taxonomy mapper. Existing 21 development examples, not fresh validation. No reserved test file read.\'}\\n    jobs=jobs_for(pack);fresh(output);write_json(output/\'soft_guidance_pack.private.json\',pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'status\':\'prepared_not_executed\',\'pairs\':21,\'jobs\':42,\'execution_ready\':ready,\\n        \'training_called\':False,\'model_calls\':0,\'reserved_test_rows_used\':0,\'code_sha256\':file_sha(Path(__file__)),\\n        \'input_hashes\':{str(p):file_sha(p) for p in (training,validation,benchmark_summary)},\\n        \'files\':{n:file_sha(output/n) for n in (\'soft_guidance_pack.private.json\',\'jobs.private.jsonl\')}})\\n    print(json.dumps({\'pairs\':21,\'jobs\':42,\'execution_ready\':ready,\'model_calls\':0,\'reserved_test_rows_used\':0}));return pack\\n\\n\\ndef execution_allowed(pack):\\n    \\"\\"\\"A passing synthetic gate is necessary, not proof of domain reliability.\\"\\"\\"\\n    if pack[\'execution_gate\'][\'ready\'] is not True:raise ValueError(\'Claim-checker readiness gate failed; comparison execution remains blocked\')\\n    return True\\n\\n\\ndef main(argv=None):\\n    p=argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\'--training\',type=Path,default=ROOT/\'data/fama_corpus_v1/train.jsonl\')\\n    p.add_argument(\'--validation\',type=Path,default=ROOT/\'data/fama_corpus_v1/validation.jsonl\')\\n    p.add_argument(\'--benchmark-summary\',type=Path,required=True);p.add_argument(\'--output-dir\',type=Path,required=True)\\n    a=p.parse_args(argv);prepare(a.training,a.validation,a.benchmark_summary,a.output_dir)\\n\\n\\nif __name__==\'__main__\':main()\\n", "src/fama_style/rhetoric/soft_guidance_experiment.py": "\\"\\"\\"Matched 21-example, count-free soft-guidance inference; no trainer/adapter.\\"\\"\\"\\nimport argparse\\nfrom collections import Counter\\nimport gc\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport statistics\\nimport time\\n\\nfrom .soft_guidance import jobs_for,soft_plan,VALIDATION_SHA,CONDITIONS\\nfrom .concise_plan import TRAIN_SHA,MODEL_ID,MODEL_REVISION,NLI_MODEL,NLI_REVISION,file_sha,read_lines\\nfrom .experiment import fresh,write_json,write_lines,secret_token,with_heartbeat\\nfrom .claim_constrained import calibration_controls\\nfrom fama_style.research import claim_verifier_v3 as verifier\\nfrom fama_style.research.claims import sha,sentences,tokens\\nfrom fama_style.research.signal import fit_profile,distance,overlap,features\\n\\nROOT=Path(__file__).resolve().parents[3]\\nPROTOCOL=\'soft_guidance_validation_v2\'\\nMAX_NEW_TOKENS=768\\nCONTEXT_LIMIT=4096\\nVERIFIER_FOLDER=ROOT/\'docs/claim_verifier_calibration_v3\'\\n\\n\\ndef runtime_paths():\\n    paths=[ROOT/\'src/fama_style/__init__.py\',ROOT/\'src/fama_style/prompting/__init__.py\',ROOT/\'src/fama_style/prompting/corpus.py\']\\n    return paths+sorted((ROOT/\'src/fama_style/research\').glob(\'*.py\'))+sorted((ROOT/\'src/fama_style/rhetoric\').glob(\'*.py\'))\\n\\n\\ndef prepare_pack(output):\\n    training=ROOT/\'data/fama_corpus_v1/train.jsonl\';validation=ROOT/\'data/fama_corpus_v1/validation.jsonl\'\\n    if file_sha(training)!=TRAIN_SHA or file_sha(validation)!=VALIDATION_SHA:raise ValueError(\'Frozen train/validation bytes changed\')\\n    train,val=read_lines(training),read_lines(validation)\\n    if len(train)!=247 or any(r[\'split\']!=\'train\' for r in train):raise ValueError(\'Wrong training split\')\\n    if len(val)!=21 or any(r[\'split\']!=\'validation\' for r in val):raise ValueError(\'Wrong validation split\')\\n    if {r[\'id\'] for r in train}&{r[\'id\'] for r in val} or {r[\'paper_key\'] for r in train}&{r[\'paper_key\'] for r in val}:raise ValueError(\'Overlapping papers or IDs\')\\n    base=ROOT/\'research_runs/claim_verifier_calibration_v3\';run=base/\'kaggle_version_1/verifier_calibration_v3\'\\n    review_path=base/\'version_1_review/verification.json\';review=json.loads(review_path.read_text())\\n    if not review[\'checks\'] or not all(review[\'checks\'].values()):raise ValueError(\'Calibration review did not pass\')\\n    for name,digest in review[\'artifact_hashes\'].items():\\n        if file_sha(run/name)!=digest:raise ValueError(\'Reviewed calibration changed\')\\n    result_path=run/\'evaluation.json\';result=json.loads(result_path.read_text());manifest=json.loads((run/\'manifest.json\').read_text())\\n    protocol,_=verifier.validate_protocol(VERIFIER_FOLDER)\\n    if (result[\'protocol\']!=verifier.PROTOCOL or result[\'execution_kind\']!=\'pinned_nli_inference\'\\n        or result[\'readiness_gate\']!=verifier.gate(result[\'splits\'][\'held_out\'],protocol[\'readiness_gate\'])\\n        or result[\'readiness_gate\'][\'passed\'] is not True or manifest[\'code_files\']!=protocol[\'code_files\']\\n        or manifest[\'protocol_sha256\']!=file_sha(VERIFIER_FOLDER/\'protocol.json\')):raise ValueError(\'Unusable verifier calibration\')\\n    runtime=json.loads((run/\'runtime.json\').read_text())\\n    if runtime[\'model_id\']!=NLI_MODEL or runtime[\'model_revision\']!=NLI_REVISION:raise ValueError(\'Changed NLI revision\')\\n    pack={\'protocol\':PROTOCOL,\'parent_training_sha256\':TRAIN_SHA,\'validation_sha256\':VALIDATION_SHA,\\n        \'style_profile\':fit_profile(train),\\n        \'rows\':[{\'id\':r[\'id\'],\'paper_key\':r[\'paper_key\'],\'split\':\'validation\',\'source\':r[\'plain_english\'],\'reference\':r[\'completion\']} for r in val],\\n        \'execution_gate\':{\'ready\':True,\'reason\':\'Reviewed v3 synthetic gate passed; exploratory inference only, not domain reliability\'},\\n        \'verifier_evidence\':{\'notebook\':\'oliverlds/verifier_calibration_v3\',\'version\':1,\'protocol\':verifier.PROTOCOL,\\n            \'evaluation_sha256\':file_sha(result_path),\'review_sha256\':file_sha(review_path),\'held_out_metrics\':result[\'splits\'][\'held_out\'],\\n            \'protocol_sha256\':manifest[\'protocol_sha256\'],\'code_files\':protocol[\'code_files\'],\'fixture_files\':protocol[\'fixture_files\']},\\n        \'generation\':{\'model_id\':MODEL_ID,\'revision\':MODEL_REVISION,\'max_new_tokens\':MAX_NEW_TOKENS,\'context_limit\':CONTEXT_LIMIT,\'do_sample\':False,\'seed\':42,\'truncate\':False},\\n        \'scope\':\'Existing 21 development examples. Fixed training-derived soft prior, not a learned full-taxonomy mapper. References only for post-generation diagnostics. Reserved paper test split never read.\'}\\n    jobs=jobs_for(pack)\\n    pack[\'runtime_code_files\']={str(p.relative_to(ROOT)):file_sha(p) for p in runtime_paths()}\\n    pack[\'jobs_sha256\']=sha(json.dumps(jobs,sort_keys=True,ensure_ascii=False))\\n    fresh(output);path=output/\'soft_guidance_pack.json\';write_json(path,pack)\\n    write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'status\':\'prepared_not_executed\',\'pairs\':21,\'jobs\':42,\'pack_sha256\':file_sha(path),\\n        \'training_called\':False,\'model_calls\':0,\'reserved_test_rows_used\':0,\\n        \'input_hashes\':{str(p.relative_to(ROOT)):file_sha(p) for p in (training,validation,result_path,review_path)},\\n        \'files\':{p.name:file_sha(p) for p in output.iterdir() if p.is_file()}})\\n    print(json.dumps({\'status\':\'prepared_not_executed\',\'pairs\':21,\'jobs\':42,\'pack_sha256\':file_sha(path)}));return pack\\n\\n\\ndef load_pack(path,expected_sha):\\n    if not re.fullmatch(\'[0-9a-f]{64}\',expected_sha) or file_sha(path)!=expected_sha:raise ValueError(\'Input pack hash mismatch\')\\n    pack=json.loads(path.read_text())\\n    if pack[\'protocol\']!=PROTOCOL or pack[\'parent_training_sha256\']!=TRAIN_SHA or pack[\'validation_sha256\']!=VALIDATION_SHA:raise ValueError(\'Wrong frozen data/protocol\')\\n    if pack[\'execution_gate\'][\'ready\'] is not True:raise ValueError(\'Verifier gate did not pass\')\\n    for name,h in pack[\'runtime_code_files\'].items():\\n        if file_sha(ROOT/name)!=h:raise ValueError(\'Frozen inference implementation changed\')\\n    profile=pack[\'style_profile\']\\n    if profile[\'fit_split\']!=\'train\' or profile[\'training_rows\']!=247:raise ValueError(\'Wrong style-profile split\')\\n    evidence=pack[\'verifier_evidence\'];protocol,_=verifier.validate_protocol(VERIFIER_FOLDER)\\n    if evidence[\'protocol_sha256\']!=file_sha(VERIFIER_FOLDER/\'protocol.json\') or evidence[\'code_files\']!=protocol[\'code_files\'] or evidence[\'fixture_files\']!=protocol[\'fixture_files\']:raise ValueError(\'Verifier evidence changed\')\\n    if not verifier.gate(evidence[\'held_out_metrics\'],protocol[\'readiness_gate\'])[\'passed\']:raise ValueError(\'Synthetic calibration is not ready\')\\n    if pack[\'generation\']!={\'model_id\':MODEL_ID,\'revision\':MODEL_REVISION,\'max_new_tokens\':MAX_NEW_TOKENS,\'context_limit\':CONTEXT_LIMIT,\'do_sample\':False,\'seed\':42,\'truncate\':False}:raise ValueError(\'Changed generation controls\')\\n    if pack[\'jobs_sha256\']!=sha(json.dumps(jobs_for(pack),sort_keys=True,ensure_ascii=False)):raise ValueError(\'Prepared prompts changed\')\\n    if any(not r[\'source\'].strip() or not r[\'reference\'].strip() for r in pack[\'rows\']):raise ValueError(\'Empty pair\')\\n    return pack\\n\\n\\ndef audit_tokens(job,ids):\\n    if sha(job[\'prompt\'])!=job[\'prompt_sha256\'] or not ids or len(ids)+MAX_NEW_TOKENS>CONTEXT_LIMIT:raise ValueError(\'Changed/overflowing prompt; no truncation allowed\')\\n    return {\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\'input_tokens\':len(ids),\'input_ids_sha256\':sha(json.dumps(ids))}\\n\\n\\ndef paragraph_format(text):\\n    return not bool(re.match(r\'\\\\s*(?:```|[\\\\[{]|#+\\\\s|[-*]\\\\s|\\\\d+\\\\.\\\\s)\',text) or re.search(r\'\\\\n\\\\s*\\\\n\',text))\\n\\n\\ndef evaluate(pack,jobs,outputs,semantic=None,pair_checks=None,canary_passed=False,synthetic=False):\\n    if jobs!=jobs_for(pack) or len(outputs)!=len(jobs):raise ValueError(\'Changed jobs or incomplete outputs\')\\n    expected={(j[\'id\'],j[\'condition\']):j for j in jobs};seen=set();rows={r[\'id\']:r for r in pack[\'rows\']};records=[]\\n    if canary_passed and not synthetic and set(semantic or {})!=set(expected):raise ValueError(\'Incomplete verifier results\')\\n    for out in outputs:\\n        key=(out[\'id\'],out[\'condition\']);job=expected.get(key)\\n        if key in seen or job is None or out[\'prompt_sha256\']!=job[\'prompt_sha256\']:raise ValueError(\'Duplicate/swapped output\')\\n        if not out.get(\'output\',\'\').strip() or out.get(\'hit_output_limit\') is not False or out.get(\'synthetic\') is not synthetic:raise ValueError(\'Empty/cutoff/mislabeled output\')\\n        seen.add(key);row=rows[out[\'id\']];text=out[\'output\'];verdict=(semantic or {}).get(key);valid=paragraph_format(text)\\n        status=verdict[\'status\'] if verdict else \'not_run\';sim=overlap(row[\'source\'],text)\\n        if verdict is not None and status not in (\'model_consistent\',\'risk_signal\',\'abstain\'):raise ValueError(\'Unknown verifier status\')\\n        gain=distance(row[\'source\'],pack[\'style_profile\'])-distance(text,pack[\'style_profile\'])\\n        usable=canary_passed and not synthetic and verdict is not None\\n        records.append({\'id\':row[\'id\'],\'condition\':out[\'condition\'],\'output_sha256\':sha(text),\'format_valid\':valid,\\n            \'verifier_status\':status,\'verifier\':verdict,\'source_similarity\':sim,\'near_copy\':sim>=.9,\\n            \'exact_copy\':\' \'.join(row[\'source\'].split())==\' \'.join(text.split()),\'token_normalized_copy\':tokens(row[\'source\'])==tokens(text),\\n            \'style_gain_proxy\':gain,\'source_features\':features(row[\'source\']),\'output_features\':features(text),\\n            \'source_sentence_count\':len(sentences(row[\'source\'])),\'output_sentence_count\':len(sentences(text)),\\n            \'reference_similarity_diagnostic_only\':overlap(row[\'reference\'],text),\\n            \'valid_semantic_proxy_pass\':valid and status==\'model_consistent\' if usable else None,\\n            \'joint_proxy_success\':valid and status==\'model_consistent\' and sim<.9 and gain>0 if usable else None,\\n            \'input_tokens\':out.get(\'input_tokens\',0),\'output_tokens\':out.get(\'output_tokens\',0),\'generation_seconds\':out.get(\'generation_seconds\',0)})\\n    aggregate={}\\n    for c in CONDITIONS:\\n        group=[r for r in records if r[\'condition\']==c];valid=[r for r in group if r[\'format_valid\']]\\n        aggregate[c]={\'rows\':len(group),\'valid_formats\':len(valid),\'verifier_status_counts\':dict(Counter(r[\'verifier_status\'] for r in group)),\\n            \'valid_semantic_proxy_passes\':sum(bool(r[\'valid_semantic_proxy_pass\']) for r in group) if canary_passed and not synthetic else None,\\n            \'joint_proxy_successes\':sum(bool(r[\'joint_proxy_success\']) for r in group) if canary_passed and not synthetic else None,\\n            \'near_copies\':sum(r[\'near_copy\'] for r in group),\'exact_copies\':sum(r[\'exact_copy\'] for r in group),\\n            \'mean_style_gain_proxy_all_outputs\':statistics.mean(r[\'style_gain_proxy\'] for r in group),\\n            \'mean_style_gain_proxy_valid_only\':statistics.mean(r[\'style_gain_proxy\'] for r in valid) if valid else None,\\n            \'mean_source_similarity\':statistics.mean(r[\'source_similarity\'] for r in group),\\n            **{k:sum(r[k] or 0 for r in group) for k in (\'input_tokens\',\'output_tokens\',\'generation_seconds\')}}\\n    lookup={(r[\'id\'],r[\'condition\']):r for r in records};paired=[]\\n    for row in pack[\'rows\']:\\n        a,b=(lookup[(row[\'id\'],c)] for c in CONDITIONS)\\n        paired.append({\'id\':row[\'id\'],\'A_verifier_status\':a[\'verifier_status\'],\'B_verifier_status\':b[\'verifier_status\'],\\n            \'A_format_valid\':a[\'format_valid\'],\'B_format_valid\':b[\'format_valid\'],\\n            \'style_gain_difference_B_minus_A\':b[\'style_gain_proxy\']-a[\'style_gain_proxy\'],\\n            \'source_similarity_difference_B_minus_A\':b[\'source_similarity\']-a[\'source_similarity\'],\\n            \'A_joint_proxy_success\':a[\'joint_proxy_success\'],\'B_joint_proxy_success\':b[\'joint_proxy_success\']})\\n    return records,{\'protocol\':PROTOCOL,\'execution_kind\':\'synthetic_wiring_check\' if synthetic else \'model_inference\',\\n        \'conditions\':aggregate,\'paired\':paired,\'pair_source_reference_verifier\':pair_checks,\'runtime_canary_passed\':canary_passed,\\n        \'training_called\':False,\'reserved_test_rows_used\':0,\'development_validation_rows_used\':21,\'model_selection_test_rows_used\':0,\\n        \'training_ready\':False,\'semantic_accuracy\':None,\'author_style_accuracy\':None,\'oracle\':False,\\n        \'limitations\':\'Previously used development set from one paper. Fixed source-bound soft prior, not a learned full-taxonomy mapper. Verifier consistency and style distance are fallible proxies; unknowns and bad formats remain in all-example denominators. No hard sentence quotas or automatic repairs.\'}\\n\\n\\ndef run_training(pack,output,uploaded_pack_sha):\\n    \\"\\"\\"Inference only. Check public verifier canary before accessing the Gemma Secret.\\"\\"\\"\\n    fresh(output);jobs=jobs_for(pack);write_lines(output/\'jobs.private.jsonl\',jobs)\\n    write_json(output/\'uploaded_pack_identity.json\',{\'sha256\':uploaded_pack_sha,\'format\':\'uploaded file bytes\'})\\n    status={\'complete\':False,\'protocol\':PROTOCOL,\'training_called\':False,\'reserved_test_rows_used\':0,\'completed_jobs\':0,\'total_jobs\':len(jobs)}\\n    started=time.monotonic();outputs=[]\\n    def phase(name,**extra):\\n        status.update(phase=name,**extra);write_json(output/\'run_status.json\',status);print(\'Phase: \'+name,flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        import os\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\';os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\';os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor,AutoTokenizer,AutoModelForSequenceClassification,\\n            Gemma3ForConditionalGeneration,BitsAndBytesConfig,LogitsProcessor,LogitsProcessorList)\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:raise RuntimeError(\'Expose one Kaggle GPU before importing torch\')\\n        torch.cuda.init();torch.cuda.set_device(0);torch.manual_seed(42)\\n        if \'T4\' not in torch.cuda.get_device_name(0):raise RuntimeError(\'Select Kaggle T4 x2\')\\n        cache={};cache_hits=0\\n        def load_nli():\\n            tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=NLI_REVISION,token=False),\'NLI tokenizer load\')\\n            nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=NLI_REVISION,\\n                token=False,dtype=torch.float32).to(\'cuda:0\'),\'NLI model load\')\\n            nli.eval();nli.requires_grad_(False);labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n            if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}:raise ValueError(\'Unexpected NLI labels\')\\n            limit=min(512,int(tokenizer.model_max_length),int(getattr(nli.config,\'max_position_embeddings\',512)))\\n            def predict(premise,hypothesis):\\n                nonlocal cache_hits\\n                key=(sha(premise),sha(hypothesis))\\n                if key in cache:cache_hits+=1;return cache[key]\\n                batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n                if int(batch[\'input_ids\'].shape[-1])>limit:cache[key]=None;return None\\n                with torch.inference_mode():\\n                    probs=torch.softmax(nli(**{k:v.to(\'cuda:0\') for k,v in batch.items()}).logits[0],dim=-1).tolist()\\n                cache[key]={labels[i]:float(v) for i,v in enumerate(probs)};return cache[key]\\n            return tokenizer,nli,predict\\n        phase(\'runtime_verifier_canary\');torch.cuda.reset_peak_memory_stats(0);canary_started=time.monotonic()\\n        nli_tokenizer,nli,predict=load_nli()\\n        canary=verifier.evaluate(VERIFIER_FOLDER,output/\'runtime_calibration\',predict,\'pinned_nli_inference\')\\n        canary_seconds=time.monotonic()-canary_started\\n        if canary[\'readiness_gate\'][\'passed\'] is not True:raise RuntimeError(\'Runtime verifier canary failed; no Gemma generation performed\')\\n        del nli_tokenizer,nli,predict;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'checking_secret\');token=secret_token()\\n        processor=with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token),\'Processor load\')\\n        def tokenize(job):\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[audit_tokens(job,tokenize(job)[\'input_ids\'][0].tolist()) for job in jobs];write_json(output/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)),\'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):raise RuntimeError(\'Unexpected model placement\')\\n        model.eval();model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        by_key={(r[\'id\'],r[\'condition\']):r for r in audits}\\n        for job in jobs:\\n            phase(\'generation\',completed_jobs=len(outputs));batch=tokenize(job);ids=batch[\'input_ids\'][0].tolist()\\n            if audit_tokens(job,ids)!=by_key[(job[\'id\'],job[\'condition\'])]:raise ValueError(\'Tokenized prompt changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()};before=time.monotonic()\\n            with torch.inference_mode():\\n                generated=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,use_cache=True,\\n                    pad_token_id=processor.tokenizer.pad_token_id,remove_invalid_values=False,\\n                    logits_processor=LogitsProcessorList([FiniteScores()])),\'Generation\')\\n            new=generated[0,len(ids):];eos=model.generation_config.eos_token_id;eos=eos if isinstance(eos,list) else [eos]\\n            text=processor.tokenizer.decode(new,skip_special_tokens=True).strip()\\n            result={\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'prompt_sha256\':job[\'prompt_sha256\'],\'output\':text,\\n                \'input_tokens\':len(ids),\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            outputs.append(result);write_lines(output/\'generations.private.jsonl\',outputs)\\n            if not text or result[\'hit_output_limit\']:raise RuntimeError(\'Empty/cutoff output; partial results remain incomplete\')\\n        gemma_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del model,processor,batch,generated,new,tokenize;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)\\n        phase(\'loading_verifier_after_gemma_unload\');nli_started=time.monotonic();nli_tokenizer,nli,predict=load_nli()\\n        semantic={};pair_checks={};sources={r[\'id\']:r[\'source\'] for r in pack[\'rows\']}\\n        for index,row in enumerate(pack[\'rows\'],1):\\n            phase(\'pair_fidelity_assessment\',completed_pairs=index-1)\\n            pair_checks[row[\'id\']]=with_heartbeat(lambda:verifier.verify(row[\'source\'],row[\'reference\'],predict),\'Pair verifier\')\\n        for index,result in enumerate(outputs,1):\\n            phase(\'output_semantic_assessment\',completed_verifier_outputs=index-1)\\n            semantic[(result[\'id\'],result[\'condition\'])]=with_heartbeat(lambda:verifier.verify(sources[result[\'id\']],result[\'output\'],predict),\'Output verifier\')\\n        records,summary=evaluate(pack,jobs,outputs,semantic,pair_checks,canary_passed=True)\\n        write_lines(output/\'metrics.private.jsonl\',records)\\n        write_lines(output/\'nli_probability_cache.private.jsonl\',[{\'premise_sha256\':a,\'hypothesis_sha256\':b,\'probabilities\':p} for (a,b),p in cache.items()])\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=NLI_REVISION,\\n            canary_seconds=canary_seconds,verifier_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,\\n            nli_cache_entries=len(cache),nli_cache_hits=cache_hits,gemma_memory=gemma_memory,\\n            nli_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)},\\n            generation=pack[\'generation\'],versions={n:importlib.metadata.version(n) for n in (\'torch\',\'transformers\',\'accelerate\',\'bitsandbytes\',\'huggingface-hub\')})\\n        write_json(output/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True,completed_jobs=42,completed_pairs=21,completed_verifier_outputs=42)\\n        write_json(output/\'manifest.json\',{\'protocol\':PROTOCOL,\'training_called\':False,\'input_pack_sha256\':uploaded_pack_sha,\\n            \'files\':{str(p.relative_to(output)):file_sha(p) for p in output.rglob(\'*\') if p.is_file()}})\\n    except Exception as error:\\n        failed_at=status.get(\'phase\');locations=[];tb=error.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:locations.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,failed_at=failed_at,error_code=\'soft_guidance_inference_failed\',local_line_numbers=locations)\\n        raise RuntimeError(\'Soft-guidance inference failed; inspect run_status.json and saved partial files. No training was called.\') from None\\n\\n\\ndef synthetic_pack():\\n    examples=[{\'split\':\'train\',\'plain_english\':\'Firms report sales. Investors read reports.\',\'completion\':\'Firms report sales; investors read reports.\'}]\\n    return {\'protocol\':PROTOCOL,\'style_profile\':fit_profile(examples),\\n        \'rows\':[{\'id\':\'synthetic-\'+str(i),\'paper_key\':\'synthetic-validation\',\'split\':\'validation\',\'source\':\'Firms report sales. Investors read reports.\',\\n                 \'reference\':\'Firms report sales; investors read reports.\'} for i in range(21)]}\\n\\n\\ndef main(argv=None):\\n    p=argparse.ArgumentParser(description=__doc__);actions=p.add_mutually_exclusive_group()\\n    actions.add_argument(\'--prepare-pack\',action=\'store_true\');actions.add_argument(\'--dry-run\',action=\'store_true\');actions.add_argument(\'--execute-inference\',action=\'store_true\')\\n    p.add_argument(\'--pack\',type=Path);p.add_argument(\'--expected-pack-sha\');p.add_argument(\'--output-dir\',type=Path,required=True);a=p.parse_args(argv)\\n    if a.prepare_pack:prepare_pack(a.output_dir);return\\n    if a.dry_run:\\n        pack=synthetic_pack();jobs=jobs_for(pack);by_id={r[\'id\']:r for r in pack[\'rows\']};fresh(a.output_dir)\\n        outputs=[{\'id\':j[\'id\'],\'condition\':j[\'condition\'],\'prompt_sha256\':j[\'prompt_sha256\'],\'output\':by_id[j[\'id\']][\'source\'],\'hit_output_limit\':False,\'synthetic\':True} for j in jobs]\\n        records,summary=evaluate(pack,jobs,outputs,synthetic=True)\\n        write_json(a.output_dir/\'synthetic_evaluation.json\',summary);write_lines(a.output_dir/\'synthetic_jobs.jsonl\',jobs)\\n        print(\'Synthetic wiring check: 42 jobs; no model calls or training.\');return\\n    if a.pack is None or a.expected_pack_sha is None:p.error(\'--pack and --expected-pack-sha required\')\\n    pack=load_pack(a.pack,a.expected_pack_sha)\\n    if a.execute_inference:run_training(pack,a.output_dir,a.expected_pack_sha)\\n    else:\\n        fresh(a.output_dir);write_json(a.output_dir/\'preflight.json\',{\'protocol\':PROTOCOL,\'pack_sha256\':a.expected_pack_sha,\'pairs\':21,\'jobs\':42,\'model_calls\':0,\'training_called\':False})\\n        print(\'Preflight passed; no Secret access, model download or inference.\')\\n\\n\\nif __name__==\'__main__\':main()\\n", "docs/claim_verifier_calibration_v3/development.json": "[\\n  {\\n    \\"id\\": \\"v2_v1_development_01_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The approximation is useful only when noise does not exceed 6 units.\\",\\n    \\"output\\": \\"The approximation is useful only when noise does not exceed 6 units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_01_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The approximation is useful only when noise does not exceed 6 units.\\",\\n    \\"output\\": \\"The approximation is useful only if noise is at most six units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_01_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The approximation is useful only when noise does not exceed 6 units.\\",\\n    \\"output\\": \\"The approximation is useful only when noise does not fall below 6 units.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_02_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The broker pays the fund manager.\\",\\n    \\"output\\": \\"The broker pays the fund manager.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_02_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The broker pays the fund manager.\\",\\n    \\"output\\": \\"The fund manager is paid by the broker.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_02_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The broker pays the fund manager.\\",\\n    \\"output\\": \\"The fund manager pays the broker.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_03_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Cedar owns 4 shares and Maple owns 9 shares.\\",\\n    \\"output\\": \\"Cedar owns 4 shares and Maple owns 9 shares.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_03_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Cedar owns 4 shares and Maple owns 9 shares.\\",\\n    \\"output\\": \\"Four shares belong to Cedar, while nine belong to Maple.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_03_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Cedar owns 4 shares and Maple owns 9 shares.\\",\\n    \\"output\\": \\"Cedar owns 9 shares and Maple owns 4 shares.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_04_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Platforms issue forecasts each year.\\",\\n    \\"output\\": \\"Platforms issue forecasts each year.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_04_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Platforms issue forecasts each year.\\",\\n    \\"output\\": \\"Platforms issue forecasts every twelve months.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_04_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Platforms issue forecasts each year.\\",\\n    \\"output\\": \\"Platforms issue forecasts.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_05_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"If demand falls, shops reduce inventory.\\",\\n    \\"output\\": \\"If demand falls, shops reduce inventory.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_05_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"If demand falls, shops reduce inventory.\\",\\n    \\"output\\": \\"Shops reduce inventory if demand falls.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_05_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"If demand falls, shops reduce inventory.\\",\\n    \\"output\\": \\"Shops reduce inventory.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_06_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The project may fail.\\",\\n    \\"output\\": \\"The project may fail.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_06_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The project may fail.\\",\\n    \\"output\\": \\"The project could fail.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_06_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The project may fail.\\",\\n    \\"output\\": \\"The project will certainly fail.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_07_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Most firms are small.\\",\\n    \\"output\\": \\"Most firms are small.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_07_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Most firms are small.\\",\\n    \\"output\\": \\"A majority of firms are small.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_07_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Most firms are small.\\",\\n    \\"output\\": \\"All firms are small.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_08_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Northbridge costs rose.\\",\\n    \\"output\\": \\"Northbridge costs rose.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_08_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Northbridge costs rose.\\",\\n    \\"output\\": \\"Northbridge experienced an increase in costs.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_08_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Northbridge costs rose.\\",\\n    \\"output\\": \\"Northbridge costs rose. This proves that its investments are safe.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_09_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The company does not invest.\\",\\n    \\"output\\": \\"The company does not invest.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_09_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The company does not invest.\\",\\n    \\"output\\": \\"The company refrains from investing.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_09_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The company does not invest.\\",\\n    \\"output\\": \\"The company invests.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_10_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Higher fees cause lower trading volume.\\",\\n    \\"output\\": \\"Higher fees cause lower trading volume.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_10_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Higher fees cause lower trading volume.\\",\\n    \\"output\\": \\"Lower trading volume is caused by higher fees.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_10_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Higher fees cause lower trading volume.\\",\\n    \\"output\\": \\"Lower trading volume causes higher fees.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_11_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The service fee is 12 percent.\\",\\n    \\"output\\": \\"The service fee is 12 percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_11_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The service fee is 12 percent.\\",\\n    \\"output\\": \\"The service fee is twelve percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_11_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The service fee is 12 percent.\\",\\n    \\"output\\": \\"The service fee is 18 percent.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_12_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_development_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"All units operate except the backup unit.\\",\\n    \\"output\\": \\"All units operate except the backup unit.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_12_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_development_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"All units operate except the backup unit.\\",\\n    \\"output\\": \\"Every unit other than the backup unit operates.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_development_12_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_development_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"All units operate except the backup unit.\\",\\n    \\"output\\": \\"All units operate.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_01_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The device works only when temperature does not exceed 8 degrees.\\",\\n    \\"output\\": \\"The device works only when temperature does not exceed 8 degrees.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_01_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The device works only when temperature does not exceed 8 degrees.\\",\\n    \\"output\\": \\"The device works only if temperature is no greater than eight degrees.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_01_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The device works only when temperature does not exceed 8 degrees.\\",\\n    \\"output\\": \\"The device works only when temperature does not fall below 8 degrees.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_02_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The tenant reimburses the owner.\\",\\n    \\"output\\": \\"The tenant reimburses the owner.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_02_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The tenant reimburses the owner.\\",\\n    \\"output\\": \\"The owner receives reimbursement from the tenant.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_02_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The tenant reimburses the owner.\\",\\n    \\"output\\": \\"The owner reimburses the tenant.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_03_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Harbor holds 7 bonds and Ridge holds 15 bonds.\\",\\n    \\"output\\": \\"Harbor holds 7 bonds and Ridge holds 15 bonds.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_03_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Harbor holds 7 bonds and Ridge holds 15 bonds.\\",\\n    \\"output\\": \\"Seven bonds are held by Harbor; fifteen are held by Ridge.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_03_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Harbor holds 7 bonds and Ridge holds 15 bonds.\\",\\n    \\"output\\": \\"Harbor holds 15 bonds and Ridge holds 7 bonds.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_04_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Laboratories publish accounts every year.\\",\\n    \\"output\\": \\"Laboratories publish accounts every year.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_04_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Laboratories publish accounts every year.\\",\\n    \\"output\\": \\"Laboratories publish accounts at twelve-month intervals.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_04_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Laboratories publish accounts every year.\\",\\n    \\"output\\": \\"Laboratories publish accounts.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_05_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"If rainfall declines, growers reduce planting.\\",\\n    \\"output\\": \\"If rainfall declines, growers reduce planting.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_05_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"If rainfall declines, growers reduce planting.\\",\\n    \\"output\\": \\"Growers reduce planting if rainfall declines.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_05_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"If rainfall declines, growers reduce planting.\\",\\n    \\"output\\": \\"Growers reduce planting.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_06_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The trial might succeed.\\",\\n    \\"output\\": \\"The trial might succeed.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_06_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The trial might succeed.\\",\\n    \\"output\\": \\"The trial could succeed.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_06_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The trial might succeed.\\",\\n    \\"output\\": \\"The trial will definitely succeed.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_07_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Most suppliers are local.\\",\\n    \\"output\\": \\"Most suppliers are local.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_07_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Most suppliers are local.\\",\\n    \\"output\\": \\"More than half of suppliers are local.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_07_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Most suppliers are local.\\",\\n    \\"output\\": \\"Every supplier is local.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_08_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Westhaven sales declined.\\",\\n    \\"output\\": \\"Westhaven sales declined.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_08_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Westhaven sales declined.\\",\\n    \\"output\\": \\"Westhaven experienced a decline in sales.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_08_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Westhaven sales declined.\\",\\n    \\"output\\": \\"Westhaven sales declined. This establishes that its management is dishonest.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_09_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The partnership does not borrow.\\",\\n    \\"output\\": \\"The partnership does not borrow.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_09_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The partnership does not borrow.\\",\\n    \\"output\\": \\"The partnership refrains from borrowing.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_09_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The partnership does not borrow.\\",\\n    \\"output\\": \\"The partnership borrows.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_10_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Higher rents cause lower occupancy.\\",\\n    \\"output\\": \\"Higher rents cause lower occupancy.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_10_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Higher rents cause lower occupancy.\\",\\n    \\"output\\": \\"Lower occupancy results from higher rents.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_10_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Higher rents cause lower occupancy.\\",\\n    \\"output\\": \\"Lower occupancy causes higher rents.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_11_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The levy is 14 percent.\\",\\n    \\"output\\": \\"The levy is 14 percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_11_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The levy is 14 percent.\\",\\n    \\"output\\": \\"The levy is fourteen percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_11_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The levy is 14 percent.\\",\\n    \\"output\\": \\"The levy is 19 percent.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_12_identity\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"All offices open except the coastal office.\\",\\n    \\"output\\": \\"All offices open except the coastal office.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_12_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"All offices open except the coastal office.\\",\\n    \\"output\\": \\"Every office other than the coastal office opens.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_v1_held_out_12_mutation\\",\\n    \\"scenario_group\\": \\"v2_v1_held_out_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"All offices open except the coastal office.\\",\\n    \\"output\\": \\"All offices open.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_01_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Calibration remains valid only when drift does not exceed 9 units.\\",\\n    \\"output\\": \\"Calibration remains valid only when drift does not exceed 9 units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_01_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Calibration remains valid only when drift does not exceed 9 units.\\",\\n    \\"output\\": \\"Calibration remains valid only if drift is at most nine units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_01_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_01\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_direction\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Calibration remains valid only when drift does not exceed 9 units.\\",\\n    \\"output\\": \\"Calibration remains valid only when drift does not fall below 9 units.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_02_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The insurer reimburses the merchant.\\",\\n    \\"output\\": \\"The insurer reimburses the merchant.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_02_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The insurer reimburses the merchant.\\",\\n    \\"output\\": \\"The merchant is reimbursed by the insurer.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_02_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_02\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The insurer reimburses the merchant.\\",\\n    \\"output\\": \\"The merchant reimburses the insurer.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_03_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Oriole holds 6 bonds and Willow holds 13 bonds.\\",\\n    \\"output\\": \\"Oriole holds 6 bonds and Willow holds 13 bonds.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_03_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Oriole holds 6 bonds and Willow holds 13 bonds.\\",\\n    \\"output\\": \\"Six bonds are held by Oriole; thirteen bonds are held by Willow.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_03_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_03\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Oriole holds 6 bonds and Willow holds 13 bonds.\\",\\n    \\"output\\": \\"Oriole holds 13 bonds and Willow holds 6 bonds.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_04_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Auditors inspect records each year.\\",\\n    \\"output\\": \\"Auditors inspect records each year.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_04_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Auditors inspect records each year.\\",\\n    \\"output\\": \\"Auditors inspect records at twelve-month intervals.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_04_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_04\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"frequency_qualification\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Auditors inspect records each year.\\",\\n    \\"output\\": \\"Auditors inspect records.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_05_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"If wages rise, clinics reduce hiring.\\",\\n    \\"output\\": \\"If wages rise, clinics reduce hiring.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_05_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"If wages rise, clinics reduce hiring.\\",\\n    \\"output\\": \\"Clinics reduce hiring if wages rise.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_05_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_05\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"If wages rise, clinics reduce hiring.\\",\\n    \\"output\\": \\"Clinics reduce hiring.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_06_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The agreement might lapse.\\",\\n    \\"output\\": \\"The agreement might lapse.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_06_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The agreement might lapse.\\",\\n    \\"output\\": \\"It is possible that the agreement will lapse.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_06_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_06\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The agreement might lapse.\\",\\n    \\"output\\": \\"The agreement will certainly lapse.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_07_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Most shipments are delayed.\\",\\n    \\"output\\": \\"Most shipments are delayed.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_07_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Most shipments are delayed.\\",\\n    \\"output\\": \\"More than half of shipments are delayed.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_07_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_07\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Most shipments are delayed.\\",\\n    \\"output\\": \\"All shipments are delayed.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_08_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Meadowgate revenue increased.\\",\\n    \\"output\\": \\"Meadowgate revenue increased.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_08_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Meadowgate revenue increased.\\",\\n    \\"output\\": \\"Meadowgate experienced an increase in revenue.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_08_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_08\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"added_assertion\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Meadowgate revenue increased.\\",\\n    \\"output\\": \\"Meadowgate revenue increased. Its directors therefore deserve higher salaries.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_09_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The municipality does not issue guarantees.\\",\\n    \\"output\\": \\"The municipality does not issue guarantees.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_09_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The municipality does not issue guarantees.\\",\\n    \\"output\\": \\"The municipality issues no guarantees.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_09_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_09\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The municipality does not issue guarantees.\\",\\n    \\"output\\": \\"The municipality issues guarantees.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_10_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Higher tariffs cause lower imports.\\",\\n    \\"output\\": \\"Higher tariffs cause lower imports.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_10_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Higher tariffs cause lower imports.\\",\\n    \\"output\\": \\"Lower imports are caused by higher tariffs.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_10_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_10\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"causal_direction\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Higher tariffs cause lower imports.\\",\\n    \\"output\\": \\"Lower imports cause higher tariffs.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_11_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The reserve ratio is 17 percent.\\",\\n    \\"output\\": \\"The reserve ratio is 17 percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_11_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The reserve ratio is 17 percent.\\",\\n    \\"output\\": \\"The reserve ratio is seventeen percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_11_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_11\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"numeric_value\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The reserve ratio is 17 percent.\\",\\n    \\"output\\": \\"The reserve ratio is 11 percent.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_12_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"All warehouses reopen except the northern warehouse.\\",\\n    \\"output\\": \\"All warehouses reopen except the northern warehouse.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_12_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"All warehouses reopen except the northern warehouse.\\",\\n    \\"output\\": \\"Every warehouse other than the northern warehouse reopens.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_12_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_12\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"All warehouses reopen except the northern warehouse.\\",\\n    \\"output\\": \\"All warehouses reopen.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_13_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_13\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_endpoint\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Admission is permitted only when the score is at least 10.\\",\\n    \\"output\\": \\"Admission is permitted only when the score is at least 10.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_13_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_13\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_endpoint\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Admission is permitted only when the score is at least 10.\\",\\n    \\"output\\": \\"Admission is permitted only when the score is ten or higher.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_13_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_13\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"bound_endpoint\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Admission is permitted only when the score is at least 10.\\",\\n    \\"output\\": \\"Admission is permitted only when the score is higher than 10.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_14_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_14\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation_scope\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Not all applicants receive grants.\\",\\n    \\"output\\": \\"Not all applicants receive grants.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_14_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_14\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation_scope\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Not all applicants receive grants.\\",\\n    \\"output\\": \\"Some applicants do not receive grants.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_14_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_14\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"negation_scope\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Not all applicants receive grants.\\",\\n    \\"output\\": \\"No applicants receive grants.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_15_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_15\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"association_causality\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Advertising spending is associated with sales growth.\\",\\n    \\"output\\": \\"Advertising spending is associated with sales growth.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_15_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_15\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"association_causality\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Advertising spending is associated with sales growth.\\",\\n    \\"output\\": \\"Sales growth and advertising spending are associated.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_15_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_15\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"association_causality\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Advertising spending is associated with sales growth.\\",\\n    \\"output\\": \\"Advertising spending causes sales growth.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_16_identity\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_16\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"attribution\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The authors hypothesize that storage costs will decline.\\",\\n    \\"output\\": \\"The authors hypothesize that storage costs will decline.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_16_paraphrase\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_16\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"attribution\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The authors hypothesize that storage costs will decline.\\",\\n    \\"output\\": \\"A future decline in storage costs is hypothesized by the authors.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  },\\n  {\\n    \\"id\\": \\"v2_fresh_v2_16_mutation\\",\\n    \\"scenario_group\\": \\"v2_fresh_v2_16\\",\\n    \\"split\\": \\"development\\",\\n    \\"category\\": \\"attribution\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The authors hypothesize that storage costs will decline.\\",\\n    \\"output\\": \\"The authors establish that storage costs will decline.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"previously_exposed_v2_case_development_only\\"\\n  }\\n]\\n", "docs/claim_verifier_calibration_v3/held_out.json": "[\\n  {\\n    \\"id\\": \\"final_fresh_v3_01_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_01\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"upper_endpoint\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The order is accepted only when the error rate is at most 2 percent.\\",\\n    \\"output\\": \\"The order is accepted only when the error rate is at most 2 percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_01_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_01\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"upper_endpoint\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The order is accepted only when the error rate is at most 2 percent.\\",\\n    \\"output\\": \\"The order is accepted only when the error rate is two percent or lower.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_01_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_01\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"upper_endpoint\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The order is accepted only when the error rate is at most 2 percent.\\",\\n    \\"output\\": \\"The order is accepted only when the error rate is less than 2 percent.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_02_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_02\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"negative_endpoint\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The balance is below -5 units.\\",\\n    \\"output\\": \\"The balance is below -5 units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_02_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_02\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"negative_endpoint\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The balance is below -5 units.\\",\\n    \\"output\\": \\"The balance is less than -5 units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_02_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_02\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"negative_endpoint\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The balance is below -5 units.\\",\\n    \\"output\\": \\"The balance is at most -5 units.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_03_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_03\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"decimal_endpoint\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The return hurdle is at least 8.25 percent.\\",\\n    \\"output\\": \\"The return hurdle is at least 8.25 percent.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_03_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_03\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"decimal_endpoint\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The return hurdle is at least 8.25 percent.\\",\\n    \\"output\\": \\"The return hurdle is 8.25% or higher.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_03_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_03\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"decimal_endpoint\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The return hurdle is at least 8.25 percent.\\",\\n    \\"output\\": \\"The return hurdle is greater than 8.25 percent.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_04_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_04\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"bound_scope\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The rating is at least 9. The charge is below 9 units.\\",\\n    \\"output\\": \\"The rating is at least 9. The charge is below 9 units.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_04_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_04\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"bound_scope\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The rating is at least 9. The charge is below 9 units.\\",\\n    \\"output\\": \\"The charge is below 9 units. The rating is at least nine.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_04_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_04\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"bound_scope\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The rating is at least 9. The charge is below 9 units.\\",\\n    \\"output\\": \\"The rating is higher than 9. The charge is below 9 units.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_05_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_05\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"association_to_cause\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Delivery speed is associated with customer satisfaction.\\",\\n    \\"output\\": \\"Delivery speed is associated with customer satisfaction.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_05_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_05\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"association_to_cause\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Delivery speed is associated with customer satisfaction.\\",\\n    \\"output\\": \\"Customer satisfaction and delivery speed are correlated.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_05_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_05\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"association_to_cause\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Delivery speed is associated with customer satisfaction.\\",\\n    \\"output\\": \\"Delivery speed causes customer satisfaction.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_06_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_06\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"association_reverse_cause\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Research spending is correlated with productivity.\\",\\n    \\"output\\": \\"Research spending is correlated with productivity.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_06_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_06\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"association_reverse_cause\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Research spending is correlated with productivity.\\",\\n    \\"output\\": \\"Productivity correlates with research spending.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_06_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_06\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"association_reverse_cause\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Research spending is correlated with productivity.\\",\\n    \\"output\\": \\"Research spending is caused by productivity.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_07_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_07\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"causal_evidence_scope\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Leverage is associated with default frequency. Prices cause revenue.\\",\\n    \\"output\\": \\"Leverage is associated with default frequency. Prices cause revenue.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_07_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_07\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"causal_evidence_scope\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Leverage is associated with default frequency. Prices cause revenue.\\",\\n    \\"output\\": \\"Default frequency and leverage are correlated. Revenue is caused by prices.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_07_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_07\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"causal_evidence_scope\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Leverage is associated with default frequency. Prices cause revenue.\\",\\n    \\"output\\": \\"Leverage causes default frequency. Prices cause revenue.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_08_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_08\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"explicit_causal_evidence\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Training is associated with performance. Training causes performance.\\",\\n    \\"output\\": \\"Training is associated with performance. Training causes performance.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_08_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_08\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"explicit_causal_evidence\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Training is associated with performance. Training causes performance.\\",\\n    \\"output\\": \\"Performance is caused by training. Performance and training are associated.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_08_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_08\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"explicit_causal_evidence\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Training is associated with performance. Training causes performance.\\",\\n    \\"output\\": \\"Performance causes training. Training is associated with performance.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_09_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_09\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The administrator reimburses the claimant.\\",\\n    \\"output\\": \\"The administrator reimburses the claimant.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_09_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_09\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The administrator reimburses the claimant.\\",\\n    \\"output\\": \\"The claimant is reimbursed by the administrator.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_09_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_09\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"actor_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The administrator reimburses the claimant.\\",\\n    \\"output\\": \\"The claimant reimburses the administrator.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_10_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_10\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Juniper holds 4 bonds and Larch holds 16 bonds.\\",\\n    \\"output\\": \\"Juniper holds 4 bonds and Larch holds 16 bonds.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_10_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_10\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Juniper holds 4 bonds and Larch holds 16 bonds.\\",\\n    \\"output\\": \\"Four bonds are held by Juniper; sixteen bonds are held by Larch.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_10_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_10\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"quantity_assignment\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Juniper holds 4 bonds and Larch holds 16 bonds.\\",\\n    \\"output\\": \\"Juniper holds 16 bonds and Larch holds 4 bonds.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_11_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_11\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"If donations fall, charities reduce staffing.\\",\\n    \\"output\\": \\"If donations fall, charities reduce staffing.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_11_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_11\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"If donations fall, charities reduce staffing.\\",\\n    \\"output\\": \\"Charities reduce staffing if donations fall.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_11_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_11\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"condition\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"If donations fall, charities reduce staffing.\\",\\n    \\"output\\": \\"Charities reduce staffing.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_12_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_12\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The warehouse might reopen.\\",\\n    \\"output\\": \\"The warehouse might reopen.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_12_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_12\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The warehouse might reopen.\\",\\n    \\"output\\": \\"The warehouse could reopen.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_12_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_12\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"modality\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The warehouse might reopen.\\",\\n    \\"output\\": \\"The warehouse will certainly reopen.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_13_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_13\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"Most submissions are complete.\\",\\n    \\"output\\": \\"Most submissions are complete.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_13_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_13\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"Most submissions are complete.\\",\\n    \\"output\\": \\"More than half of submissions are complete.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_13_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_13\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"quantifier\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"Most submissions are complete.\\",\\n    \\"output\\": \\"Every submission is complete.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_14_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_14\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The association does not provide loans.\\",\\n    \\"output\\": \\"The association does not provide loans.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_14_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_14\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The association does not provide loans.\\",\\n    \\"output\\": \\"The association provides no loans.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_14_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_14\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"negation\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The association does not provide loans.\\",\\n    \\"output\\": \\"The association provides loans.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_15_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_15\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"All laboratories reopen except the western laboratory.\\",\\n    \\"output\\": \\"All laboratories reopen except the western laboratory.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_15_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_15\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"All laboratories reopen except the western laboratory.\\",\\n    \\"output\\": \\"Every laboratory other than the western laboratory reopens.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_15_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_15\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"exception\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"All laboratories reopen except the western laboratory.\\",\\n    \\"output\\": \\"All laboratories reopen.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_16_identity\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_16\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"attribution\\",\\n    \\"variant\\": \\"identity\\",\\n    \\"source\\": \\"The analysts hypothesize that shipping costs will fall.\\",\\n    \\"output\\": \\"The analysts hypothesize that shipping costs will fall.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_16_paraphrase\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_16\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"attribution\\",\\n    \\"variant\\": \\"paraphrase\\",\\n    \\"source\\": \\"The analysts hypothesize that shipping costs will fall.\\",\\n    \\"output\\": \\"A future fall in shipping costs is hypothesized by the analysts.\\",\\n    \\"expected\\": \\"faithful\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  },\\n  {\\n    \\"id\\": \\"final_fresh_v3_16_mutation\\",\\n    \\"scenario_group\\": \\"final_fresh_v3_16\\",\\n    \\"split\\": \\"held_out\\",\\n    \\"category\\": \\"attribution\\",\\n    \\"variant\\": \\"mutation\\",\\n    \\"source\\": \\"The analysts hypothesize that shipping costs will fall.\\",\\n    \\"output\\": \\"The analysts establish that shipping costs will fall.\\",\\n    \\"expected\\": \\"changed\\",\\n    \\"provenance\\": \\"fresh_authored_v3_case_after_code_freeze\\"\\n  }\\n]\\n", "docs/claim_verifier_calibration_v3/protocol.json": "{\\n  \\"protocol\\": \\"claim_verifier_calibration_v3\\",\\n  \\"code_files\\": {\\n    \\"src/fama_style/research/claim_verifier_v3.py\\": \\"66345673dc1bc92b7c98773eee4900576ea76877f0e6576cbdd0e7e426949ae4\\",\\n    \\"src/fama_style/research/relation_guards.py\\": \\"d693a12ae4d306116a3f229354cac9f87a282c908af2a8693cdcaa948c1e8691\\",\\n    \\"src/fama_style/research/claim_verifier.py\\": \\"723f046b4b3424f828d5db52ea4a778d5b81617053c1be480fa38603f87d8717\\",\\n    \\"src/fama_style/research/claim_audit.py\\": \\"15a77b54720d04acae3b6a39f846a3969023883c3081f658481c396f4539e2a9\\",\\n    \\"src/fama_style/research/claims.py\\": \\"46c52280a1930c779493bf690529c08647e29623579167e10ad0c0eb67735229\\",\\n    \\"src/fama_style/rhetoric/semantics.py\\": \\"256913e4f33978d2c4dbaa9d2a9bebfbe430a17ce0fa61f719e54772a1a94578\\"\\n  },\\n  \\"fixture_files\\": {\\n    \\"development.json\\": \\"7d51ec62eee725b1cd20914fcc34caf6418d8b115e43cc3c95b4828cbf5688a4\\",\\n    \\"held_out.json\\": \\"9efd80277d5f2061597ebfc58ce5f33cdfefecafe483fa226573bdaad9979c35\\"\\n  },\\n  \\"expected_counts\\": {\\n    \\"development\\": 120,\\n    \\"held_out\\": 48\\n  },\\n  \\"readiness_gate\\": {\\n    \\"minimum_error_detection_rate\\": 0.8,\\n    \\"maximum_faithful_false_alarm_rate\\": 0.1,\\n    \\"minimum_nontrivial_faithful_acceptance_rate\\": 0.5,\\n    \\"maximum_changed_false_acceptance_rate\\": 0.0\\n  },\\n  \\"provenance\\": \\"All previously exposed v2 examples are development-only. Final v3 controls authored after the final code freeze; earlier unexecuted candidate fixtures archived locally. Known grammar, no model-result tuning. Targeted authored synthetic challenge, not blinded, independent annotation or economic-domain accuracy.\\",\\n  \\"nli\\": {\\n    \\"model\\": \\"MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli\\",\\n    \\"revision\\": \\"6f5cf0a2b59cabb106aca4c287eed12e357e90eb\\",\\n    \\"entailment_minimum\\": 0.8,\\n    \\"contradiction_maximum\\": 0.1,\\n    \\"dtype\\": \\"float32\\",\\n    \\"max_input_tokens\\": 512,\\n    \\"truncation\\": false\\n  }\\n}\\n"}')
for name, source in files.items():
    path = runtime / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(runtime / "src"))


In [ ]:
import importlib
experiment = importlib.import_module("fama_style.rhetoric.ledger_ablation")
matches = list(Path("/kaggle/input").rglob("ledger_ablation_pack.json"))
if len(matches) != 1:
    raise ValueError("Attach exactly one private ledger_ablation_pack.json")
arguments = ["--pack", str(matches[0]), "--expected-pack-sha", '27fb70dc4a75c8b214dfa38fec9f4b42f96ac255033f70d0bdaffbbff8fbbfee']
experiment.main(arguments + ["--output-dir", "/kaggle/working/ledger_ablation_preflight"])


In [ ]:
experiment.main(arguments + ["--execute-inference", "--output-dir", "/kaggle/working/ledger_grouping_ablation_v1"])
